# Detector clássico de cilindros — V2

Pipeline:

1. pré-processamento e Canny;
2. Hough para gerar segmentos de reta;
3. união, filtragem e validação das retas;
4. formulação de ROIs somente por pares de retas paralelas;
5. score geométrico baseado em overlap, paralelismo e razão comprimento/largura;
6. HOG + SVM para classificar as ROIs;
7. validação visual e runtime em webcam.


In [1]:
# 1. Imports e caminhos principais


from pathlib import Path
from collections import OrderedDict
import os
import sys
import shutil
import cv2
import json
import time
import math
import hashlib
import traceback
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

try:
    import joblib
except Exception as e:
    raise ImportError("Instale o joblib para salvar o modelo: pip install joblib") from e


# Extensões de imagem aceitas pelo notebook.
# Mantido aqui para que as funções de listagem do dataset funcionem
# independentemente da ordem de execução das células.
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}



# ============================================================
# Caminhos fornecidos para o projeto
# ============================================================

# Caminhos portáteis do projeto BLAZE
CURRENT_DIR = Path.cwd().resolve()

for _candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (_candidate / 'src' / 'blaze_paths.py').exists():
        PROJECT_ROOT = _candidate
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a raiz do projeto BLAZE. '
        'Abra este notebook a partir de uma pasta dentro do repositório BLAZE.'
    )

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from blaze_paths import (
    VISION_DATASETS_DIR,
    VISION_RESULTS_DIR,
    VISION_OFFICIAL_SETUPS_DIR,
    VISION_USER_SETUPS_DIR,
    ensure_base_dirs,
)

ensure_base_dirs()

DATASET_DIR = VISION_DATASETS_DIR / 'cylinders' / 'CylinDeRS-1'

TRAIN_IMG_DIR = DATASET_DIR / 'train' / 'images'
TRAIN_LABEL_DIR = DATASET_DIR / 'train' / 'labels'
TEST_IMG_DIR = DATASET_DIR / 'test' / 'images'
TEST_LABEL_DIR = DATASET_DIR / 'test' / 'labels'

RESULTS_DIR = VISION_RESULTS_DIR
GEOM_DIR = RESULTS_DIR / 'geometric_realtime_detector'
GEOM_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_DIR = GEOM_DIR / 'configs'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

CLASSICAL_DIR = RESULTS_DIR / 'classical_cylinder_detector'
CLASSICAL_DIR.mkdir(parents=True, exist_ok=True)

SELECAO_TREINO_PATH = CLASSICAL_DIR / 'imagens_selecionadas_treino.json'
SELECAO_TESTE_PATH = CLASSICAL_DIR / 'imagens_selecionadas_teste.json'

ZOOMOUT_TREINO_PATH = CLASSICAL_DIR / 'imagens_zoomout_treino.json'
ZOOMOUT_TESTE_PATH = CLASSICAL_DIR / 'imagens_zoomout_teste.json'
ZOOMOUT_CONFIG_PATH = CLASSICAL_DIR / 'zoomout_config.json'

ZOOMOUT_DATASET_DIR = CLASSICAL_DIR / 'dataset_zoomout_aplicado'
ZOOMOUT_DATASET_DIR.mkdir(parents=True, exist_ok=True)
ZOOMOUT_MANIFEST_PATH = CLASSICAL_DIR / 'dataset_zoomout_manifest.json'

MODEL_PATH = CLASSICAL_DIR / 'modelo_hog_svm_cilindros_v2.joblib'
SCALER_PATH = CLASSICAL_DIR / 'scaler_hog_svm_cilindros.joblib'
METADATA_PATH = CLASSICAL_DIR / 'metadata_modelo_v2.json'
HISTORICO_PATH = CLASSICAL_DIR / 'historico_treinos_v2.csv'
# Compatibilidade com funções antigas do notebook.
HISTORY_PATH = HISTORICO_PATH

OFFICIAL_PARAM_SETUPS_PATH = VISION_OFFICIAL_SETUPS_DIR / 'cylinders_detect' / 'setups_parametricos_v2.json'
USER_PARAM_SETUPS_PATH = VISION_USER_SETUPS_DIR / 'cylinders_detect' / 'setups_parametricos_v2_user.json'

OFFICIAL_REFINE_SETUPS_PATH = VISION_OFFICIAL_SETUPS_DIR / 'cylinders_detect' / 'setups_refinamento_expansao_v2_nao_usado.json'
USER_REFINE_SETUPS_PATH = VISION_USER_SETUPS_DIR / 'cylinders_detect' / 'setups_refinamento_expansao_v2_nao_usado_user.json'


def inicializar_arquivo_setup_local(official_path, user_path):
    official_path = Path(official_path)
    user_path = Path(user_path)
    user_path.parent.mkdir(parents=True, exist_ok=True)

    if not user_path.exists() and official_path.exists():
        shutil.copy2(official_path, user_path)
        print(f'Arquivo oficial copiado para edição local: {user_path}')

    return user_path


PARAM_SETUPS_PATH = inicializar_arquivo_setup_local(
    OFFICIAL_PARAM_SETUPS_PATH,
    USER_PARAM_SETUPS_PATH
)

REFINE_SETUPS_PATH = inicializar_arquivo_setup_local(
    OFFICIAL_REFINE_SETUPS_PATH,
    USER_REFINE_SETUPS_PATH
)


print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_DIR:', DATASET_DIR)

if not DATASET_DIR.exists():
    print('')
    print('ATENÇÃO: dataset CylinDeRS-1 não encontrado.')
    print('Coloque ou vincule o dataset em:')
    print(DATASET_DIR)
    print('')
    print('Estrutura esperada:')
    print('CylinDeRS-1/train/images')
    print('CylinDeRS-1/train/labels')
    print('CylinDeRS-1/test/images')
    print('CylinDeRS-1/test/labels')
else:
    print('Dataset encontrado.')
print('TRAIN_IMG_DIR:', TRAIN_IMG_DIR)
print('TRAIN_LABEL_DIR:', TRAIN_LABEL_DIR)
print('TEST_IMG_DIR:', TEST_IMG_DIR)
print('TEST_LABEL_DIR:', TEST_LABEL_DIR)
print('CLASSICAL_DIR:', CLASSICAL_DIR)
print('ZOOMOUT_DATASET_DIR:', ZOOMOUT_DATASET_DIR)


PROJECT_ROOT: .
DATASET_DIR: vision/datasets/cylinders/CylinDeRS-1
Dataset encontrado.
TRAIN_IMG_DIR: vision/datasets/cylinders/CylinDeRS-1/train/images
TRAIN_LABEL_DIR: vision/datasets/cylinders/CylinDeRS-1/train/labels
TEST_IMG_DIR: vision/datasets/cylinders/CylinDeRS-1/test/images
TEST_LABEL_DIR: vision/datasets/cylinders/CylinDeRS-1/test/labels
CLASSICAL_DIR: vision/results/classical_cylinder_detector
ZOOMOUT_DATASET_DIR: vision/results/classical_cylinder_detector/dataset_zoomout_aplicado


In [2]:
# ============================================================
# 1C. Verificação de GPUs disponíveis — somente Jupyter/UFSC
# ============================================================
# Esta célula é opcional e foi feita para o servidor Jupyter da UFSC.
# Em execução local pelo VS Code, ela é ignorada automaticamente.

import os
import subprocess
from IPython.display import display, HTML


def ambiente_jupyter_ufsc():
    """Retorna True quando o notebook parece estar rodando no Jupyter/UFSC.

    Observação:
    - VS Code também pode executar notebooks, então verificamos variáveis
      típicas do VS Code e evitamos rodar esta célula nesse caso.
    - O servidor da UFSC costuma usar ambiente Jupyter/JupyterHub e caminhos
      como /home/jovyan.
    """
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
    except Exception:
        shell = ""

    em_notebook = shell == "ZMQInteractiveShell"
    em_vscode = any(k in os.environ for k in [
        "VSCODE_PID",
        "VSCODE_CWD",
        "VSCODE_NLS_CONFIG",
        "VSCODE_IPC_HOOK_CLI",
    ])
    sinais_jupyter_ufsc = any(k in os.environ for k in [
        "JUPYTERHUB_USER",
        "JUPYTERHUB_SERVICE_PREFIX",
        "JPY_PARENT_PID",
    ]) or os.getcwd().startswith("/home/jovyan")

    return bool(em_notebook and sinais_jupyter_ufsc and not em_vscode)


def consultar_gpus_nvidia():
    """Consulta GPUs via nvidia-smi e retorna lista de dicionários."""
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,name,memory.used,memory.total,utilization.gpu,temperature.gpu",
        "--format=csv,noheader,nounits",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
    if proc.returncode != 0:
        return [], proc.stderr.strip() or proc.stdout.strip()

    gpus = []
    for line in proc.stdout.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 6:
            continue
        idx, name, mem_used, mem_total, util, temp = parts[:6]
        try:
            mem_used_i = int(float(mem_used))
            mem_total_i = int(float(mem_total))
            util_i = int(float(util))
            temp_i = int(float(temp))
        except Exception:
            mem_used_i = mem_total_i = util_i = temp_i = 0
        livre = max(0, mem_total_i - mem_used_i)
        gpus.append({
            "GPU": int(idx),
            "Nome": name,
            "Memória usada (MiB)": mem_used_i,
            "Memória total (MiB)": mem_total_i,
            "Memória livre (MiB)": livre,
            "Uso GPU (%)": util_i,
            "Temp. (°C)": temp_i,
        })
    return gpus, ""


if not ambiente_jupyter_ufsc():
    display(HTML(
        "<b>Verificação de GPU ignorada.</b><br>"
        "Esta célula foi configurada para rodar somente no Jupyter/UFSC. "
        "Em uso local pelo VS Code, nenhuma GPU será selecionada por aqui."
    ))
    GPUS_DISPONIVEIS = []
else:
    gpus, erro_gpu = consultar_gpus_nvidia()
    GPUS_DISPONIVEIS = gpus

    if not gpus:
        display(HTML(
            "<b>Nenhuma GPU NVIDIA foi listada.</b><br>"
            f"Mensagem: <code>{erro_gpu or 'nvidia-smi não retornou GPUs.'}</code>"
        ))
    else:
        try:
            import pandas as pd
            df_gpus = pd.DataFrame(gpus)
            display(HTML("<b>GPUs disponíveis no servidor Jupyter</b>"))
            display(df_gpus)
        except Exception:
            print("GPUs disponíveis:")
            for g in gpus:
                print(g)

        melhor = sorted(gpus, key=lambda g: (g["Uso GPU (%)"], -g["Memória livre (MiB)"]))[0]
        display(HTML(
            "<small>Use a próxima célula para selecionar uma GPU. "
            f"Sugestão automática: GPU {melhor['GPU']} — menor uso atual e boa memória livre.</small>"
        ))

In [3]:
# ============================================================
# 1D. Seleção de GPU para esta sessão — somente Jupyter/UFSC
# ============================================================
# Esta célula define CUDA_VISIBLE_DEVICES apenas nesta sessão do notebook.
# Rode antes de qualquer biblioteca que inicialize CUDA, como torch/tensorflow.
#
# Observação importante:
# - O detector clássico HOG/SVM usa majoritariamente CPU.
# - Esta seleção é útil caso alguma etapa auxiliar use CUDA ou caso você
#   aproveite o mesmo kernel para outros testes com GPU.

import os
from IPython.display import display, HTML
import ipywidgets as widgets

if not ambiente_jupyter_ufsc():
    display(HTML(
        "<b>Seleção de GPU ignorada.</b><br>"
        "Esta célula foi configurada para rodar somente no Jupyter/UFSC. "
        "No VS Code/local, ela não altera <code>CUDA_VISIBLE_DEVICES</code>."
    ))
else:
    if 'GPUS_DISPONIVEIS' not in globals() or not GPUS_DISPONIVEIS:
        GPUS_DISPONIVEIS, _erro_gpu = consultar_gpus_nvidia()

    if not GPUS_DISPONIVEIS:
        display(HTML(
            "<b>Nenhuma GPU disponível para seleção.</b><br>"
            "Execute a célula anterior para verificar o diagnóstico."
        ))
    else:
        opcoes_gpu = []
        for g in GPUS_DISPONIVEIS:
            rotulo = (
                f"GPU {g['GPU']} | {g['Nome']} | "
                f"livre {g['Memória livre (MiB)']} MiB / {g['Memória total (MiB)']} MiB | "
                f"uso {g['Uso GPU (%)']}%"
            )
            opcoes_gpu.append((rotulo, str(g['GPU'])))

        gpu_dropdown = widgets.Dropdown(
            options=opcoes_gpu,
            description='GPU:',
            layout=widgets.Layout(width='95%'),
            style={'description_width': '50px'}
        )

        usar_cpu_checkbox = widgets.Checkbox(
            value=False,
            description='Forçar CPU nesta sessão',
            indent=False
        )

        aplicar_gpu_btn = widgets.Button(
            description='Aplicar seleção',
            button_style='success',
            icon='check'
        )

        saida_gpu = widgets.Output()

        def aplicar_gpu(_=None):
            with saida_gpu:
                saida_gpu.clear_output()

                if usar_cpu_checkbox.value:
                    os.environ['CUDA_VISIBLE_DEVICES'] = ''
                    print('CPU forçada nesta sessão: CUDA_VISIBLE_DEVICES=""')
                    return

                gpu_escolhida = str(gpu_dropdown.value)
                os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
                os.environ['CUDA_VISIBLE_DEVICES'] = gpu_escolhida
                print(f'GPU selecionada para esta sessão: {gpu_escolhida}')
                print(f'CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES", "")!r}')
                print('\nObservação: se torch/tensorflow já tiverem inicializado CUDA, reinicie o kernel e rode esta célula antes deles.')

        aplicar_gpu_btn.on_click(aplicar_gpu)

        display(HTML(
            '<b>Seleção de GPU</b><br>'
            '<small>Escolha uma GPU antes de executar etapas pesadas. '
            'Esta escolha vale apenas para o kernel atual.</small>'
        ))
        display(widgets.VBox([gpu_dropdown, usar_cpu_checkbox, aplicar_gpu_btn, saida_gpu]))

In [4]:
# ============================================================
# 1B. Controle interno de pendências para Git
# ============================================================
# Este bloco NÃO executa commit nem push.
# Ele apenas registra quais arquivos de configuração foram alterados por botões do notebook,
# para que o painel final possa preparar somente caminhos permitidos.

GIT_PENDING_PATHS = globals().get('GIT_PENDING_PATHS', OrderedDict())

def _relpath_git(path):
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve())).replace('\\', '/')
    except Exception:
        return str(path).replace('\\', '/')

def registrar_pendencia_git(path, motivo='alteração feita no notebook'):
    rel = _relpath_git(path)
    GIT_PENDING_PATHS[rel] = {
        'motivo': str(motivo),
        'registrado_em': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    return rel

def listar_pendencias_git():
    return OrderedDict(GIT_PENDING_PATHS)


In [5]:
# ============================================================
# 2. Funções utilitárias gerais e leitura do dataset YOLO
# ============================================================

def caminhos_dataset(dataset_dir):
    """Retorna os caminhos padronizados para um dataset no formato YOLO train/test."""
    dataset_dir = Path(dataset_dir).expanduser().resolve()
    return {
        'dataset_dir': dataset_dir,
        'train_img_dir': dataset_dir / 'train' / 'images',
        'train_label_dir': dataset_dir / 'train' / 'labels',
        'test_img_dir': dataset_dir / 'test' / 'images',
        'test_label_dir': dataset_dir / 'test' / 'labels',
    }


def label_path_para_imagem(img_path, img_dir, label_dir):
    """Converte caminho de imagem em caminho esperado do label YOLO correspondente."""
    img_path = Path(img_path)
    img_dir = Path(img_dir)
    label_dir = Path(label_dir)
    try:
        rel = img_path.relative_to(img_dir)
    except ValueError:
        rel = Path(img_path.name)
    return (label_dir / rel).with_suffix('.txt')


def contar_labels_yolo(label_path):
    """Conta apenas linhas YOLO válidas e finitas em um arquivo de label.

    Alguns arquivos podem conter valores ausentes/NaN. Esses registros são ignorados
    para evitar erros posteriores na conversão para bounding boxes.
    """
    label_path = Path(label_path)
    if not label_path.exists():
        return 0

    n = 0
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls = float(parts[0])
                vals = list(map(float, parts[1:5]))
            except Exception:
                continue
            if not (np.isfinite(cls) and all(np.isfinite(v) for v in vals)):
                continue
            xc, yc, bw, bh = vals
            if bw <= 0 or bh <= 0:
                continue
            n += 1
    return n


def listar_imagens_split(split, img_dir, label_dir):
    """Lista imagens de um split e associa cada uma ao arquivo de label YOLO."""
    img_dir = Path(img_dir).expanduser().resolve()
    label_dir = Path(label_dir).expanduser().resolve()

    rows = []
    if not img_dir.exists():
        return rows

    for p in sorted(img_dir.rglob('*')):
        if not (p.is_file() and p.suffix.lower() in IMG_EXTS):
            continue

        lab = label_path_para_imagem(p, img_dir, label_dir)
        n_obj = contar_labels_yolo(lab)
        rows.append({
            'split': split,
            'path': str(p),
            'arquivo': p.name,
            'stem': p.stem,
            'label_path': str(lab),
            'label_existe': lab.exists(),
            'n_obj': int(n_obj),
            'classe': 'com_anotacao' if n_obj > 0 else 'sem_anotacao'
        })

    return rows


def listar_imagens_dataset(dataset_dir):
    """Lista imagens de train e test a partir da raiz do CylinDeRS-1."""
    paths = caminhos_dataset(dataset_dir)

    rows = []
    rows.extend(listar_imagens_split('train', paths['train_img_dir'], paths['train_label_dir']))
    rows.extend(listar_imagens_split('test', paths['test_img_dir'], paths['test_label_dir']))

    return pd.DataFrame(rows, columns=[
        'split', 'path', 'arquivo', 'stem', 'label_path', 'label_existe', 'n_obj', 'classe'
    ])


def ler_imagem_bgr(path_img):
    """Lê imagem com OpenCV em BGR."""
    img = cv2.imread(str(path_img), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Não foi possível carregar a imagem: {path_img}')
    return img


def bgr_para_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def clip_bbox(x1, y1, x2, y2, w, h):
    x1 = max(0, min(int(x1), w - 1))
    y1 = max(0, min(int(y1), h - 1))
    x2 = max(1, min(int(x2), w))
    y2 = max(1, min(int(y2), h))
    return x1, y1, x2, y2


def resumo_dataset(df):
    if df is None or len(df) == 0:
        return 'Nenhuma imagem encontrada.'

    total = len(df)
    n_train = int((df['split'] == 'train').sum())
    n_test = int((df['split'] == 'test').sum())
    com_label = int(df['label_existe'].sum())
    imagens_com_obj = int((df['n_obj'] > 0).sum())
    total_obj = int(df['n_obj'].sum())

    return (
        f'Imagens: {total} | Train: {n_train} | Test: {n_test} | '
        f'Labels existentes: {com_label} | Imagens com objeto: {imagens_com_obj} | '
        f'Objetos anotados: {total_obj}'
    )


def exibir_tabela_compacta(df, max_rows=12):
    """
    Mostra uma tabela compacta com rolagem horizontal controlada,
    para evitar que ela ultrapasse a largura visual do notebook.
    """
    if df is None or len(df) == 0:
        display(HTML('<em>Nenhum registro para mostrar.</em>'))
        return

    df_show = df.tail(max_rows).copy()

    for col in df_show.columns:
        if df_show[col].dtype == object:
            df_show[col] = df_show[col].astype(str).str.slice(0, 46)

    html = df_show.to_html(index=False, escape=False)
    display(HTML(f'''
    <div style="
        max-width: 100%;
        overflow-x: auto;
        border: 1px solid #444;
        padding: 6px;
        border-radius: 8px;
    ">
        <style>
            table.dataframe {{
                font-size: 11px;
                border-collapse: collapse;
                width: max-content;
                max-width: 100%;
            }}
            table.dataframe th, table.dataframe td {{
                white-space: nowrap;
                padding: 4px 7px;
                text-align: center;
            }}
        </style>
        {html}
    </div>
    '''))


def salvar_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def carregar_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


In [6]:
# ============================================================
# 3. Pipeline de pré-processamento com dois mapas de bordas
# ============================================================


def filtrar_componentes_borda(edge_map, p):
    """
    Remove pequenos componentes conectados do mapa binário de bordas.

    Objetivo:
        - preservar o Canny completo como diagnóstico visual;
        - gerar um segundo mapa mais limpo para Hough e geração de ROIs;
        - reduzir falsas bordas antes da formulação geométrica.

    Critérios de permanência:
        componente permanece se:
            área >= edge_min_area E maior dimensão >= edge_min_extent

    Retorna:
        edge_filtrado, info
    """
    if not bool(p.get('edge_filter_ativo', True)):
        return edge_map.copy(), {
            'enabled': False,
            'n_components': 0,
            'n_removed': 0,
            'n_kept': 0,
            'min_area': 0,
            'min_extent': 0
        }

    min_area = max(0, int(p.get('edge_min_area', 12)))
    min_extent = max(0, int(p.get('edge_min_extent', 10)))

    bin_map = (edge_map > 0).astype(np.uint8)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_map, connectivity=8)

    out = np.zeros_like(edge_map)
    kept = 0
    removed = 0

    for lab in range(1, n_labels):
        x, y, w, h, area = stats[lab]
        extent = max(int(w), int(h))
        keep = (int(area) >= min_area) and (extent >= min_extent)
        if keep:
            out[labels == lab] = 255
            kept += 1
        else:
            removed += 1

    info = {
        'enabled': True,
        'n_components': int(max(0, n_labels - 1)),
        'n_removed': int(removed),
        'n_kept': int(kept),
        'min_area': int(min_area),
        'min_extent': int(min_extent)
    }
    return out, info


def aplicar_preprocessamento(img_bgr, p):
    """
    Aplica a preparação da imagem sem etapa morfológica.

    Mapas produzidos:
        - Mapa A: Canny completo, usado para diagnóstico visual;
        - Mapa B: Canny filtrado, usado para Hough e geração de ROIs.
    """
    rgb = bgr_para_rgb(img_bgr)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    if p.get("clahe_ativo", True):
        clahe = cv2.createCLAHE(
            clipLimit=float(p.get("clahe_clip", 2.0)),
            tileGridSize=(int(p.get("clahe_grid", 8)), int(p.get("clahe_grid", 8)))
        )
        gray_eq = clahe.apply(gray)
    else:
        gray_eq = gray.copy()

    d = int(p.get("bilateral_d", 5))
    if d < 1:
        d = 1
    if d % 2 == 0:
        d += 1

    bilateral = cv2.bilateralFilter(
        gray_eq,
        d=d,
        sigmaColor=float(p.get("bilateral_sigma_color", 50)),
        sigmaSpace=float(p.get("bilateral_sigma_space", 50))
    )

    med = float(np.median(bilateral))
    sigma = float(p.get("canny_sigma", 0.33))
    lower = int(max(0, (1.0 - sigma) * med))
    upper = int(min(255, (1.0 + sigma) * med))

    if upper <= lower:
        upper = min(255, lower + 20)

    aperture = int(p.get("canny_aperture", 3))
    if aperture not in [3, 5, 7]:
        aperture = 3

    canny_full = cv2.Canny(
        bilateral,
        threshold1=lower,
        threshold2=upper,
        apertureSize=aperture,
        L2gradient=bool(p.get("canny_l2gradient", True))
    )

    canny_hough, edge_filter_info = filtrar_componentes_borda(canny_full, p)

    # Gradiente usado para validar contornos do objeto.
    # O valor é normalizado entre 0 e 1 para que o parâmetro de
    # gradiente mínimo seja estável entre imagens diferentes.
    grad_x = cv2.Sobel(bilateral, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(bilateral, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = cv2.magnitude(grad_x, grad_y)
    grad_norm = grad_mag / max(1e-6, float(np.percentile(grad_mag, 99)))
    grad_norm = np.clip(grad_norm, 0.0, 1.0).astype(np.float32)

    return {
        "rgb": rgb,
        "gray": gray,
        "gray_eq": gray_eq,
        "bilateral": bilateral,
        "grad_mag": grad_norm,
        "canny": canny_full,
        "canny_full": canny_full,
        "canny_hough": canny_hough,
        "canny_filtrado": canny_hough,
        "canny_lower": lower,
        "canny_upper": upper,
        "edge_filter_info": edge_filter_info
    }


def selecionar_imagem_hog(pre, p=None):
    """
    Retorna a imagem usada para extração HOG.

    Nesta versão simplificada, o HOG/SVM usa sempre o mapa bilateral,
    porque ele preserva bordas e textura sem transformar a imagem em
    um mapa binário de bordas.
    """
    return pre["bilateral"]


In [7]:
# ============================================================
# 4. Hough, pareamento geométrico e geração de ROIs
# ============================================================

def detectar_linhas_hough(img_edges, p):
    """
    Detecta segmentos de reta com HoughLinesP.
    """
    lines = cv2.HoughLinesP(
        img_edges,
        rho=float(p["hough_rho"]),
        theta=np.deg2rad(float(p["hough_theta_deg"])),
        threshold=int(p["hough_threshold"]),
        minLineLength=int(p["hough_min_line_length"]),
        maxLineGap=int(p["hough_max_line_gap"])
    )

    if lines is None:
        return []

    return [tuple(map(int, l[0])) for l in lines]


def metrica_linha(line):
    x1, y1, x2, y2 = line
    dx = x2 - x1
    dy = y2 - y1
    length = float(math.hypot(dx, dy))
    angle = math.degrees(math.atan2(dy, dx)) % 180.0
    center = np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=float)

    if length == 0:
        u = np.array([1.0, 0.0])
    else:
        u = np.array([dx / length, dy / length], dtype=float)

    return {
        "line": line,
        "length": length,
        "angle": angle,
        "center": center,
        "u": u
    }


def diff_angular_graus(a, b):
    """
    Diferença angular mínima considerando orientação de reta, não vetor.
    Retorna valor entre 0 e 90 graus.
    """
    d = abs((a - b + 90.0) % 180.0 - 90.0)
    return float(d)


def intervalo_projetado(line, u):
    x1, y1, x2, y2 = line
    p1 = np.array([x1, y1], dtype=float)
    p2 = np.array([x2, y2], dtype=float)
    v1 = float(np.dot(p1, u))
    v2 = float(np.dot(p2, u))
    return min(v1, v2), max(v1, v2)


def razao_sobreposicao(i1, i2):
    a1, a2 = i1
    b1, b2 = i2

    inter = max(0.0, min(a2, b2) - max(a1, b1))
    len1 = max(1e-6, a2 - a1)
    len2 = max(1e-6, b2 - b1)

    return float(inter / min(len1, len2))


def encontrar_rois_por_pares_de_linhas(lines, shape_hw, p):
    """
    Procura pares de retas paralelas e gera ROIs candidatas.
    """
    h, w = shape_hw
    metricas = [metrica_linha(l) for l in lines]
    metricas = [
        m for m in metricas
        if p["line_length_min"] <= m["length"] <= p["line_length_max"]
    ]

    candidatos = []

    for i in range(len(metricas)):
        for j in range(i + 1, len(metricas)):
            m1 = metricas[i]
            m2 = metricas[j]

            adiff = diff_angular_graus(m1["angle"], m2["angle"])
            if adiff > p["pair_angle_tol_deg"]:
                continue

            # Orientação média do par.
            u = m1["u"] + m2["u"]
            if np.linalg.norm(u) < 1e-6:
                u = m1["u"]
            else:
                u = u / np.linalg.norm(u)

            # Normal à direção da reta.
            n = np.array([-u[1], u[0]], dtype=float)

            # Distância transversal entre centros.
            dist = abs(float(np.dot(m2["center"] - m1["center"], n)))
            if not (p["pair_dist_min"] <= dist <= p["pair_dist_max"]):
                continue

            # Sobreposição longitudinal.
            int1 = intervalo_projetado(m1["line"], u)
            int2 = intervalo_projetado(m2["line"], u)
            overlap = razao_sobreposicao(int1, int2)
            if overlap < p["pair_overlap_min"]:
                continue

            pts = np.array([
                [m1["line"][0], m1["line"][1]],
                [m1["line"][2], m1["line"][3]],
                [m2["line"][0], m2["line"][1]],
                [m2["line"][2], m2["line"][3]],
            ], dtype=float)

            margin = int(p["roi_margin_px"])
            x1 = int(np.floor(pts[:, 0].min())) - margin
            y1 = int(np.floor(pts[:, 1].min())) - margin
            x2 = int(np.ceil(pts[:, 0].max())) + margin
            y2 = int(np.ceil(pts[:, 1].max())) + margin

            x1, y1, x2, y2 = clip_bbox(x1, y1, x2, y2, w, h)

            roi_w = max(1, x2 - x1)
            roi_h = max(1, y2 - y1)
            aspect = roi_h / roi_w

            if not (p["roi_aspect_min"] <= aspect <= p["roi_aspect_max"]):
                continue

            area = roi_w * roi_h

            candidatos.append({
                "bbox": (x1, y1, x2, y2),
                "line1": m1["line"],
                "line2": m2["line"],
                "angle_diff": adiff,
                "distance": dist,
                "overlap": overlap,
                "aspect": aspect,
                "area": area
            })

    # Ordenação simples: prioriza boa sobreposição e baixa diferença angular.
    candidatos = sorted(
        candidatos,
        key=lambda c: (-c["overlap"], c["angle_diff"], abs(c["distance"] - (p["pair_dist_min"] + p["pair_dist_max"]) / 2))
    )

    return candidatos[:int(p["max_rois"])]


def desenhar_linhas(img_rgb, lines, max_lines=80):
    out = img_rgb.copy()
    for line in lines[:max_lines]:
        x1, y1, x2, y2 = line
        cv2.line(out, (x1, y1), (x2, y2), (0, 220, 255), 2)
    return out


def desenhar_rois(img_rgb, candidatos, mostrar_pares=True):
    out = img_rgb.copy()

    for idx, c in enumerate(candidatos):
        x1, y1, x2, y2 = c["bbox"]

        # ROI em azul/laranja.
        cv2.rectangle(out, (x1, y1), (x2, y2), (255, 120, 0), 2)
        cv2.putText(
            out,
            f"ROI {idx+1}",
            (x1, max(15, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 120, 0),
            1,
            cv2.LINE_AA
        )

        if mostrar_pares:
            for line in [c["line1"], c["line2"]]:
                lx1, ly1, lx2, ly2 = line
                cv2.line(out, (lx1, ly1), (lx2, ly2), (0, 220, 255), 2)

    return out

In [8]:
# ============================================================
# 5. HOG, labels YOLO, assinatura de parâmetros e histórico
# ============================================================

def extrair_hog_de_imagem(img_bgr, p):
    """Extrai vetor HOG de uma imagem/crop."""
    pre = aplicar_preprocessamento(img_bgr, p)
    img_hog = selecionar_imagem_hog(pre, p)

    w = int(p['hog_resize_w'])
    h = int(p['hog_resize_h'])
    img_resized = cv2.resize(img_hog, (w, h), interpolation=cv2.INTER_AREA)

    features = hog(
        img_resized,
        orientations=int(p['hog_orientations']),
        pixels_per_cell=(int(p['hog_pixels_per_cell']), int(p['hog_pixels_per_cell'])),
        cells_per_block=(int(p['hog_cells_per_block']), int(p['hog_cells_per_block'])),
        block_norm='L2-Hys',
        transform_sqrt=True,
        feature_vector=True
    )

    return features.astype(np.float32)


def n_features_esperado_modelo(clf):
    """Retorna o número de atributos esperado pelo modelo treinado, quando disponível."""
    if clf is None:
        return None

    candidatos = []

    if hasattr(clf, "named_steps"):
        candidatos.extend([
            clf.named_steps.get("standardscaler"),
            clf.named_steps.get("scaler"),
            clf.named_steps.get("linearsvc"),
            clf.named_steps.get("svc"),
        ])

    if hasattr(clf, "steps"):
        candidatos.extend([step for _, step in clf.steps])

    candidatos.append(clf)

    for obj in candidatos:
        if obj is None:
            continue
        n = getattr(obj, "n_features_in_", None)
        if n is not None:
            return int(n)

    return None


def validar_features_modelo(feat, clf, contexto="ROI"):
    """Valida compatibilidade entre extrator HOG atual e modelo treinado.

    Esta função evita classificar uma ROI quando o tamanho do vetor HOG
    gerado no notebook não coincide com o tamanho esperado pelo modelo SVM.
    A correção limpa, nesses casos, é retreinar o modelo no notebook atual.
    """
    feat = np.asarray(feat, dtype=np.float32).reshape(1, -1)
    esperado = n_features_esperado_modelo(clf)
    atual = int(feat.shape[1])

    if esperado is not None and atual != esperado:
        raise RuntimeError(
            f"{contexto}: incompatibilidade de features HOG/SVM. "
            f"O notebook gerou {atual} atributos, mas o modelo espera {esperado}. "
            "Retreine o modelo neste notebook para alinhar extrator e classificador."
        )

    return feat


def extrair_hog_visualizacao_de_imagem(img_bgr, p):
    """
    Extrai HOG e também retorna a imagem usada e a visualização dos gradientes.

    A função é usada apenas para diagnóstico visual. O treinamento continua usando
    `extrair_hog_de_imagem`, com os mesmos parâmetros.
    """
    pre = aplicar_preprocessamento(img_bgr, p)
    img_hog = selecionar_imagem_hog(pre, p)

    w = int(p['hog_resize_w'])
    h = int(p['hog_resize_h'])
    img_resized = cv2.resize(img_hog, (w, h), interpolation=cv2.INTER_AREA)

    features, hog_image = hog(
        img_resized,
        orientations=int(p['hog_orientations']),
        pixels_per_cell=(int(p['hog_pixels_per_cell']), int(p['hog_pixels_per_cell'])),
        cells_per_block=(int(p['hog_cells_per_block']), int(p['hog_cells_per_block'])),
        block_norm='L2-Hys',
        transform_sqrt=True,
        visualize=True,
        feature_vector=True
    )

    hog_image = np.asarray(hog_image, dtype=np.float32)
    if hog_image.max() > hog_image.min():
        hog_image = (hog_image - hog_image.min()) / (hog_image.max() - hog_image.min())

    return img_resized, hog_image, features.astype(np.float32)


def ler_bboxes_yolo(label_path, img_w, img_h, class_filter=None):
    """
    Lê bboxes YOLO normalizadas e retorna bboxes em pixel.

    Retorna lista de dicionários:
        {'class_id': int, 'bbox': (x1, y1, x2, y2)}

    Registros inválidos, NaN, infinitos, largura/altura não positivas ou caixas
    degeneradas são ignorados. Isso evita erros do tipo
    "cannot convert float NaN to integer" durante a validação visual.
    """
    label_path = Path(label_path)
    if not label_path.exists():
        return []

    boxes = []
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls_float = float(parts[0])
                vals = list(map(float, parts[1:5]))
            except Exception:
                continue

            if not (np.isfinite(cls_float) and all(np.isfinite(v) for v in vals)):
                continue

            cls = int(cls_float)
            xc, yc, bw, bh = vals
            if bw <= 0 or bh <= 0:
                continue

            if class_filter is not None and cls not in class_filter:
                continue

            # Aceita anotações levemente fora de [0, 1], mas descarta caixas totalmente inválidas.
            x1 = (xc - bw / 2.0) * img_w
            y1 = (yc - bh / 2.0) * img_h
            x2 = (xc + bw / 2.0) * img_w
            y2 = (yc + bh / 2.0) * img_h
            if not all(np.isfinite(v) for v in [x1, y1, x2, y2]):
                continue

            # Se a caixa inteira ficou fora da imagem, ignora.
            if x2 <= 0 or y2 <= 0 or x1 >= img_w or y1 >= img_h:
                continue

            x1, y1, x2, y2 = clip_bbox(x1, y1, x2, y2, img_w, img_h)

            if (x2 - x1) >= 4 and (y2 - y1) >= 4:
                boxes.append({'class_id': cls, 'bbox': (x1, y1, x2, y2)})

    return boxes


def expandir_bbox(bbox, margin_pct, img_w, img_h):
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1
    mx = bw * float(margin_pct)
    my = bh * float(margin_pct)
    return clip_bbox(x1 - mx, y1 - my, x2 + mx, y2 + my, img_w, img_h)


def iou_bbox(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(1, (ax2 - ax1) * (ay2 - ay1))
    area_b = max(1, (bx2 - bx1) * (by2 - by1))
    return inter / float(area_a + area_b - inter + 1e-9)


def bbox_aleatorio(img_w, img_h, target_w, target_h, rng):
    target_w = int(max(8, min(target_w, img_w)))
    target_h = int(max(8, min(target_h, img_h)))
    if img_w <= target_w or img_h <= target_h:
        return (0, 0, img_w, img_h)
    x1 = int(rng.integers(0, img_w - target_w))
    y1 = int(rng.integers(0, img_h - target_h))
    return (x1, y1, x1 + target_w, y1 + target_h)


def recortar_bbox(img_bgr, bbox):
    x1, y1, x2, y2 = map(int, bbox)
    return img_bgr[y1:y2, x1:x2].copy()


def extrair_strings_json(obj):
    """Extrai recursivamente strings de um JSON de seleção."""
    vals = []
    if obj is None:
        return vals
    if isinstance(obj, str):
        return [obj]
    if isinstance(obj, dict):
        for v in obj.values():
            vals.extend(extrair_strings_json(v))
    elif isinstance(obj, (list, tuple, set)):
        for v in obj:
            vals.extend(extrair_strings_json(v))
    return vals


def carregar_chaves_selecao(path):
    """Lê um JSON de seleção e retorna chaves por caminho, nome de arquivo e stem."""
    obj = carregar_json(path, default=None)
    strings = extrair_strings_json(obj)
    chaves = set()
    for s in strings:
        p = Path(str(s))
        chaves.add(str(s))
        chaves.add(p.name)
        chaves.add(p.stem)
    return chaves



def _json_zoomout_split(split):
    """Caminho do JSON que guarda quais imagens receberão zoom out em cada split."""
    return ZOOMOUT_TREINO_PATH if split == 'train' else ZOOMOUT_TESTE_PATH


def carregar_lista_zoomout(split):
    """Lista persistente das imagens marcadas para gerar cópia com zoom out aplicado."""
    obj = carregar_json(_json_zoomout_split(split), default=[])
    strings = extrair_strings_json(obj)
    # Remove duplicatas preservando a ordem.
    vistos = set()
    saida = []
    for s in strings:
        ss = str(s)
        if ss not in vistos:
            saida.append(ss)
            vistos.add(ss)
    return saida


def carregar_chaves_zoomout(split):
    """Lê a seleção persistente de imagens que receberão zoom out e retorna chaves de comparação."""
    strings = carregar_lista_zoomout(split)
    chaves = set()
    for s in strings:
        p = Path(str(s))
        chaves.add(str(s))
        chaves.add(str(p))
        chaves.add(p.name)
        chaves.add(p.stem)
    return chaves


def carregar_zoomout_config():
    """Configuração global do zoom out artificial."""
    cfg = carregar_json(ZOOMOUT_CONFIG_PATH, default={})
    if not isinstance(cfg, dict):
        cfg = {}
    cfg.setdefault('zoom_out_pct', 0)
    return cfg


def salvar_zoomout_config(zoom_out_pct):
    """Salva o percentual global de zoom out."""
    salvar_json(ZOOMOUT_CONFIG_PATH, {
        'zoom_out_pct': int(zoom_out_pct),
        'atualizado_em': time.strftime('%Y-%m-%d %H:%M:%S')
    })


def obter_zoom_out_pct():
    """Retorna o percentual atual de zoom out, priorizando o estado do notebook."""
    state = globals().get('STATE', {})
    if isinstance(state, dict) and 'zoom_out_pct' in state:
        try:
            return int(state.get('zoom_out_pct', 0))
        except Exception:
            return 0
    return int(carregar_zoomout_config().get('zoom_out_pct', 0))


def _path_em_chaves(path, chaves):
    p = Path(str(path))
    return (str(path) in chaves) or (str(p) in chaves) or (p.name in chaves) or (p.stem in chaves)


def imagem_tem_zoomout_path(path, split=None):
    """Verifica se uma imagem original foi marcada para gerar versão com zoom out."""
    path = Path(str(path))
    # Evita aplicar zoom out duas vezes em cópias já geradas.
    if '__zoomout_' in path.stem:
        return False
    try:
        if ZOOMOUT_DATASET_DIR in path.parents:
            return False
    except Exception:
        pass

    splits = [split] if split in ['train', 'test'] else ['train', 'test']
    for sp in splits:
        if _path_em_chaves(path, carregar_chaves_zoomout(sp)):
            return True
    return False


def aplicar_zoom_out_imagem_e_bboxes(img_bgr, boxes=None, zoom_pct=0):
    """
    Aplica zoom out artificial mantendo o tamanho final da imagem.

    Exemplo: zoom_pct=30 reduz o conteúdo para 70% do tamanho original e
    preenche as margens resultantes com preto. As caixas em pixels são
    transformadas para a nova posição.
    """
    boxes = boxes or []
    pct = max(0, min(90, int(zoom_pct)))
    if pct <= 0:
        return img_bgr, list(boxes)

    h, w = img_bgr.shape[:2]
    scale = max(0.05, 1.0 - pct / 100.0)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    img_small = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros_like(img_bgr)

    x0 = (w - new_w) // 2
    y0 = (h - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = img_small

    boxes_zoom = []
    for bbox in boxes:
        x1, y1, x2, y2 = bbox
        nx1 = x0 + x1 * scale
        ny1 = y0 + y1 * scale
        nx2 = x0 + x2 * scale
        ny2 = y0 + y2 * scale
        boxes_zoom.append(clip_bbox(nx1, ny1, nx2, ny2, w, h))

    return canvas, boxes_zoom


def _zoomout_nome_arquivo(row, zoom_pct):
    p = Path(str(row['path']))
    return f"{p.stem}__zoomout_{int(zoom_pct):02d}{p.suffix.lower()}"


def caminhos_zoomout_para_row(row, zoom_pct):
    """Caminhos da cópia com zoom out aplicada para uma imagem do split."""
    split = row.get('split', 'train')
    nome_img = _zoomout_nome_arquivo(row, zoom_pct)
    stem = Path(nome_img).stem
    img_dir = ZOOMOUT_DATASET_DIR / split / 'images'
    label_dir = ZOOMOUT_DATASET_DIR / split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    return img_dir / nome_img, label_dir / f'{stem}.txt'


def escrever_bboxes_yolo(label_path, objetos, img_w, img_h):
    """Escreve bboxes em formato YOLO normalizado."""
    label_path = Path(label_path)
    label_path.parent.mkdir(parents=True, exist_ok=True)
    linhas = []
    for obj in objetos:
        cls = int(obj.get('class_id', 0))
        x1, y1, x2, y2 = obj['bbox']
        x1, y1, x2, y2 = clip_bbox(x1, y1, x2, y2, img_w, img_h)
        bw = max(0.0, x2 - x1)
        bh = max(0.0, y2 - y1)
        if bw < 1 or bh < 1:
            continue
        xc = ((x1 + x2) / 2.0) / img_w
        yc = ((y1 + y2) / 2.0) / img_h
        wn = bw / img_w
        hn = bh / img_h
        linhas.append(f"{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}")
    label_path.write_text('\n'.join(linhas) + ('\n' if linhas else ''), encoding='utf-8')


def criar_imagem_zoomout_salva(row, zoom_pct=None, sobrescrever=False):
    """
    Cria uma cópia da imagem com zoom out aplicado e uma label YOLO ajustada.

    As imagens originais não são sobrescritas. A cópia é salva em
    `ZOOMOUT_DATASET_DIR/<split>/images` e a label correspondente em
    `ZOOMOUT_DATASET_DIR/<split>/labels`.
    """
    pct = int(obter_zoom_out_pct() if zoom_pct is None else zoom_pct)
    if pct <= 0:
        return None, None

    img_out_path, label_out_path = caminhos_zoomout_para_row(row, pct)
    if img_out_path.exists() and label_out_path.exists() and not sobrescrever:
        return img_out_path, label_out_path

    img = ler_imagem_bgr(row['path'])
    h, w = img.shape[:2]
    objetos_orig = ler_bboxes_yolo(row['label_path'], w, h)
    bboxes_orig = [obj['bbox'] for obj in objetos_orig]
    img_zoom, bboxes_zoom = aplicar_zoom_out_imagem_e_bboxes(img, bboxes_orig, pct)

    ok = cv2.imwrite(str(img_out_path), img_zoom)
    if not ok:
        raise RuntimeError(f'Falha ao salvar imagem com zoom out: {img_out_path}')

    objetos_zoom = []
    for obj, bbox_zoom in zip(objetos_orig, bboxes_zoom):
        objetos_zoom.append({'class_id': int(obj.get('class_id', 0)), 'bbox': bbox_zoom})
    escrever_bboxes_yolo(label_out_path, objetos_zoom, w, h)

    return img_out_path, label_out_path


def row_com_zoomout_salvo(row, zoom_pct=None):
    """Retorna uma cópia da linha apontando para a imagem/label com zoom out salva."""
    pct = int(obter_zoom_out_pct() if zoom_pct is None else zoom_pct)
    if pct <= 0:
        return row

    # Se a linha já aponta para uma cópia gerada, não gera outra.
    p_atual = Path(str(row['path']))
    if '__zoomout_' in p_atual.stem:
        return row

    img_out_path, label_out_path = criar_imagem_zoomout_salva(row, pct, sobrescrever=False)
    if img_out_path is None:
        return row

    new_row = row.copy()
    new_row['orig_path'] = str(row['path'])
    new_row['orig_label_path'] = str(row.get('label_path', ''))
    new_row['path'] = str(img_out_path)
    new_row['label_path'] = str(label_out_path)
    new_row['arquivo'] = img_out_path.name
    new_row['stem'] = img_out_path.stem
    new_row['zoom_out_aplicado_salvo'] = True
    new_row['zoom_out_pct'] = pct
    return new_row


def aplicar_zoomout_salvo_df(df_split, split):
    """Substitui linhas marcadas por linhas que apontam para cópias com zoom out já salvas."""
    if df_split is None or len(df_split) == 0:
        return df_split
    pct = int(obter_zoom_out_pct())
    if pct <= 0:
        return df_split.copy()

    chaves = carregar_chaves_zoomout(split)
    if not chaves:
        return df_split.copy()

    rows = []
    for _, row in df_split.iterrows():
        if imagem_tem_zoomout_path(row['path'], split):
            rows.append(row_com_zoomout_salvo(row, pct))
        else:
            rows.append(row.copy())
    return pd.DataFrame(rows).reset_index(drop=True)


def salvar_dataset_zoomout_para_selecoes(df=None, splits=('train', 'test'), zoom_pct=None, sobrescrever=False):
    """Gera/atualiza no disco as cópias com zoom out das imagens marcadas nos JSONs."""
    if df is None:
        df = globals().get('STATE', {}).get('df_dataset', pd.DataFrame())
    if df is None or len(df) == 0:
        return []

    pct = int(obter_zoom_out_pct() if zoom_pct is None else zoom_pct)
    gerados = []
    if pct <= 0:
        salvar_json(ZOOMOUT_MANIFEST_PATH, {'zoom_out_pct': pct, 'gerados': [], 'atualizado_em': time.strftime('%Y-%m-%d %H:%M:%S')})
        return gerados

    for split in splits:
        chaves = carregar_chaves_zoomout(split)
        if not chaves:
            continue
        df_split = df[df['split'] == split].copy()
        for _, row in df_split.iterrows():
            if imagem_tem_zoomout_path(row['path'], split):
                img_out, lab_out = criar_imagem_zoomout_salva(row, pct, sobrescrever=sobrescrever)
                if img_out is not None:
                    gerados.append({
                        'split': split,
                        'orig_path': str(row['path']),
                        'orig_label_path': str(row.get('label_path', '')),
                        'zoom_img_path': str(img_out),
                        'zoom_label_path': str(lab_out),
                        'zoom_out_pct': pct
                    })

    salvar_json(ZOOMOUT_MANIFEST_PATH, {
        'zoom_out_pct': pct,
        'n_gerados': len(gerados),
        'gerados': gerados,
        'atualizado_em': time.strftime('%Y-%m-%d %H:%M:%S')
    })
    return gerados


def ler_imagem_row_com_zoomout(row, retornar_info=False):
    """Lê a imagem. Se a linha original foi marcada, usa a cópia salva com zoom out."""
    if imagem_tem_zoomout_path(row['path'], row.get('split')) and int(obter_zoom_out_pct()) > 0:
        row = row_com_zoomout_salvo(row)
    img = ler_imagem_bgr(row['path'])
    aplicar = bool(row.get('zoom_out_aplicado_salvo', False)) or ('__zoomout_' in Path(str(row['path'])).stem)
    info = {'zoom_out_aplicado': bool(aplicar), 'zoom_out_pct': int(row.get('zoom_out_pct', obter_zoom_out_pct() if aplicar else 0))}
    return (img, info) if retornar_info else img


def carregar_imagem_e_bboxes_row(row):
    """Lê imagem e bboxes YOLO. Se marcada, usa a cópia salva com zoom out e label já ajustada."""
    if imagem_tem_zoomout_path(row['path'], row.get('split')) and int(obter_zoom_out_pct()) > 0:
        row = row_com_zoomout_salvo(row)

    img = ler_imagem_bgr(row['path'])
    h, w = img.shape[:2]
    boxes_raw = ler_bboxes_yolo(row['label_path'], w, h)
    boxes = [b['bbox'] for b in boxes_raw]

    aplicar = bool(row.get('zoom_out_aplicado_salvo', False)) or ('__zoomout_' in Path(str(row['path'])).stem)
    return img, boxes, {'zoom_out_aplicado': bool(aplicar), 'zoom_out_pct': int(row.get('zoom_out_pct', obter_zoom_out_pct() if aplicar else 0))}


def aplicar_selecao_salva(df, split, usar_selecao=True):
    """Filtra o split usando a seleção manual persistente.

    Quando `usar_selecao=True`, os JSONs de seleção comandam quais imagens entram
    em treino/teste. Se o JSON ainda não existir, usa o split completo como fallback
    inicial. Se o JSON existir, mas nenhuma imagem casar com os nomes atuais, retorna
    vazio para deixar claro que a curadoria precisa ser revisada.
    """
    df_split = df[df['split'] == split].copy()
    if not usar_selecao:
        return aplicar_zoomout_salvo_df(df_split, split)

    sel_path = SELECAO_TREINO_PATH if split == 'train' else SELECAO_TESTE_PATH
    if not Path(sel_path).exists():
        return aplicar_zoomout_salvo_df(df_split, split)

    chaves = carregar_chaves_selecao(sel_path)
    if not chaves:
        return df_split.iloc[0:0].copy()

    mask = df_split.apply(
        lambda r: (str(r['path']) in chaves) or (r['arquivo'] in chaves) or (r['stem'] in chaves),
        axis=1
    )
    df_sel = df_split[mask].copy()
    return aplicar_zoomout_salvo_df(df_sel, split)


def limitar_df(df, max_n, random_state=42):
    max_n = int(max_n)
    if max_n <= 0 or len(df) <= max_n:
        return df
    return df.sample(n=max_n, random_state=int(random_state)).copy()


def gerar_amostras_de_split(df_split, p, split_nome='train', status_callback=None):
    """
    Gera amostras positivas e negativas a partir de um split YOLO.

    Positivos: crops das bboxes anotadas.
    Negativos: crops aleatórios com IoU baixo em relação às bboxes.
    """
    rng = np.random.default_rng(int(p['random_state']) + (0 if split_nome == 'train' else 999))

    X = []
    y = []
    erros = []
    info = []

    neg_por_pos = int(p['negativos_por_positivo'])
    neg_iou_max = float(p['neg_iou_max'])
    pos_margin = float(p['pos_margin_pct'])
    neg_attempts = int(p['neg_attempts'])

    # Tamanho reserva para negativos em imagens sem objeto.
    wh_positivos = []

    total = len(df_split)
    for k, (_, row) in enumerate(df_split.iterrows(), start=1):
        if status_callback is not None and (k == 1 or k % 20 == 0 or k == total):
            status_callback(f'{split_nome}: gerando amostras {k}/{total}')

        try:
            img, boxes, zoom_info = carregar_imagem_e_bboxes_row(row)
            h, w = img.shape[:2]

            for bbox in boxes:
                bbox_pos = expandir_bbox(bbox, pos_margin, w, h)
                crop = recortar_bbox(img, bbox_pos)
                if crop.size == 0:
                    continue
                X.append(extrair_hog_de_imagem(crop, p))
                y.append(1)
                info.append({'split': split_nome, 'tipo': 'pos', 'arquivo': row['arquivo'], 'bbox': bbox_pos, **zoom_info})
                wh_positivos.append((bbox_pos[2] - bbox_pos[0], bbox_pos[3] - bbox_pos[1]))

            # Negativos proporcionais aos positivos da imagem.
            # Se a imagem não tiver objetos, gera pelo menos 1 negativo com tamanho mediano.
            n_base = max(1, len(boxes))
            n_neg = neg_por_pos * n_base

            if wh_positivos:
                median_w = int(np.median([v[0] for v in wh_positivos]))
                median_h = int(np.median([v[1] for v in wh_positivos]))
            else:
                median_w = max(24, int(w * 0.12))
                median_h = max(48, int(h * 0.25))

            for _ in range(n_neg):
                # Usa tamanho parecido com caixas reais quando possível.
                if boxes:
                    ref = boxes[int(rng.integers(0, len(boxes)))]
                    target_w = ref[2] - ref[0]
                    target_h = ref[3] - ref[1]
                else:
                    target_w = median_w
                    target_h = median_h

                # pequena variação de escala para aumentar diversidade.
                scale = float(rng.uniform(0.80, 1.25))
                tw = int(target_w * scale)
                th = int(target_h * scale)

                escolhido = None
                for _try in range(neg_attempts):
                    cand = bbox_aleatorio(w, h, tw, th, rng)
                    if not boxes or max(iou_bbox(cand, b) for b in boxes) <= neg_iou_max:
                        escolhido = cand
                        break

                if escolhido is None:
                    continue

                crop = recortar_bbox(img, escolhido)
                if crop.size == 0:
                    continue
                X.append(extrair_hog_de_imagem(crop, p))
                y.append(0)
                info.append({'split': split_nome, 'tipo': 'neg', 'arquivo': row['arquivo'], 'bbox': escolhido, **zoom_info})

        except Exception as e:
            erros.append((row.get('path', ''), str(e)))

    if len(X) == 0:
        return np.empty((0, 0), dtype=np.float32), np.array([], dtype=int), erros, info

    return np.vstack(X), np.array(y, dtype=int), erros, info


def preparar_dados_treino_teste(df, p, status_callback=None):
    """Prepara X_train, X_test, y_train, y_test usando train/test reais quando possível."""
    usar_selecao = bool(p.get('usar_selecao_salva', True))
    rs = int(p['random_state'])

    df_train = aplicar_selecao_salva(df, 'train', usar_selecao=usar_selecao)
    df_test = aplicar_selecao_salva(df, 'test', usar_selecao=usar_selecao)

    df_train = limitar_df(df_train, int(p['max_train_images']), rs)
    df_test = limitar_df(df_test, int(p['max_test_images']), rs + 1)

    X_train_all, y_train_all, erros_train, info_train = gerar_amostras_de_split(
        df_train, p, split_nome='train', status_callback=status_callback
    )

    X_test_real, y_test_real, erros_test, info_test = gerar_amostras_de_split(
        df_test, p, split_nome='test', status_callback=status_callback
    )

    erros = erros_train + erros_test

    if len(y_train_all) < 4 or len(np.unique(y_train_all)) < 2:
        raise RuntimeError(
            'Amostras de treino insuficientes. Verifique se train/labels possui anotações YOLO válidas.'
        )

    usar_test_real = bool(p.get('usar_test_real', True))
    if usar_test_real and len(y_test_real) >= 2 and len(np.unique(y_test_real)) == 2:
        X_train, y_train = X_train_all, y_train_all
        X_test, y_test = X_test_real, y_test_real
        modo_eval = 'pasta_test'
    else:
        test_size = float(p['test_size'])
        X_train, X_test, y_train, y_test = train_test_split(
            X_train_all, y_train_all,
            test_size=test_size,
            random_state=rs,
            stratify=y_train_all
        )
        modo_eval = 'holdout_train'

    stats = {
        'modo_eval': modo_eval,
        'n_img_train': int(len(df_train)),
        'n_img_test': int(len(df_test)),
        'n_train': int(len(y_train)),
        'n_test': int(len(y_test)),
        'n_total': int(len(y_train) + len(y_test)),
        'n_pos_train': int((y_train == 1).sum()),
        'n_neg_train': int((y_train == 0).sum()),
        'n_pos_test': int((y_test == 1).sum()),
        'n_neg_test': int((y_test == 0).sum()),
        'n_pos_total': int((np.concatenate([y_train, y_test]) == 1).sum()),
        'n_neg_total': int((np.concatenate([y_train, y_test]) == 0).sum()),
        'n_erros': int(len(erros)),
    }

    return X_train, X_test, y_train, y_test, stats, erros, info_train + info_test


def assinatura_dataset(df):
    """Cria assinatura do dataset considerando imagens e arquivos de label."""
    if df is None or len(df) == 0:
        return 'dataset_vazio'

    rows = []
    for _, row in df.sort_values(['split', 'path']).iterrows():
        item = {
            'split': row.get('split'),
            'path': str(row.get('path')),
            'label_path': str(row.get('label_path')),
            'n_obj': int(row.get('n_obj', 0)),
        }
        for key in ['path', 'label_path']:
            pp = Path(item[key])
            if pp.exists():
                st = pp.stat()
                item[f'{key}_size'] = int(st.st_size)
                item[f'{key}_mtime'] = float(st.st_mtime)
            else:
                item[f'{key}_size'] = None
                item[f'{key}_mtime'] = None
        rows.append(item)

    # Também inclui os JSONs opcionais de seleção/zoom out, se existirem.
    for sel in [SELECAO_TREINO_PATH, SELECAO_TESTE_PATH, ZOOMOUT_TREINO_PATH, ZOOMOUT_TESTE_PATH, ZOOMOUT_CONFIG_PATH, ZOOMOUT_MANIFEST_PATH]:
        if sel.exists():
            st = sel.stat()
            rows.append({'selection_json': str(sel), 'size': int(st.st_size), 'mtime': float(st.st_mtime)})

    payload = json.dumps(rows, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def assinatura_treinamento(df, p):
    """Assinatura completa: dataset + parâmetros."""
    payload = {
        'versao_notebook': 'cilindros_hough_hog_svm_cylinders_yolo_v4_zoomout_salvo',
        'dataset': assinatura_dataset(df),
        'params': p,
        'zoom_out_pct': obter_zoom_out_pct()
    }
    payload_str = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload_str.encode('utf-8')).hexdigest()


def carregar_historico():
    if HISTORY_PATH.exists():
        return pd.read_csv(HISTORY_PATH)
    return pd.DataFrame()


def salvar_linha_historico(row):
    hist = carregar_historico()
    hist = pd.concat([hist, pd.DataFrame([row])], ignore_index=True)
    HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
    hist.to_csv(HISTORY_PATH, index=False)
    return hist


def abreviar_float(x, nd=4):
    try:
        return round(float(x), nd)
    except Exception:
        return x


In [9]:
# ============================================================
# 6. Estado compartilhado e atualização automática de visualizações
# ============================================================

STATE = {
    'data_path': DATASET_DIR,
    'df_dataset': pd.DataFrame(),
    'ref_path': None,
    'params': OrderedDict(),
    'renderers': OrderedDict(),
    'refresh_em_andamento': False,
    'setup_id': 'PADRAO',
    'setup_label': 'Setup padrão',
    'setup_modificado': False,
    'loading_setup': False,
    'setup_dropdown_updating': False,
    'zoom_out_pct': int(carregar_zoomout_config().get('zoom_out_pct', 0)),
    'validacao_random_seed': 42
}


def registrar_renderer(nome, out_widget, funcao_renderer):
    """Registra uma visualização posterior para atualização automática."""
    STATE['renderers'][nome] = {
        'out': out_widget,
        'fn': funcao_renderer
    }
    atualizar_renderer(nome)


def atualizar_renderer(nome):
    """Atualiza uma visualização específica."""
    if nome not in STATE['renderers']:
        return

    item = STATE['renderers'][nome]
    out = item['out']
    fn = item['fn']

    with out:
        clear_output(wait=True)
        try:
            fn()
        except Exception as e:
            print(f"Erro ao atualizar '{nome}': {e}")
            traceback.print_exc(limit=1)


def atualizar_todas_visualizacoes(*args, **kwargs):
    """Atualiza todas as células posteriores que registraram visualização."""
    if STATE['refresh_em_andamento']:
        return

    STATE['refresh_em_andamento'] = True

    try:
        for nome in list(STATE['renderers'].keys()):
            atualizar_renderer(nome)
    finally:
        STATE['refresh_em_andamento'] = False


def obter_imagem_referencia():
    ref_path = STATE.get('ref_path')
    if ref_path is None:
        return None

    df = STATE.get('df_dataset', pd.DataFrame())
    if df is not None and len(df) > 0:
        row_df = df[df['path'].astype(str) == str(ref_path)]
        if len(row_df) > 0:
            return ler_imagem_row_com_zoomout(row_df.iloc[0])

    return ler_imagem_bgr(ref_path)


def obter_preprocessamento_referencia():
    img = obter_imagem_referencia()
    if img is None:
        return None
    return aplicar_preprocessamento(img, STATE['params'])


def definir_params_a_partir_widgets(widgets_param):
    """Copia os valores dos widgets para STATE['params']."""
    p = OrderedDict()
    for nome, wid in widgets_param.items():
        p[nome] = wid.value
    STATE['params'] = p


In [10]:
# ============================================================
# 7. Seleção manual persistente antes dos ajustes paramétricos
# ============================================================

# Esta célula deve ser executada antes da célula de parâmetros.
# Ela carrega o dataset, mantém a curadoria manual de treino/teste,
# permite marcar imagens que terão cópias salvas com zoom out artificial e restringe
# a imagem de referência às imagens selecionadas.

w_data_path = widgets.Text(
    value=str(DATASET_DIR),
    description='Dataset:',
    layout=widgets.Layout(width='90%')
)

btn_recarregar_dataset = widgets.Button(
    description='Recarregar dataset',
    button_style='info',
    icon='refresh',
    layout=widgets.Layout(width='200px')
)

w_ref_image = widgets.Dropdown(
    options=[],
    description='Referência:',
    layout=widgets.Layout(width='90%')
)

# Percentual global aplicado às imagens marcadas para gerar cópias com zoom out.
_zoom_cfg = carregar_zoomout_config()
w_zoom_out_pct = widgets.IntSlider(
    value=int(_zoom_cfg.get('zoom_out_pct', 0)),
    min=0,
    max=80,
    step=5,
    description='zoom out %',
    continuous_update=False,
    layout=widgets.Layout(width='420px'),
    style={'description_width': '95px'}
)
STATE['zoom_out_pct'] = int(w_zoom_out_pct.value)

THUMB_CACHE = {}
GRID_SELECTION_STATE = {
    'select_checkboxes': {},
    'zoom_checkboxes': {},
    'paths': [],
    'page': 1,
    'visible_paths': set(),
}

PAGE_SIZE_CURADORIA = 200

split_curadoria_widget = widgets.Dropdown(
    options=[('Treino', 'train'), ('Teste', 'test')],
    value='train',
    description='conjunto:',
    layout=widgets.Layout(width='220px')
)

btn_exibir_grade = widgets.Button(
    description='Exibir/atualizar grade',
    button_style='info',
    icon='th',
    layout=widgets.Layout(width='210px')
)

btn_pagina_anterior = widgets.Button(
    description='◀ Página anterior',
    button_style='',
    layout=widgets.Layout(width='160px')
)

w_pagina_curadoria = widgets.BoundedIntText(
    value=1,
    min=1,
    max=1,
    step=1,
    description='página:',
    layout=widgets.Layout(width='145px'),
    style={'description_width': '55px'}
)

lbl_total_paginas = widgets.HTML('<span style="opacity:.8;">/ 1</span>')

btn_pagina_proxima = widgets.Button(
    description='Próxima página ▶',
    button_style='',
    layout=widgets.Layout(width='160px')
)

btn_salvar_grade = widgets.Button(
    description='Salvar seleção',
    button_style='success',
    icon='save',
    layout=widgets.Layout(width='160px')
)

btn_marcar_todas_visiveis = widgets.Button(
    description='Marcar todas',
    button_style='',
    layout=widgets.Layout(width='135px')
)

btn_desmarcar_todas_visiveis = widgets.Button(
    description='Desmarcar todas',
    button_style='warning',
    layout=widgets.Layout(width='150px')
)

btn_limpar_zoom_visivel = widgets.Button(
    description='Limpar zoom',
    button_style='',
    layout=widgets.Layout(width='125px')
)

status_curadoria = widgets.HTML('')
out_curadoria_resumo = widgets.Output()
out_ref_preview = widgets.Output(layout=widgets.Layout(width='100%'))
out_zoom_preview = widgets.Output(layout=widgets.Layout(width='100%'))
out_grade_curadoria = widgets.Output()


def _nome_split_curadoria(split):
    return 'treino' if split == 'train' else 'teste'


def _json_split_curadoria(split):
    return SELECAO_TREINO_PATH if split == 'train' else SELECAO_TESTE_PATH


def _json_zoomout_curadoria(split):
    return ZOOMOUT_TREINO_PATH if split == 'train' else ZOOMOUT_TESTE_PATH



def _salvar_zoomout_split(split, paths):
    """Salva múltiplas imagens marcadas para gerar cópias com zoom out no split."""
    paths = [str(p) for p in paths]
    salvar_json(_json_zoomout_curadoria(split), paths)


def _chaves_zoomout_split(split):
    return carregar_chaves_zoomout(split)


def _lista_zoomout_split(split):
    return carregar_lista_zoomout(split)


def _df_split_completo(split):
    df = STATE.get('df_dataset', pd.DataFrame())
    if df is None or len(df) == 0:
        return pd.DataFrame()
    return df[df['split'] == split].copy().sort_values('arquivo').reset_index(drop=True)


def _chaves_selecionadas_split(split):
    return carregar_chaves_selecao(_json_split_curadoria(split))


def _chaves_zoomout_split(split):
    return carregar_chaves_zoomout(split)


def _linha_esta_em_chaves(row, chaves):
    return (
        str(row['path']) in chaves
        or str(Path(row['path'])) in chaves
        or row['arquivo'] in chaves
        or row['stem'] in chaves
    )


def _linha_esta_selecionada(row, chaves):
    return _linha_esta_em_chaves(row, chaves)


def _linha_esta_zoomout(row, chaves):
    return _linha_esta_em_chaves(row, chaves)


def df_por_selecao_persistente(df, split):
    """Retorna apenas as imagens selecionadas no JSON persistente daquele split."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=df.columns if df is not None else [])
    return aplicar_selecao_salva(df, split, usar_selecao=True)


def df_referencia_por_selecao(df):
    """A imagem de referência deve vir somente da seleção manual de treino/teste."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=df.columns if df is not None else [])

    partes = []
    for split in ['train', 'test']:
        sel = df_por_selecao_persistente(df, split)
        if sel is not None and len(sel) > 0:
            partes.append(sel)

    if not partes:
        return pd.DataFrame(columns=df.columns)

    return pd.concat(partes, ignore_index=True).sort_values(['split', 'arquivo']).reset_index(drop=True)


def _thumb_bytes(img_path, size=96):
    img_path = str(img_path)
    if img_path in THUMB_CACHE:
        return THUMB_CACHE[img_path]

    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    escala = size / max(h, w)
    novo_w = max(1, int(w * escala))
    novo_h = max(1, int(h * escala))
    thumb = cv2.resize(img, (novo_w, novo_h), interpolation=cv2.INTER_AREA)

    canvas = np.full((size, size, 3), 245, dtype=np.uint8)
    y0 = (size - novo_h) // 2
    x0 = (size - novo_w) // 2
    canvas[y0:y0 + novo_h, x0:x0 + novo_w] = thumb

    ok, buf = cv2.imencode('.jpg', cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 82])
    if not ok:
        return None

    data = bytes(buf)
    THUMB_CACHE[img_path] = data
    return data


def _criar_card_curadoria(row, marcada, zoom_marcado):
    path = row['path']

    thumb = _thumb_bytes(path)
    if thumb is None:
        img_widget = widgets.HTML(
            "<div style='width:96px;height:96px;display:flex;align-items:center;justify-content:center;border:1px solid #777;'>erro</div>"
        )
    else:
        img_widget = widgets.Image(
            value=thumb,
            format='jpg',
            layout=widgets.Layout(width='96px', height='96px')
        )

    cb_select = widgets.Checkbox(
        value=bool(marcada),
        description='',
        indent=False,
        layout=widgets.Layout(width='24px', height='22px', margin='0')
    )
    cb_zoom = widgets.Checkbox(
        value=bool(zoom_marcado),
        description='',
        indent=False,
        layout=widgets.Layout(width='24px', height='22px', margin='0')
    )

    path_str = str(path)
    GRID_SELECTION_STATE['select_checkboxes'][path_str] = cb_select
    GRID_SELECTION_STATE['zoom_checkboxes'][path_str] = cb_zoom

    def _on_select_change(change):
        if change.get('name') == 'value':
            renderizar_preview_zoom_out()

    def _on_zoom_change(change):
        if change.get('name') != 'value':
            return
        # A miniatura da grade permanece original. Apenas a prévia abaixo é recalculada.
        renderizar_preview_zoom_out()

    cb_select.observe(_on_select_change, names='value')
    cb_zoom.observe(_on_zoom_change, names='value')

    labels = widgets.HTML(
        "<div style='font-size:9px; line-height:11px; width:96px; display:flex; justify-content:space-around; opacity:.85;'>"
        "<span>seleção</span><span>zoom out</span></div>"
    )

    card_border = '2px solid #2a7' if marcada else '1px solid #555'
    if zoom_marcado:
        card_border = '2px solid #d98200'

    return widgets.VBox(
        [
            img_widget,
            widgets.HBox(
                [cb_select, cb_zoom],
                layout=widgets.Layout(justify_content='space-around', width='96px', height='24px')
            ),
            labels
        ],
        layout=widgets.Layout(
            width='112px',
            min_width='112px',
            max_width='112px',
            height='142px',
            overflow='hidden',
            align_items='center',
            border=card_border,
            border_radius='6px',
            padding='3px',
            margin='2px'
        )
    )


def _atualizar_dropdown_referencia():
    df = STATE.get('df_dataset', pd.DataFrame())
    df_ref = df_referencia_por_selecao(df)

    opcoes = []
    for _, row in df_ref.iterrows():
        opcoes.append((f"{row['split']} | {row['arquivo']} [{row['n_obj']} obj]", row['path']))

    atual = STATE.get('ref_path')
    valores = [v for _, v in opcoes]
    w_ref_image.options = opcoes

    if valores:
        if atual in valores:
            w_ref_image.value = atual
        else:
            STATE['ref_path'] = valores[0]
            w_ref_image.value = valores[0]
    else:
        STATE['ref_path'] = None
        try:
            w_ref_image.value = None
        except Exception:
            pass


def atualizar_resumo_curadoria(mensagem=''):
    with out_curadoria_resumo:
        clear_output(wait=True)

        df = STATE.get('df_dataset', pd.DataFrame())
        if df is None or len(df) == 0:
            print('Dataset ainda não carregado.')
            return

        train_sel = df_por_selecao_persistente(df, 'train')
        test_sel = df_por_selecao_persistente(df, 'test')
        df_ref = df_referencia_por_selecao(df)
        z_train = len(carregar_lista_zoomout('train'))
        z_test = len(carregar_lista_zoomout('test'))

        if mensagem:
            print(mensagem)
            print()

        print(resumo_dataset(df))
        print()
        print('Curadoria persistente em uso:')
        print(f'- treino selecionado: {len(train_sel)} imagens | JSON: {SELECAO_TREINO_PATH.exists()}')
        print(f'- teste selecionado : {len(test_sel)} imagens | JSON: {SELECAO_TESTE_PATH.exists()}')
        print(f'- imagens marcadas para zoom out: treino={z_train} | teste={z_test} | percentual={STATE.get("zoom_out_pct", 0)}%')
        print(f'- referências disponíveis: {len(df_ref)} imagens selecionadas')
        print()
        print('Arquivos JSON:')
        print('-', SELECAO_TREINO_PATH)
        print('-', SELECAO_TESTE_PATH)
        print('-', ZOOMOUT_TREINO_PATH)
        print('-', ZOOMOUT_TESTE_PATH)
        print('-', ZOOMOUT_CONFIG_PATH)
        if len(df_ref) == 0:
            print('\nAtenção: nenhuma imagem selecionada foi encontrada. Marque imagens na grade e salve a seleção.')


def renderizar_preview_referencia():
    """Função mantida apenas por compatibilidade; a prévia duplicada da referência não é exibida."""
    with out_ref_preview:
        clear_output(wait=True)


def _row_para_preview_zoom():
    # Prioridade: primeira imagem marcada com zoom na grade atual.
    for path, cb in GRID_SELECTION_STATE.get('zoom_checkboxes', {}).items():
        if cb.value:
            df = STATE.get('df_dataset', pd.DataFrame())
            row_df = df[df['path'].astype(str) == str(path)]
            if len(row_df) > 0:
                return row_df.iloc[0]

    # Depois, primeira imagem marcada para zoom out salva no split atual ou no outro split.
    df = STATE.get('df_dataset', pd.DataFrame())
    if df is not None and len(df) > 0:
        for sp in [split_curadoria_widget.value, 'train', 'test']:
            for zoom_path in carregar_lista_zoomout(sp):
                pz = Path(str(zoom_path))
                row_df = df[
                    (df['path'].astype(str) == str(zoom_path))
                    | (df['arquivo'].astype(str) == pz.name)
                    | (df['stem'].astype(str) == pz.stem)
                ]
                if len(row_df) > 0:
                    return row_df.iloc[0]

    # Depois, primeira imagem selecionada na grade atual.
    for path, cb in GRID_SELECTION_STATE.get('select_checkboxes', {}).items():
        if cb.value:
            df = STATE.get('df_dataset', pd.DataFrame())
            row_df = df[df['path'].astype(str) == str(path)]
            if len(row_df) > 0:
                return row_df.iloc[0]

    # Por fim, imagem de referência atual ou primeira imagem do split.
    ref_path = STATE.get('ref_path')
    df = STATE.get('df_dataset', pd.DataFrame())
    if ref_path and df is not None and len(df) > 0:
        row_df = df[df['path'].astype(str) == str(ref_path)]
        if len(row_df) > 0:
            return row_df.iloc[0]

    df_split = _df_split_completo(split_curadoria_widget.value)
    if len(df_split) > 0:
        return df_split.iloc[0]

    return None


def renderizar_preview_zoom_out(*args, **kwargs):
    with out_zoom_preview:
        clear_output(wait=True)

        row = _row_para_preview_zoom()
        if row is None:
            display(HTML('<em>Nenhuma imagem disponível para pré-visualizar o zoom out.</em>'))
            return

        try:
            img_bgr = ler_imagem_bgr(row['path'])
            img_zoom, _ = aplicar_zoom_out_imagem_e_bboxes(img_bgr, [], int(w_zoom_out_pct.value))
            img_rgb = bgr_para_rgb(img_bgr)
            img_zoom_rgb = bgr_para_rgb(img_zoom)
        except Exception as e:
            print('Erro ao gerar preview de zoom out:', e)
            return

        fig, axs = plt.subplots(1, 2, figsize=(5.4, 2.7))
        axs[0].imshow(img_rgb)
        axs[0].set_title('Original', fontsize=9)
        axs[1].imshow(img_zoom_rgb)
        axs[1].set_title(f'Zoom out {int(w_zoom_out_pct.value)}%', fontsize=9)
        for ax in axs:
            ax.axis('off')
        plt.tight_layout()
        plt.show()


def recarregar_dataset(_=None):
    STATE['data_path'] = Path(w_data_path.value).expanduser().resolve()
    STATE['zoom_out_pct'] = int(w_zoom_out_pct.value)
    df = listar_imagens_dataset(STATE['data_path'])
    STATE['df_dataset'] = df

    _atualizar_dropdown_referencia()
    atualizar_resumo_curadoria('Dataset/seleção recarregados.')
    renderizar_preview_zoom_out()
    atualizar_todas_visualizacoes()


def on_ref_image_change(change):
    if change.get('new') is not None:
        STATE['ref_path'] = change['new']
        atualizar_todas_visualizacoes()


def on_zoom_out_pct_change(change):
    if change.get('name') == 'value':
        STATE['zoom_out_pct'] = int(change['new'])
        salvar_zoomout_config(STATE['zoom_out_pct'])
        atualizar_resumo_curadoria('Percentual de zoom out atualizado.')
        renderizar_preview_zoom_out()
        atualizar_todas_visualizacoes()


def _total_paginas_curadoria(split):
    df_split = _df_split_completo(split)
    if df_split is None or len(df_split) == 0:
        return 1
    return max(1, int(np.ceil(len(df_split) / PAGE_SIZE_CURADORIA)))


def _sincronizar_controle_pagina(split=None, manter=True):
    split = split or split_curadoria_widget.value
    total = _total_paginas_curadoria(split)
    pagina_atual = int(GRID_SELECTION_STATE.get('page', 1)) if manter else 1
    pagina_atual = max(1, min(pagina_atual, total))
    GRID_SELECTION_STATE['page'] = pagina_atual
    w_pagina_curadoria.max = total
    w_pagina_curadoria.value = pagina_atual
    lbl_total_paginas.value = f'<span style="opacity:.8;">/ {total}</span>'
    btn_pagina_anterior.disabled = pagina_atual <= 1
    btn_pagina_proxima.disabled = pagina_atual >= total
    return pagina_atual, total


def renderizar_grade_curadoria(_=None):
    split = split_curadoria_widget.value
    df_split = _df_split_completo(split)

    GRID_SELECTION_STATE['select_checkboxes'] = {}
    GRID_SELECTION_STATE['zoom_checkboxes'] = {}
    GRID_SELECTION_STATE['paths'] = []
    GRID_SELECTION_STATE['visible_paths'] = set()

    pagina, total_paginas = _sincronizar_controle_pagina(split, manter=True)
    ini = (pagina - 1) * PAGE_SIZE_CURADORIA
    fim = ini + PAGE_SIZE_CURADORIA
    df_pagina = df_split.iloc[ini:fim].copy() if df_split is not None else pd.DataFrame()

    btn_exibir_grade.disabled = True
    btn_salvar_grade.disabled = True
    status_curadoria.value = "<span style='color:#d98200; font-weight:bold;'>Carregando miniaturas da página...</span>"

    try:
        with out_grade_curadoria:
            clear_output(wait=True)

            if df_split is None or len(df_split) == 0:
                print(f"Nenhuma imagem encontrada no split {_nome_split_curadoria(split)}.")
                return

            chaves_sel = _chaves_selecionadas_split(split)
            chaves_zoom = _chaves_zoomout_split(split)
            cards = []

            for _, row in df_pagina.iterrows():
                marcada = _linha_esta_selecionada(row, chaves_sel)
                zoom_marcado = _linha_esta_zoomout(row, chaves_zoom)
                cards.append(_criar_card_curadoria(row, marcada, zoom_marcado))
                path_str = str(row['path'])
                GRID_SELECTION_STATE['paths'].append(path_str)
                GRID_SELECTION_STATE['visible_paths'].add(path_str)

            n_marcadas_visiveis = sum(cb.value for cb in GRID_SELECTION_STATE['select_checkboxes'].values())
            n_zoom_visivel = sum(cb.value for cb in GRID_SELECTION_STATE['zoom_checkboxes'].values())
            n_zoom_split = len(carregar_lista_zoomout(split))

            display(widgets.HTML(
                f"<b>Curadoria de {_nome_split_curadoria(split)}</b> | "
                f"página {pagina}/{total_paginas} | exibindo {len(df_pagina)} de {len(df_split)} imagens | "
                f"{n_marcadas_visiveis} selecionadas nesta página | "
                f"{n_zoom_visivel} com zoom nesta página | {n_zoom_split} com zoom salvo no split<br>"
                "<small>São exibidas 200 imagens por página. A miniatura permanece original; "
                "a segunda caixa marca imagens para gerar <b>cópias salvas</b> com zoom out aplicado. "
                "Apenas a prévia abaixo muda ao alterar o slider. Clique em <b>Salvar seleção</b> para persistir e gerar as cópias.</small>"
            ))

            grid = widgets.GridBox(
                cards,
                layout=widgets.Layout(
                    display='grid',
                    grid_template_columns='repeat(auto-fill, minmax(116px, 1fr))',
                    grid_gap='6px 4px',
                    width='100%',
                    overflow='visible',
                    align_items='flex-start'
                )
            )
            display(grid)

        renderizar_preview_zoom_out()

    finally:
        btn_exibir_grade.disabled = False
        btn_salvar_grade.disabled = False
        _sincronizar_controle_pagina(split, manter=True)
        status_curadoria.value = ''



def salvar_grade_curadoria(_=None):
    split = split_curadoria_widget.value

    if not GRID_SELECTION_STATE['select_checkboxes']:
        status_curadoria.value = "<span style='color:#b66;'>Exiba a grade antes de salvar alterações.</span>"
        return

    df = STATE.get('df_dataset', pd.DataFrame())
    df_split = _df_split_completo(split)
    visible_paths = set(GRID_SELECTION_STATE.get('visible_paths', set()))

    # Mantém a seleção das outras páginas e atualiza somente a página visível.
    chaves_sel_existentes = _chaves_selecionadas_split(split)
    selecionadas_anteriores = []
    for _, row in df_split.iterrows():
        path_str = str(row['path'])
        if path_str in visible_paths:
            continue
        if _linha_esta_selecionada(row, chaves_sel_existentes):
            selecionadas_anteriores.append(path_str)

    selecionadas_visiveis = [
        path for path, checkbox in GRID_SELECTION_STATE['select_checkboxes'].items()
        if checkbox.value
    ]
    selecionadas = selecionadas_anteriores + selecionadas_visiveis

    # Zoom out: agora podem existir várias imagens marcadas. Mantém outras páginas e atualiza a página visível.
    chaves_zoom_existentes = _chaves_zoomout_split(split)
    zoom_anteriores = []
    for _, row in df_split.iterrows():
        path_str = str(row['path'])
        if path_str in visible_paths:
            continue
        if _linha_esta_zoomout(row, chaves_zoom_existentes):
            zoom_anteriores.append(path_str)

    zoom_visiveis = [
        path for path, checkbox in GRID_SELECTION_STATE['zoom_checkboxes'].items()
        if checkbox.value
    ]
    zoom_marcadas = zoom_anteriores + zoom_visiveis

    salvar_json(_json_split_curadoria(split), selecionadas)
    _salvar_zoomout_split(split, zoom_marcadas)
    salvar_zoomout_config(int(w_zoom_out_pct.value))

    # Gera as cópias com zoom out no dataset derivado. As miniaturas originais da grade não são alteradas.
    status_curadoria.value = "<span style='color:#d98200;'>Salvando seleção e gerando cópias com zoom out...</span>"
    gerados = salvar_dataset_zoomout_para_selecoes(df, splits=('train', 'test'), zoom_pct=int(w_zoom_out_pct.value), sobrescrever=True)

    recarregar_dataset()
    renderizar_grade_curadoria()

    atualizar_resumo_curadoria(
        f"Seleção de {_nome_split_curadoria(split)} salva. "
        f"Total selecionado no split: {len(selecionadas)} imagens. "
        f"Imagens marcadas para zoom out neste split: {len(zoom_marcadas)}. "
        f"Cópias com zoom out geradas/atualizadas: {len(gerados)}."
    )


def marcar_todas_visiveis(_=None):
    for cb in GRID_SELECTION_STATE['select_checkboxes'].values():
        cb.value = True
    renderizar_preview_zoom_out()


def desmarcar_todas_visiveis(_=None):
    for cb in GRID_SELECTION_STATE['select_checkboxes'].values():
        cb.value = False
    renderizar_preview_zoom_out()


def limpar_zoom_visivel(_=None):
    for cb in GRID_SELECTION_STATE['zoom_checkboxes'].values():
        cb.value = False
    status_curadoria.value = "<span style='color:#d98200;'>Zoom removido da página visível. Clique em Salvar seleção para persistir.</span>"
    renderizar_preview_zoom_out()


def ao_mudar_split_curadoria(change):
    if change.get('name') == 'value':
        GRID_SELECTION_STATE['page'] = 1
        _sincronizar_controle_pagina(change.get('new'), manter=False)
        with out_grade_curadoria:
            clear_output(wait=True)
            print("Clique em 'Exibir/atualizar grade' para carregar a curadoria deste conjunto.")
        renderizar_preview_zoom_out()




btn_recarregar_dataset.on_click(recarregar_dataset)
w_ref_image.observe(on_ref_image_change, names='value')
w_zoom_out_pct.observe(on_zoom_out_pct_change, names='value')
btn_exibir_grade.on_click(renderizar_grade_curadoria)
btn_salvar_grade.on_click(salvar_grade_curadoria)
btn_marcar_todas_visiveis.on_click(marcar_todas_visiveis)
btn_desmarcar_todas_visiveis.on_click(desmarcar_todas_visiveis)
btn_limpar_zoom_visivel.on_click(limpar_zoom_visivel)
btn_pagina_anterior.on_click(lambda _: (_sincronizar_controle_pagina(split_curadoria_widget.value, manter=True), setattr(w_pagina_curadoria, 'value', max(1, int(w_pagina_curadoria.value) - 1)), GRID_SELECTION_STATE.update({'page': int(w_pagina_curadoria.value)}), renderizar_grade_curadoria()))
btn_pagina_proxima.on_click(lambda _: (_sincronizar_controle_pagina(split_curadoria_widget.value, manter=True), setattr(w_pagina_curadoria, 'value', min(int(w_pagina_curadoria.max), int(w_pagina_curadoria.value) + 1)), GRID_SELECTION_STATE.update({'page': int(w_pagina_curadoria.value)}), renderizar_grade_curadoria()))

def _on_pagina_manual_change(change):
    if change.get('name') == 'value':
        GRID_SELECTION_STATE['page'] = int(change.get('new', 1))
        with out_grade_curadoria:
            clear_output(wait=True)
            print("Clique em 'Exibir/atualizar grade' para carregar a página selecionada.")

w_pagina_curadoria.observe(_on_pagina_manual_change, names='value')
split_curadoria_widget.observe(ao_mudar_split_curadoria, names='value')

painel_curadoria = widgets.VBox([
    widgets.HTML('<b>1) Seleção manual persistente de imagens</b>'),
    widgets.HTML(
        '<small>Esta célula controla as imagens usadas em treino/teste, as imagens que terão cópias salvas com zoom out artificial e também alimenta a lista de imagem de referência. '
        'As imagens originais do dataset não são sobrescritas; as cópias com zoom out são gravadas no dataset derivado do notebook.</small>'
    ),
    w_data_path,
    widgets.HBox([btn_recarregar_dataset]),
    widgets.HBox(
        [
            split_curadoria_widget,
            btn_exibir_grade,
            btn_salvar_grade,
            btn_marcar_todas_visiveis,
            btn_desmarcar_todas_visiveis,
            btn_limpar_zoom_visivel,
            status_curadoria
        ],
        layout=widgets.Layout(align_items='center', gap='8px', flex_flow='row wrap')
    ),
    widgets.HBox(
        [btn_pagina_anterior, w_pagina_curadoria, lbl_total_paginas, btn_pagina_proxima],
        layout=widgets.Layout(align_items='center', gap='6px', flex_flow='row wrap')
    ),
    widgets.HTML('<b>Zoom out artificial para imagens marcadas</b>'),
    widgets.HTML(
        '<small>Use a segunda caixa de seleção de cada miniatura para marcar imagens que devem ficar aparentemente mais distantes. '
        'A miniatura da grade fica original; somente a prévia abaixo muda em tempo real. Ao salvar, o notebook grava uma cópia com zoom out aplicado e label YOLO ajustada.</small>'
    ),
    w_zoom_out_pct,
    out_zoom_preview,
    out_curadoria_resumo,
    widgets.HTML(
        '<small>A imagem de referência será escolhida abaixo da seção de parâmetros, '
        'usando somente as imagens marcadas nesta curadoria.</small>'
    ),
    out_grade_curadoria
])

display(painel_curadoria)

# Carrega dataset e seleção automaticamente. Não exibe listas de imagens e não treina.
recarregar_dataset()


## Núcleo da estratégia V2

A célula seguinte substitui as versões acumuladas do detector por uma lógica única baseada somente em pares de retas paralelas. As ROIs deixam de usar reta única e não dependem de busca por extremidades curvas. A BBox é calculada por proporção comprimento/largura, e o score geométrico combina overlap, similaridade angular e proximidade com uma razão L/W alvo.


In [11]:
# ============================================================
# 8A. Funções auxiliares de avaliação, NMS e métricas
# ============================================================
# Mantém somente o necessário para treinamento/validação.
# A antiga expansão X/Y e o antigo painel de refino não fazem parte desta versão.

def _pget(p, nome, default):
    return p[nome] if nome in p else default


def _bbox_valida(bbox):
    try:
        vals = [float(v) for v in bbox]
        return all(np.isfinite(v) for v in vals) and vals[2] > vals[0] and vals[3] > vals[1]
    except Exception:
        return False


def nms_bboxes_refinado(resultados_pos, iou_thr=0.30, max_det=8):
    if not resultados_pos:
        return []
    items = []
    for r in resultados_pos:
        if not _bbox_valida(r.get('bbox')):
            continue
        score = r.get('score_final_float', r.get('score_float', 0.0))
        if score is None:
            score = 0.0
        items.append((float(score), r))
    items.sort(key=lambda x: x[0], reverse=True)

    keep = []
    for _, r in items:
        if len(keep) >= int(max_det):
            break
        if all(iou_bbox(r['bbox'], k['bbox']) <= float(iou_thr) for k in keep):
            keep.append(r)
    return keep


def associar_predicoes_gt(pred_boxes, gt_boxes, iou_thr=0.50):
    """Associa predições a GT por IoU de forma gulosa."""
    pred_boxes = [tuple(map(float, b)) for b in pred_boxes if _bbox_valida(b)]
    gt_boxes = [tuple(map(float, b)) for b in gt_boxes if _bbox_valida(b)]
    pares = []
    for i, pb in enumerate(pred_boxes):
        for j, gb in enumerate(gt_boxes):
            pares.append((iou_bbox(pb, gb), i, j))
    pares.sort(reverse=True, key=lambda x: x[0])

    usados_p = set()
    usados_g = set()
    ious_tp = []
    for iou, i, j in pares:
        if iou < float(iou_thr):
            break
        if i in usados_p or j in usados_g:
            continue
        usados_p.add(i)
        usados_g.add(j)
        ious_tp.append(float(iou))

    tp = len(ious_tp)
    fp = max(0, len(pred_boxes) - tp)
    fn = max(0, len(gt_boxes) - tp)
    return {'tp': tp, 'fp': fp, 'fn': fn, 'ious_tp': ious_tp, 'n_pred': len(pred_boxes), 'n_gt': len(gt_boxes)}


def _metricas_det_agregadas(contadores, iou_thr):
    tp = int(contadores.get('tp', 0)); fp = int(contadores.get('fp', 0)); fn = int(contadores.get('fn', 0))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    ious = contadores.get('ious_tp', [])
    return {
        f'precision_det_{int(iou_thr*100)}': prec,
        f'recall_det_{int(iou_thr*100)}': rec,
        f'f1_det_{int(iou_thr*100)}': f1,
        f'tp_det_{int(iou_thr*100)}': tp,
        f'fp_det_{int(iou_thr*100)}': fp,
        f'fn_det_{int(iou_thr*100)}': fn,
        f'mean_iou_tp_{int(iou_thr*100)}': float(np.mean(ious)) if ious else 0.0,
    }


def avaliar_detector_completo(df, clf, p, status_callback=None):
    """Avalia o detector completo em imagens selecionadas de teste, comparando com YOLO por IoU."""
    if df is None or len(df) == 0 or clf is None:
        return {}

    usar_selecao = bool(_pget(p, 'usar_selecao_salva', True))
    df_eval = aplicar_selecao_salva(df, 'test', usar_selecao=usar_selecao)
    if df_eval is None or len(df_eval) == 0:
        df_eval = aplicar_selecao_salva(df, 'train', usar_selecao=usar_selecao)
    if df_eval is None or len(df_eval) == 0:
        return {}

    max_imgs = int(_pget(p, 'det_eval_max_images', 120))
    if max_imgs > 0 and len(df_eval) > max_imgs:
        df_eval = df_eval.sample(max_imgs, random_state=int(_pget(p, 'random_state', 42))).reset_index(drop=True)
    else:
        df_eval = df_eval.reset_index(drop=True)

    thr_main = float(_pget(p, 'det_iou_thr', 0.50))
    thrs = sorted(set([0.30, 0.50, round(thr_main, 2)]))
    cont = {thr: {'tp': 0, 'fp': 0, 'fn': 0, 'ious_tp': []} for thr in thrs}
    n_gt_total = 0
    n_pred_total = 0
    best_ious_all = []
    erros = 0

    for k, row in df_eval.iterrows():
        try:
            img_bgr, boxes_gt, _ = carregar_imagem_e_bboxes_row(row)
            det = detectar_candidatos_cilindro(
                img_bgr, clf=clf, p=p,
                score_min=float(_pget(p, 'det_score_min', 0.0)),
                nms_iou=float(_pget(p, 'det_nms_iou', 0.30)),
                max_det=int(_pget(p, 'det_max_det', 8)),
                aplicar_nms=True
            )
            pred_boxes = [r['bbox'] for r in det['positivos_finais'] if _bbox_valida(r.get('bbox'))]
            gt_boxes = [b for b in boxes_gt if _bbox_valida(b)]
            n_gt_total += len(gt_boxes)
            n_pred_total += len(pred_boxes)
            for pb in pred_boxes:
                best_ious_all.append(max([iou_bbox(pb, gb) for gb in gt_boxes], default=0.0))
            for thr in thrs:
                m = associar_predicoes_gt(pred_boxes, gt_boxes, iou_thr=thr)
                cont[thr]['tp'] += m['tp']; cont[thr]['fp'] += m['fp']; cont[thr]['fn'] += m['fn']; cont[thr]['ious_tp'].extend(m['ious_tp'])
        except Exception:
            erros += 1
        if status_callback is not None and (k + 1) % 25 == 0:
            status_callback(f'Avaliação detector completo: {k+1}/{len(df_eval)} imagens...')

    out = {
        'det_eval_imgs': int(len(df_eval)),
        'n_gt_det': int(n_gt_total),
        'n_pred_det': int(n_pred_total),
        'det_erros': int(erros),
        'det_iou_thr': float(thr_main),
        'mean_iou_best': float(np.mean(best_ious_all)) if best_ious_all else 0.0,
        'median_iou_best': float(np.median(best_ious_all)) if best_ious_all else 0.0,
    }
    for thr in thrs:
        out.update(_metricas_det_agregadas(cont[thr], thr))
    # aliases para a IoU principal escolhida no widget
    key = int(thr_main * 100)
    for prefix in ['precision_det', 'recall_det', 'f1_det', 'tp_det', 'fp_det', 'fn_det', 'mean_iou_tp']:
        out[prefix] = out.get(f'{prefix}_{key}', 0.0)
    return out


# ------------------------------------------------------------
# Visualizações atualizadas: ROI original, ROI final e pares de retas
# ------------------------------------------------------------
COLOR_LINE_CANDIDATE = (0, 220, 255)  # ciano: retas usadas para formar a ROI candidata


In [12]:
# ============================================================
# 8B. Núcleo V2: ROIs somente por pares de retas paralelas
# ============================================================
# Estratégia desta versão:
# 1) detectar retas por Hough;
# 2) unir segmentos colineares e filtrar retas por geometria + suporte real;
# 3) formar ROIs somente com pares de retas paralelas;
# 4) calcular a BBox a partir do par de retas, com margem fixa em pixels;
# 5) pontuar cada ROI por overlap, similaridade angular e proximidade da razão L/W ao alvo ajustável.

# ------------------------------------------------------------
# Geometria básica de retas e caixas orientadas
# ------------------------------------------------------------
def _line_info(line):
    x1, y1, x2, y2 = map(float, line)
    dx = x2 - x1
    dy = y2 - y1
    L = float(math.hypot(dx, dy))
    if L < 1e-6:
        return None
    ux, uy = dx / L, dy / L
    angle = (math.degrees(math.atan2(uy, ux)) + 180.0) % 180.0
    cx, cy = (x1 + x2) * 0.5, (y1 + y2) * 0.5
    return {
        'line': tuple(map(int, [round(x1), round(y1), round(x2), round(y2)])),
        'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
        'length': L,
        'u': np.array([ux, uy], dtype=float),
        'v': np.array([-uy, ux], dtype=float),
        'center': np.array([cx, cy], dtype=float),
        'angle': angle
    }


def _angle_diff_180(a, b):
    d = abs(float(a) - float(b)) % 180.0
    return min(d, 180.0 - d)


def _interval_gap(a1, a2, b1, b2):
    if a2 < b1:
        return b1 - a2
    if b2 < a1:
        return a1 - b2
    return 0.0


def _projecoes_linha_em_base(line, origin, u, v):
    pts = np.array([[line[0], line[1]], [line[2], line[3]]], dtype=float)
    rel = pts - origin.reshape(1, 2)
    ts = rel @ u
    ss = rel @ v
    return float(np.min(ts)), float(np.max(ts)), float(np.mean(ss))


def _bbox_axis_from_points(pts, shape, margin=0):
    h, w = shape[:2]
    pts = np.asarray(pts, dtype=float).reshape(-1, 2)
    if pts.size == 0:
        return None
    x1 = int(max(0, math.floor(np.min(pts[:, 0]) - margin)))
    y1 = int(max(0, math.floor(np.min(pts[:, 1]) - margin)))
    x2 = int(min(w, math.ceil(np.max(pts[:, 0]) + margin)))
    y2 = int(min(h, math.ceil(np.max(pts[:, 1]) + margin)))
    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None
    return (x1, y1, x2, y2)


def _oriented_corners(origin, u, v, tmin, tmax, smin, smax):
    pts = [
        origin + u * tmin + v * smin,
        origin + u * tmax + v * smin,
        origin + u * tmax + v * smax,
        origin + u * tmin + v * smax,
    ]
    return np.asarray(pts, dtype=np.float32)


def _bbox_center(bbox):
    if not _bbox_valida(bbox):
        return np.array([0.0, 0.0], dtype=float)
    x1, y1, x2, y2 = map(float, bbox)
    return np.array([(x1 + x2) * 0.5, (y1 + y2) * 0.5], dtype=float)


def _bbox_area(bbox):
    if not _bbox_valida(bbox):
        return 0.0
    x1, y1, x2, y2 = map(float, bbox)
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


# ------------------------------------------------------------
# União de segmentos e suporte real das retas
# ------------------------------------------------------------
def unir_segmentos_colineares(lines, p):
    """Une segmentos quase colineares antes de formar pares."""
    if not bool(_pget(p, 'merge_collinear_enabled', True)):
        return [tuple(map(int, l)) for l in lines]

    infos = [_line_info(l) for l in lines]
    infos = [i for i in infos if i is not None and i['length'] >= float(_pget(p, 'line_length_min', 0))]
    if not infos:
        return []

    gap_max = float(_pget(p, 'merge_gap_px', 18))
    angle_tol = float(_pget(p, 'merge_angle_tol_deg', 5.0))
    perp_tol = float(_pget(p, 'merge_perp_tol_px', 8.0))

    infos = sorted(infos, key=lambda d: d['length'], reverse=True)
    usados = set()
    merged = []

    for i, base in enumerate(infos):
        if i in usados:
            continue
        origin = base['center']
        u = base['u'].copy()
        v = base['v'].copy()
        t1, t2, _ = _projecoes_linha_em_base(base['line'], origin, u, v)
        intervalo_atual = [min(t1, t2), max(t1, t2)]
        grupo = [base]
        usados.add(i)

        changed = True
        while changed:
            changed = False
            for j, other in enumerate(infos):
                if j in usados:
                    continue
                if _angle_diff_180(base['angle'], other['angle']) > angle_tol:
                    continue
                ot1, ot2, os = _projecoes_linha_em_base(other['line'], origin, u, v)
                omin, omax = min(ot1, ot2), max(ot1, ot2)
                if abs(os) > perp_tol:
                    continue
                if _interval_gap(intervalo_atual[0], intervalo_atual[1], omin, omax) > gap_max:
                    continue
                usados.add(j)
                grupo.append(other)
                intervalo_atual[0] = min(intervalo_atual[0], omin)
                intervalo_atual[1] = max(intervalo_atual[1], omax)
                changed = True

        t_vals, s_vals = [], []
        for g in grupo:
            gt1, gt2, gs = _projecoes_linha_em_base(g['line'], origin, u, v)
            t_vals.extend([gt1, gt2])
            s_vals.append(gs)
        tmin, tmax = float(np.min(t_vals)), float(np.max(t_vals))
        smed = float(np.mean(s_vals)) if s_vals else 0.0
        p1 = origin + u * tmin + v * smed
        p2 = origin + u * tmax + v * smed
        merged.append(tuple(map(int, [round(p1[0]), round(p1[1]), round(p2[0]), round(p2[1])])))

    return merged


def filtrar_linhas_remanescentes(lines, p):
    """Filtra linhas por comprimento. A orientação fica livre para captar cilindros inclinados."""
    out = []
    Lmin = float(_pget(p, 'line_length_min', 0))
    Lmax = float(_pget(p, 'line_length_max', 999999))
    for line in lines:
        info = _line_info(line)
        if info is None:
            continue
        if info['length'] < Lmin or info['length'] > Lmax:
            continue
        out.append(tuple(map(int, line)))
    return out


def _edge_has_support(edge_map, x, y, radius=2):
    if edge_map is None:
        return False
    h, w = edge_map.shape[:2]
    xi = int(round(x)); yi = int(round(y))
    r = int(max(0, radius))
    if xi < 0 or yi < 0 or xi >= w or yi >= h:
        return False
    x1 = max(0, xi - r); x2 = min(w, xi + r + 1)
    y1 = max(0, yi - r); y2 = min(h, yi + r + 1)
    return bool(np.any(edge_map[y1:y2, x1:x2] > 0))


def _max_run_bool(vals):
    best = 0; cur = 0
    for v in vals:
        if bool(v):
            cur += 1; best = max(best, cur)
        else:
            cur = 0
    return best


def avaliar_suporte_linha(line, pre, p):
    """Verifica se a reta tem pixels de borda acompanhando seu comprimento."""
    info = _line_info(line)
    if info is None:
        return {'ok': False, 'ratio': 0.0, 'max_gap_px': 9999.0}

    source = _pget(p, 'line_support_source', 'canny_hough')
    edge_map = pre.get(source, pre.get('canny_hough', pre.get('canny_full')))
    if edge_map is None:
        return {'ok': True, 'ratio': 1.0, 'max_gap_px': 0.0}

    radius = int(_pget(p, 'line_support_radius_px', 2))
    min_ratio = float(_pget(p, 'line_support_min_ratio', 0.30))
    max_gap_px = float(_pget(p, 'line_support_max_gap_px', 30))
    L = max(1.0, float(info['length']))
    step_px = 3.0
    n = int(np.clip(math.ceil(L / step_px) + 1, 8, 240))
    tvals = np.linspace(-L * 0.5, L * 0.5, n)
    pts = info['center'].reshape(1, 2) + tvals.reshape(-1, 1) * info['u'].reshape(1, 2)

    support = np.asarray([_edge_has_support(edge_map, x, y, radius=radius) for x, y in pts], dtype=bool)
    ratio = float(np.mean(support)) if len(support) else 0.0
    gap_px = float(_max_run_bool(~support) * step_px) if len(support) else 9999.0
    ok = (ratio >= min_ratio) and (gap_px <= max_gap_px or max_gap_px <= 0)
    return {'ok': bool(ok), 'ratio': ratio, 'max_gap_px': gap_px, 'n_samples': int(n)}


def filtrar_linhas_por_suporte(lines, pre, p):
    """Remove retas que não têm suporte real de borda."""
    if not bool(_pget(p, 'line_support_enabled', True)):
        return list(lines), {'n_linhas_suporte_testadas': int(len(lines)), 'n_linhas_suporte_rejeitadas': 0, 'line_support_debug': []}

    filtradas, debug = [], []
    for line in lines:
        info = avaliar_suporte_linha(line, pre, p)
        debug.append({'line': tuple(map(int, line)), **info})
        if info.get('ok', False):
            filtradas.append(tuple(map(int, line)))
    return filtradas, {
        'n_linhas_suporte_testadas': int(len(lines)),
        'n_linhas_suporte_rejeitadas': int(len(lines) - len(filtradas)),
        'line_support_debug': debug
    }


# ------------------------------------------------------------
# Pares paralelos: única fonte de ROIs
# ------------------------------------------------------------
def _score_lw_ratio(ratio, target, tol_pct):
    """Pontua a razão comprimento/largura da ROI em torno de um valor alvo."""
    ratio = float(ratio)
    target = max(1e-6, float(target))
    tol_pct = max(1e-6, float(tol_pct))
    erro_rel = abs(ratio - target) / target
    return float(max(0.0, 1.0 - min(1.0, erro_rel / tol_pct)))


def _normalizar_pesos_score(p):
    w_overlap = max(0.0, float(_pget(p, 'pair_score_overlap_weight', 0.45)))
    w_angle = max(0.0, float(_pget(p, 'pair_score_angle_weight', 0.30)))
    w_ratio = max(0.0, float(_pget(p, 'pair_score_ratio_weight', 0.25)))
    s = w_overlap + w_angle + w_ratio
    if s <= 1e-9:
        return 0.45, 0.30, 0.25
    return w_overlap / s, w_angle / s, w_ratio / s


def _score_par_v2(overlap, angle_diff, pair_tol, lw_ratio, p):
    """Score geométrico V2 baseado em overlap, paralelismo e razão comprimento/largura."""
    overlap_score = float(np.clip(overlap, 0.0, 1.0))
    angle_score = 1.0 - min(1.0, float(angle_diff) / max(1e-6, float(pair_tol)))
    ratio_score = _score_lw_ratio(
        lw_ratio,
        _pget(p, 'pair_lw_ratio_target', 5.0),
        _pget(p, 'pair_lw_ratio_tol_pct', 0.60),
    )
    w_overlap, w_angle, w_ratio = _normalizar_pesos_score(p)
    score = w_overlap * overlap_score + w_angle * angle_score + w_ratio * ratio_score
    return float(np.clip(score, 0.0, 1.0)), {
        'score_overlap': float(overlap_score),
        'score_angle': float(angle_score),
        'score_lw_ratio': float(ratio_score),
        'lw_ratio': float(lw_ratio),
        'lw_target': float(_pget(p, 'pair_lw_ratio_target', 5.0)),
    }


def _bbox_par_por_margem_fixa(origin, u, v, tmin, tmax, smin, smax, img_shape, p):
    """
    Calcula a BBox do par usando somente as extremidades das duas retas.

    Não há expansão por arcos, reta única ou proporção L/W. A razão L/W continua
    sendo usada apenas para pontuar a qualidade geométrica do par. A caixa final
    contém apenas o retângulo orientado definido pelo par de retas, acrescido de
    uma margem fixa em pixels (`roi_margin_px`).
    """
    width = max(1.0, float(smax - smin))
    base_len = max(1.0, float(tmax - tmin))
    margin_px = int(_pget(p, 'roi_margin_px', 10))

    # Retângulo orientado estritamente definido pelas extremidades do par.
    corners = _oriented_corners(
        origin,
        u,
        v,
        float(tmin),
        float(tmax),
        float(smin),
        float(smax)
    )

    # A margem é fixa em pixels e aplicada apenas ao converter para BBox no eixo da imagem.
    bbox = _bbox_axis_from_points(corners, img_shape, margin=margin_px)

    return bbox, corners, {
        'tmin': float(tmin),
        'tmax': float(tmax),
        'smin': float(smin),
        'smax': float(smax),
        'base_len': float(base_len),
        'width': float(width),
        'fixed_margin_px': int(margin_px),
        'lw_ratio_base': float(base_len / max(1.0, width)),
        'lw_ratio_bbox': float(base_len / max(1.0, width)),
    }


def gerar_pares_orientados(lines, img_shape, p, return_debug=False):
    """Gera ROIs iniciais somente a partir de pares de retas paralelas."""
    infos = [_line_info(l) for l in lines]
    infos = [i for i in infos if i is not None]
    candidatos = []
    dbg = {
        'n_linhas_para_pares': len(infos),
        'n_pares_testados': 0,
        'n_pares_ang_ok': 0,
        'n_pares_dist_ok': 0,
        'n_pares_overlap_ok': 0,
        'n_candidatos_par': 0,
    }

    pair_tol = float(_pget(p, 'pair_angle_tol_deg', 3.0))
    dist_min = float(_pget(p, 'pair_dist_min', 30))
    dist_max = float(_pget(p, 'pair_dist_max', 110))
    overlap_min = float(_pget(p, 'pair_overlap_min', 0.60))
    axis_gap_max = float(_pget(p, 'pair_axis_gap_px', 10))
    max_rois = int(_pget(p, 'max_rois', 50))
    pair_score_min = float(_pget(p, 'pair_score_min', 0.0))

    for i in range(len(infos)):
        a = infos[i]
        origin = a['center']
        u = a['u']; v = a['v']
        at1, at2, as0 = _projecoes_linha_em_base(a['line'], origin, u, v)
        amin, amax = min(at1, at2), max(at1, at2)

        for j in range(i + 1, len(infos)):
            dbg['n_pares_testados'] += 1
            b = infos[j]
            adiff = _angle_diff_180(a['angle'], b['angle'])
            if adiff > pair_tol:
                continue
            dbg['n_pares_ang_ok'] += 1

            bt1, bt2, bs0 = _projecoes_linha_em_base(b['line'], origin, u, v)
            bmin, bmax = min(bt1, bt2), max(bt1, bt2)

            dist = abs(bs0 - as0)
            if dist < dist_min or dist > dist_max:
                continue
            dbg['n_pares_dist_ok'] += 1

            ov = razao_sobreposicao((amin, amax), (bmin, bmax))
            gap_axis = _interval_gap(amin, amax, bmin, bmax)

            if ov < overlap_min and gap_axis > axis_gap_max:
                continue
            if ov >= overlap_min:
                dbg['n_pares_overlap_ok'] += 1

            tmin = min(amin, bmin)
            tmax = max(amax, bmax)
            smin, smax = sorted([as0, bs0])

            width_pair = max(1.0, smax - smin)
            base_len = max(1.0, tmax - tmin)
            lw_ratio = base_len / width_pair

            score, score_parts = _score_par_v2(ov, adiff, pair_tol, lw_ratio, p)
            if score < pair_score_min:
                continue

            bbox, corners, bbox_info = _bbox_par_por_margem_fixa(
                origin, u, v, tmin, tmax, smin, smax, img_shape, p
            )
            if bbox is None:
                continue

            x1, y1, x2, y2 = bbox
            roi_w = max(1, x2 - x1)
            roi_h = max(1, y2 - y1)
            aspect = roi_h / roi_w

            if not (float(_pget(p, 'roi_aspect_min', 0.3)) <= aspect <= float(_pget(p, 'roi_aspect_max', 15.0))):
                continue

            candidatos.append({
                'line1': a['line'], 'line2': b['line'],
                'line_base': a['line'], 'line_oposta': b['line'],
                'bbox': tuple(map(int, bbox)),
                'oriented_corners': corners,
                'origin': origin, 'u': u, 'v': v,
                'tmin': float(bbox_info['tmin']), 'tmax': float(bbox_info['tmax']),
                'smin': float(bbox_info['smin']), 'smax': float(bbox_info['smax']),
                'distance': float(dist),
                'overlap': float(ov),
                'gap_axis': float(gap_axis),
                'angle_diff': float(adiff),
                'angle': float(a['angle']),
                'area': int(roi_w * roi_h),
                'aspect': float(aspect),
                'lw_ratio': float(lw_ratio),
                'lw_ratio_bbox': float(bbox_info['lw_ratio_bbox']),
                'score_pair': float(score),
                'score_roi': float(score),
                'roi_score': float(score),
                'roi_evidence_score': float(score),
                'roi_evidence_class': '2_retas_paralelas',
                'tipo': 'par_retas_v2',
                **score_parts,
                **bbox_info,
            })

    candidatos = sorted(
        candidatos,
        key=lambda c: (c['score_pair'], c['overlap'], c['score_lw_ratio'], -c['angle_diff'], c['area']),
        reverse=True
    )[:max_rois]

    dbg['n_candidatos_par'] = len(candidatos)
    return (candidatos, dbg) if return_debug else candidatos


def expandir_candidato_orientado_por_score(img_bgr, pre, candidato, clf, p):
    """
    Compatibilidade com a pipeline anterior.

    Na V2, a BBox já é calculada com margem fixa na formação do par de retas.
    Portanto, esta função apenas reaproveita a BBox do candidato e, se houver
    classificador, calcula o score HOG/SVM.
    """
    bbox = candidato.get('bbox')
    if bbox is None or not _bbox_valida(bbox):
        return None, {'cap_score': 0.0}, {'refine_mode': 'bbox_invalida'}

    x1, y1, x2, y2 = map(int, bbox)
    crop = img_bgr[y1:y2, x1:x2].copy()
    pred = None
    svm_score = 0.0

    if clf is not None and crop.size > 0:
        try:
            feat = validar_features_modelo(extrair_hog_de_imagem(crop, p), clf, contexto="ROI V2")
            pred = int(clf.predict(feat)[0])
            try:
                svm_score = float(clf.decision_function(feat)[0])
            except Exception:
                svm_score = float(pred)
        except Exception:
            svm_score = -999.0

    geom_score = float(candidato.get('score_pair', 0.0))
    score_final = svm_score + 0.25 * geom_score if clf is not None else geom_score

    cap = {'cap_score': 0.0}
    ref_info = {
        'refine_mode': 'bbox_margem_fixa_v2',
        'tested': 1,
        'factor': 1.0,
        'score_final': float(score_final),
        'score_svm': svm_score,
        'pred_refine': pred,
        'geom_score': geom_score,
        'oriented_corners': candidato.get('oriented_corners'),
        'tmin': float(candidato.get('tmin', 0.0)),
        'tmax': float(candidato.get('tmax', 0.0)),
        'smin': float(candidato.get('smin', 0.0)),
        'smax': float(candidato.get('smax', 0.0)),
    }
    return tuple(map(int, bbox)), cap, ref_info


def _bbox_iou_e_overlap_menor(a, b):
    if not _bbox_valida(a) or not _bbox_valida(b):
        return 0.0, 0.0
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = _bbox_area(a); area_b = _bbox_area(b)
    union = area_a + area_b - inter
    iou = inter / union if union > 0 else 0.0
    ov_small = inter / max(1e-6, min(area_a, area_b))
    return float(iou), float(ov_small)


def consolidar_rois_sobrepostas(candidatos, p):
    if not bool(_pget(p, 'roi_consolidation_enabled', True)):
        return list(candidatos), {
            'n_candidatos_pre_consolidacao': len(candidatos),
            'n_candidatos_pos_consolidacao': len(candidatos),
            'n_rois_consolidadas': 0
        }

    iou_thr = float(_pget(p, 'roi_consolidation_iou', 0.45))
    overlap_thr = float(_pget(p, 'roi_consolidation_overlap_small', 0.75))
    center_thr = float(_pget(p, 'roi_consolidation_center_px', 110))

    def rank(c):
        return (
            float(c.get('score_pair', 0.0)),
            float(c.get('overlap', 0.0)),
            float(c.get('score_lw_ratio', 0.0)),
            -float(c.get('angle_diff', 0.0)),
            float(c.get('area', 0.0)),
        )

    ordenados = sorted(candidatos, key=rank, reverse=True)
    usados = [False] * len(ordenados)
    saida = []
    removidas = 0

    for i, base in enumerate(ordenados):
        if usados[i]:
            continue
        usados[i] = True
        grupo = [base]
        cb = _bbox_center(base.get('bbox'))

        for j in range(i + 1, len(ordenados)):
            if usados[j]:
                continue
            cand = ordenados[j]
            iou, ov_small = _bbox_iou_e_overlap_menor(base.get('bbox'), cand.get('bbox'))
            dist_c = float(np.linalg.norm(cb - _bbox_center(cand.get('bbox'))))
            if (iou >= iou_thr) or (ov_small >= overlap_thr and (center_thr <= 0 or dist_c <= center_thr)):
                usados[j] = True
                grupo.append(cand)

        melhor = sorted(grupo, key=rank, reverse=True)[0].copy()
        melhor['n_rois_consolidadas'] = int(len(grupo))
        removidas += max(0, len(grupo) - 1)
        saida.append(melhor)

    return saida, {
        'n_candidatos_pre_consolidacao': len(candidatos),
        'n_candidatos_pos_consolidacao': len(saida),
        'n_rois_consolidadas': int(removidas)
    }


def candidatos_linhas_pares_v2(img_bgr, pre, p):
    fonte = _pget(p, 'hough_source_any', 'canny_hough')
    edges = pre.get(fonte, pre.get('canny_hough', pre['canny_full']))

    lines_raw = detectar_linhas_hough(edges, p)
    lines_merged = unir_segmentos_colineares(lines_raw, p)
    lines_geom = filtrar_linhas_remanescentes(lines_merged, p)
    lines_ok, debug_suporte = filtrar_linhas_por_suporte(lines_geom, pre, p)

    candidatos_brutos, debug_pares = gerar_pares_orientados(lines_ok, pre['gray'].shape, p, return_debug=True)
    candidatos, debug_consolidacao = consolidar_rois_sobrepostas(candidatos_brutos, p)
    candidatos = candidatos[:int(_pget(p, 'max_rois', 50))]

    info = {
        'lines_raw': lines_raw,
        'lines_merged': lines_merged,
        'lines_filtradas_geom': lines_geom,
        'lines_filtradas': lines_ok,
        'candidatos_sem_consolidacao': list(candidatos_brutos),
        'candidatos_consolidados': list(candidatos),
        'n_lines_raw': len(lines_raw),
        'n_lines_merged': len(lines_merged),
        'n_lines_filtradas_geom': len(lines_geom),
        'n_lines_filtradas': len(lines_ok),
        'n_candidatos_par': len(candidatos_brutos),
        'n_candidatos_brutos': len(candidatos_brutos),
        'n_candidatos_total': len(candidatos),
        'n_rois_pos_evidencia': len(candidatos_brutos),
    }
    info.update(debug_suporte)
    info.update(debug_pares)
    info.update(debug_consolidacao)
    return candidatos, info


def detectar_candidatos_cilindro(img_bgr, clf=None, p=None, score_min=None, nms_iou=None, max_det=None, aplicar_nms=True):
    if p is None:
        p = STATE['params']
    score_min = float(_pget(p, 'det_score_min', 0.0) if score_min is None else score_min)
    nms_iou = float(_pget(p, 'det_nms_iou', 0.30) if nms_iou is None else nms_iou)
    max_det = int(_pget(p, 'det_max_det', 8) if max_det is None else max_det)

    pre = aplicar_preprocessamento(img_bgr, p)
    candidatos, info_linhas = candidatos_linhas_pares_v2(img_bgr, pre, p)

    resultados, positivos = [], []
    erros = 0

    for idx, c in enumerate(candidatos, start=1):
        try:
            bbox_ref, cap, ref_info = expandir_candidato_orientado_por_score(img_bgr, pre, c, clf, p)
            if bbox_ref is None or not _bbox_valida(bbox_ref):
                continue

            x1, y1, x2, y2 = map(int, bbox_ref)
            crop = img_bgr[y1:y2, x1:x2].copy()
            if crop.size == 0:
                continue

            pred = ref_info.get('pred_refine')
            score_svm = ref_info.get('score_svm')
            score_final = ref_info.get('score_final')

            if clf is not None and pred is None:
                feat = validar_features_modelo(extrair_hog_de_imagem(crop, p), clf, contexto=f"ROI {idx}")
                pred = int(clf.predict(feat)[0])
                try:
                    score_svm = float(clf.decision_function(feat)[0])
                except Exception:
                    score_svm = float(pred)
                score_final = score_svm + 0.25 * float(c.get('score_pair', 0.0))

            r = {
                'roi': idx,
                'pred': 'cilindro' if pred == 1 else ('negativo' if pred == 0 else 'sem_modelo'),
                'bbox_original': c.get('bbox'),
                'bbox': tuple(map(int, bbox_ref)),
                'oriented_corners': ref_info.get('oriented_corners', c.get('oriented_corners')),
                'score': None if score_svm is None else round(float(score_svm), 3),
                'score_float': score_svm,
                'score_final': None if score_final is None else round(float(score_final), 3),
                'score_final_float': score_final,
                'score_geom': round(float(c.get('score_pair', 0.0)), 3),
                'score_pair': round(float(c.get('score_pair', 0.0)), 3),
                'score_roi': round(float(c.get('score_roi', c.get('score_pair', 0.0))), 3),
                'score_overlap': round(float(c.get('score_overlap', 0.0)), 3),
                'score_angle': round(float(c.get('score_angle', 0.0)), 3),
                'score_lw_ratio': round(float(c.get('score_lw_ratio', 0.0)), 3),
                'lw_ratio': round(float(c.get('lw_ratio', 0.0)), 2),
                'lw_target': round(float(c.get('lw_target', _pget(p, 'pair_lw_ratio_target', 5.0))), 2),
                'refine_mode': ref_info.get('refine_mode'),
                'dist_px': round(float(c.get('distance', 0.0)), 1),
                'score par': round(float(c.get('score_pair', 0.0)), 2),
                'score ROI': round(float(c.get('score_roi', c.get('score_pair', 0.0))), 3),
                'evidência ROI': c.get('roi_evidence_class', '2_retas_paralelas'),
                'overlap': round(float(c.get('overlap', 0.0)), 2),
                'ang_diff': round(float(c.get('angle_diff', 0.0)), 2),
                'angle': round(float(c.get('angle', 0.0)), 1),
                'tipo': c.get('tipo', 'par_retas_v2'),
            }

            resultados.append(r)

            if clf is not None and pred == 1 and (score_final is None or float(score_final) >= score_min):
                positivos.append(r)

        except Exception as e:
            erros += 1
            err_msg = str(e)[:240]
            print(f"[ERRO ROI {idx}] {err_msg}", flush=True)
            resultados.append({
                'roi': idx,
                'pred': 'erro',
                'bbox_original': c.get('bbox'),
                'bbox': c.get('bbox'),
                'erro': err_msg
            })

    positivos_finais = nms_bboxes_refinado(
        positivos,
        iou_thr=nms_iou,
        max_det=max_det
    ) if aplicar_nms else positivos

    ids_finais = {r['roi'] for r in positivos_finais}
    for r in resultados:
        r['nms_keep'] = bool(r.get('roi') in ids_finais)
        r['pred_visual'] = 'suprimido' if r.get('pred') == 'cilindro' and not r['nms_keep'] else r.get('pred')

    info = {
        'pred_raw': len(positivos),
        'pred_final': len(positivos_finais),
        'erro_rois': int(erros),
        'score_min': score_min,
        'nms_iou': nms_iou,
        'detector_strategy': 'pares_retas_v2'
    }
    info.update(info_linhas)
    return {
        'pre': pre,
        'candidatos': candidatos,
        'resultados': resultados,
        'positivos_finais': positivos_finais,
        'info': info
    }


# ------------------------------------------------------------
# Cores e visualizações
# ------------------------------------------------------------
COLOR_HOUGH_RAW = (0, 220, 255)
COLOR_HOUGH_FILTERED = (0, 220, 255)
COLOR_CANDIDATE_LINE = (0, 220, 255)
COLOR_ROI_INITIAL = (255, 140, 0)
COLOR_ROI_REFINED = (0, 255, 80)
COLOR_GT = (255, 220, 0)
COLOR_NEG = COLOR_ROI_INITIAL


def _desenhar_poligono_orientado(img_rgb, pts, color=(0, 255, 255), thickness=1):
    if pts is None:
        return img_rgb
    pts = np.asarray(pts, dtype=np.int32).reshape(-1, 1, 2)
    cv2.polylines(img_rgb, [pts], True, color, thickness)
    return img_rgb


def _desenhar_retas_usadas_candidato(img_rgb, candidato, color=COLOR_CANDIDATE_LINE, thickness=1):
    """Desenha as retas usadas como evidência da detecção."""
    for line_key in ['line_base', 'line_oposta', 'line1', 'line2']:
        line = candidato.get(line_key)
        if line is None:
            continue
        x1, y1, x2, y2 = map(int, line)
        cv2.line(img_rgb, (x1, y1), (x2, y2), color, int(max(1, thickness)))


def _desenhar_numero_bbox(img_rgb, texto, x, y, color, font_scale=0.48, thickness=2):
    cv2.putText(
        img_rgb,
        str(texto),
        (int(x), int(max(12, y))),
        cv2.FONT_HERSHEY_SIMPLEX,
        float(font_scale),
        color,
        int(max(1, thickness)),
        cv2.LINE_AA
    )


def _desenhar_bbox_original_e_final(img_rgb, candidato, img_bgr_ref, pre, p, idx=None, line_thickness=1, bbox_thickness=1):
    """
    Desenha retas e BBoxes sem qualquer evidência curva.

    Convenção visual:
    - ciano: retas usadas como evidência;
    - laranja: BBox inicial/geométrica do par;
    - verde: BBox final usada no HOG/SVM.
    """
    _desenhar_retas_usadas_candidato(img_rgb, candidato, thickness=line_thickness)

    bbox_original = candidato.get('bbox')
    if _bbox_valida(bbox_original):
        x1, y1, x2, y2 = map(int, bbox_original)
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), COLOR_ROI_INITIAL, int(max(1, bbox_thickness)))

    try:
        bbox_ref, cap, ref_info = expandir_candidato_orientado_por_score(img_bgr_ref, pre, candidato, None, p)
    except Exception:
        bbox_ref, cap, ref_info = None, {}, {}

    if bbox_ref is not None and _bbox_valida(bbox_ref):
        bx1, by1, bx2, by2 = map(int, bbox_ref)
        cv2.rectangle(img_rgb, (bx1, by1), (bx2, by2), COLOR_ROI_REFINED, int(max(1, bbox_thickness)))
        if idx is not None:
            score = float(candidato.get('score_pair', candidato.get('score_roi', 0.0)))
            _desenhar_numero_bbox(img_rgb, f'{idx} ({score:.2f})', bx1, by1 - 3, COLOR_ROI_REFINED)
    elif idx is not None and _bbox_valida(bbox_original):
        x1, y1, x2, y2 = map(int, bbox_original)
        _desenhar_numero_bbox(img_rgb, idx, x1, y1 - 3, COLOR_ROI_INITIAL)

    return img_rgb


def _desenhar_roi_inicial_candidato(img_rgb, candidato, idx=None):
    _desenhar_retas_usadas_candidato(img_rgb, candidato, thickness=1)
    if _bbox_valida(candidato.get('bbox')):
        x1, y1, x2, y2 = map(int, candidato['bbox'])
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), COLOR_ROI_INITIAL, 1)
        if idx is not None:
            score = float(candidato.get('score_pair', candidato.get('score_roi', 0.0)))
            _desenhar_numero_bbox(img_rgb, f'{idx} ({score:.2f})', x1, y1 - 3, COLOR_ROI_INITIAL)


def _desenhar_roi_expandida_candidato(img_rgb, candidato, img_bgr_ref, pre, p, idx=None, line_thickness=1, bbox_thickness=1, **kwargs):
    return _desenhar_bbox_original_e_final(
        img_rgb,
        candidato,
        img_bgr_ref,
        pre,
        p,
        idx=idx,
        line_thickness=line_thickness,
        bbox_thickness=bbox_thickness
    )


def render_rois_iniciais_refinadas():
    pre = obter_preprocessamento_referencia(); img_bgr_ref = obter_imagem_referencia()
    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada.'); return
    p = STATE['params']
    det = detectar_candidatos_cilindro(img_bgr_ref, clf=None, p=p, aplicar_nms=False)
    info = det['info']
    candidatos = list(info.get('candidatos_sem_consolidacao', det.get('candidatos', [])))[:int(_pget(p, 'max_rois', 50))]
    img = pre['rgb'].copy()
    for k, c in enumerate(candidatos, start=1):
        _desenhar_roi_inicial_candidato(img, c, idx=k)
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), dpi=120)
    ax.imshow(img)
    ax.set_title(
        f"BBoxes por pares de retas antes do IoU | pares={info.get('n_candidatos_par', 0)} | "
        f"alvo L/W={_pget(p, 'pair_lw_ratio_target', 5.0)} | caixas={len(candidatos)}"
    )
    ax.axis('off')
    fig.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.01)
    plt.show()
    display(HTML('<small><b>Legenda:</b> ciano = retas usadas; laranja = BBox geométrica antes da consolidação.</small>'))
    if not candidatos:
        print('Nenhuma ROI inicial foi gerada. Ajuste Hough, suporte das retas ou critérios dos pares paralelos.')


def render_rois_expandidas_refinadas():
    pre = obter_preprocessamento_referencia(); img_bgr_ref = obter_imagem_referencia()
    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada.'); return
    p = STATE['params']
    det = detectar_candidatos_cilindro(img_bgr_ref, clf=None, p=p, aplicar_nms=False)
    candidatos = det['candidatos']; info = det['info']
    atualizar_opcoes_roi_hog(candidatos)
    img = pre['rgb'].copy()
    for k, c in enumerate(candidatos[:int(_pget(p, 'max_rois', 50))], start=1):
        _desenhar_roi_expandida_candidato(img, c, img_bgr_ref, pre, p, idx=k)
    fig, ax = plt.subplots(1, 1, figsize=(10, 7), dpi=120)
    ax.imshow(img)
    ax.set_title(
        f"BBoxes com margem fixa + IoU | antes/depois="
        f"{info.get('n_candidatos_pre_consolidacao', len(candidatos))}/"
        f"{info.get('n_candidatos_pos_consolidacao', len(candidatos))} | finais={len(candidatos)}"
    )
    ax.axis('off')
    fig.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.01)
    plt.show()
    display(HTML('<small><b>Legenda:</b> ciano = retas usadas; laranja = BBox inicial; verde = BBox final após consolidação por IoU.</small>'))
    if not candidatos:
        print('Nenhuma ROI consolidada foi gerada.')


def render_rois_refinadas():
    render_rois_iniciais_refinadas()
    render_rois_expandidas_refinadas()


def _score_texto_candidato(c):
    try:
        return f"score={float(c.get('score_pair', c.get('score_roi', 0.0))):.2f}"
    except Exception:
        return "score=-"


def render_hog_visualizacao_refinada():
    pre = obter_preprocessamento_referencia(); img_bgr_ref = obter_imagem_referencia()
    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada.'); return
    p = STATE['params']
    det = detectar_candidatos_cilindro(img_bgr_ref, clf=None, p=p, aplicar_nms=False)
    candidatos = det['candidatos']
    idx = atualizar_opcoes_roi_hog(candidatos)
    if idx is None or len(candidatos) == 0:
        print('Nenhuma ROI candidata foi gerada para esta imagem com os parâmetros atuais.'); return
    idx = int(np.clip(int(idx), 0, len(candidatos)-1))
    c = candidatos[idx]
    bbox_ref, cap, ref_info = expandir_candidato_orientado_por_score(img_bgr_ref, pre, c, None, p)
    if bbox_ref is None or not _bbox_valida(bbox_ref):
        print('A ROI selecionada gerou uma caixa inválida.'); return
    x1, y1, x2, y2 = map(int, bbox_ref)
    crop_bgr = img_bgr_ref[y1:y2, x1:x2].copy()
    if crop_bgr.size == 0:
        print('A ROI selecionada gerou um recorte vazio.'); return
    try:
        hog_input_resized, hog_image, features_hog = extrair_hog_visualizacao_de_imagem(crop_bgr, p)
        features_total = extrair_hog_de_imagem(crop_bgr, p)
    except Exception as e:
        print('Não foi possível gerar a visualização HOG:', e); traceback.print_exc(limit=1); return
    img_contexto = pre['rgb'].copy()
    _desenhar_roi_expandida_candidato(img_contexto, c, img_bgr_ref, pre, p, idx=idx+1)
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    score_txt = _score_texto_candidato(c)
    fig, axs = plt.subplots(1, 4, figsize=(16, 4.6))
    axs[0].imshow(img_contexto); axs[0].set_title(f'ROI/BBox selecionada\nROI {idx+1} | {score_txt}')
    axs[1].imshow(crop_rgb); axs[1].set_title('Recorte usado no SVM')
    axs[2].imshow(hog_input_resized, cmap='gray'); axs[2].set_title(f"Entrada HOG\n{p['hog_input']}")
    axs[3].imshow(hog_image, cmap='gray'); axs[3].set_title(f'HOG\nbase={len(features_hog)} | total={len(features_total)}')
    for ax in axs: ax.axis('off')
    plt.tight_layout(); plt.show()

    df_info = pd.DataFrame([{
        'roi': idx + 1,
        'score_roi': round(float(c.get('score_pair', c.get('score_roi', 0.0))), 3),
        'overlap': round(float(c.get('overlap', 0.0)), 2),
        'ang_diff': round(float(c.get('angle_diff', 0.0)), 2),
        'L/W': round(float(c.get('lw_ratio', 0.0)), 2),
        'alvo_L/W': round(float(c.get('lw_target', _pget(p, 'pair_lw_ratio_target', 5.0))), 2),
        'bbox': c.get('bbox'),
        'hog_input': p['hog_input'],
        'hog_size': f"{p['hog_resize_w']}x{p['hog_resize_h']}",
        'orient': p['hog_orientations'],
        'px_cell': p['hog_pixels_per_cell'],
        'cells_block': p['hog_cells_per_block'],
        'n_atributos': len(features_total)
    }])
    exibir_tabela_compacta(df_info, max_rows=1)


## Ajuste interativo único dos parâmetros V2

Todos os parâmetros principais do detector ficam nesta seção. A formulação das ROIs foi simplificada para usar somente pares de retas paralelas. O score geométrico passa a considerar overlap, paralelismo e a razão comprimento/largura da ROI em relação a um valor ótimo ajustável.


In [13]:
# ============================================================
# 8. Ajuste interativo de parâmetros + setups salvos V2
# ============================================================

from datetime import datetime
from collections import OrderedDict
import json


PARAM_WIDGETS = OrderedDict()


def _w_check(nome, value, description):
    PARAM_WIDGETS[nome] = widgets.Checkbox(value=value, description=description, indent=False)


def _w_int(nome, value, minv, maxv, step, description):
    PARAM_WIDGETS[nome] = widgets.IntSlider(
        value=value, min=minv, max=maxv, step=step,
        description=description, continuous_update=False
    )


def _w_float(nome, value, minv, maxv, step, description, readout_format='.2f'):
    PARAM_WIDGETS[nome] = widgets.FloatSlider(
        value=value, min=minv, max=maxv, step=step,
        description=description, readout_format=readout_format,
        continuous_update=False
    )


def _w_dropdown(nome, options, value, description):
    PARAM_WIDGETS[nome] = widgets.Dropdown(options=options, value=value, description=description)


def _w_hidden(nome, value):
    if isinstance(value, bool):
        PARAM_WIDGETS[nome] = widgets.Checkbox(value=value, description=nome)
    elif isinstance(value, int):
        PARAM_WIDGETS[nome] = widgets.IntText(value=value, description=nome)
    elif isinstance(value, float):
        PARAM_WIDGETS[nome] = widgets.FloatText(value=value, description=nome)
    else:
        PARAM_WIDGETS[nome] = widgets.Text(value=str(value), description=nome)


# ----------------------------
# Parâmetros visíveis — V2 baseada em S004, sem evidências curvas
# ----------------------------
# Pré-processamento e Canny
_w_check('clahe_ativo', True, 'usar CLAHE')
_w_float('clahe_clip', 1.5, 0.5, 6.0, 0.1, 'clip')
_w_int('clahe_grid', 16, 2, 32, 1, 'grid CLAHE')
_w_int('bilateral_d', 5, 1, 15, 2, 'bilateral d')
_w_float('bilateral_sigma_color', 60, 1, 150, 1, 'sigma cor', readout_format='.0f')
_w_float('bilateral_sigma_space', 50, 1, 150, 1, 'sigma espaço', readout_format='.0f')
_w_float('canny_sigma', 0.33, 0.05, 0.90, 0.01, 'sigma Canny')
_w_check('edge_filter_ativo', True, 'filtrar traços')
_w_int('edge_min_area', 41, 0, 200, 1, 'área mín')
_w_int('edge_min_extent', 48, 0, 160, 1, 'extensão mín')
_w_dropdown('hough_source_any', [('Canny completo', 'canny_full'), ('Canny filtrado', 'canny_hough')], 'canny_hough', 'Hough em:')

# Hough, retas e suporte real
_w_int('hough_threshold', 70, 5, 150, 5, 'threshold')
_w_int('hough_min_line_length', 50, 5, 300, 5, 'minLen')
_w_int('hough_max_line_gap', 20, 0, 80, 2, 'maxGap')
_w_int('line_length_min', 45, 5, 400, 5, 'L min')
_w_int('line_length_max', 250, 20, 700, 10, 'L max')
_w_check('merge_collinear_enabled', True, 'unir colineares')
_w_check('line_support_enabled', True, 'validar suporte')
_w_float('line_support_min_ratio', 0.30, 0.02, 0.95, 0.02, 'suporte mín')
_w_int('line_support_max_gap_px', 30, 2, 160, 2, 'buraco máx')

# Pares paralelos e score geométrico V2
_w_float('pair_angle_tol_deg', 3.0, 1.0, 30.0, 0.5, 'ângulo tol')
_w_int('pair_dist_min', 30, 5, 200, 5, 'dist min')
_w_int('pair_dist_max', 110, 10, 300, 5, 'dist max')
_w_float('pair_overlap_min', 0.60, 0.05, 1.00, 0.05, 'overlap mín')
_w_int('pair_axis_gap_px', 10, 0, 140, 5, 'gap eixo')
_w_float('pair_lw_ratio_target', 5.0, 1.0, 12.0, 0.25, 'L/W alvo')
_w_float('pair_lw_ratio_tol_pct', 0.60, 0.10, 2.00, 0.05, 'tol L/W')
_w_float('pair_score_overlap_weight', 0.45, 0.0, 1.0, 0.05, 'peso overlap')
_w_float('pair_score_angle_weight', 0.30, 0.0, 1.0, 0.05, 'peso ângulo')
_w_float('pair_score_ratio_weight', 0.25, 0.0, 1.0, 0.05, 'peso L/W')
_w_float('pair_score_min', 0.0, 0.0, 1.0, 0.05, 'score par mín')

# BBox, ROIs e detecção final
_w_int('roi_margin_px', 10, 0, 60, 2, 'margem ROI')
_w_int('max_rois', 50, 1, 300, 1, 'max ROIs')
_w_check('roi_consolidation_enabled', True, 'consolidar IoU')
_w_float('roi_consolidation_iou', 0.45, 0.05, 0.95, 0.05, 'IoU consol.')
_w_float('roi_consolidation_overlap_small', 0.75, 0.20, 1.00, 0.05, 'overlap menor')
_w_int('roi_consolidation_center_px', 110, 0, 200, 5, 'dist centro')
_w_float('det_score_min', 0.0, -30.0, 10.0, 0.10, 'score mín')
_w_float('det_nms_iou', 0.30, 0.05, 0.90, 0.05, 'NMS IoU')
_w_float('det_iou_thr', 0.50, 0.10, 0.90, 0.05, 'IoU acerto')
_w_int('det_max_det', 8, 1, 50, 1, 'max det')
_w_int('det_eval_max_images', 500, 0, 1000, 20, 'max aval.')

# HOG/SVM e amostragem
_w_hidden('hog_input', 'bilateral')
_w_int('hog_resize_w', 64, 32, 160, 8, 'HOG W')
_w_int('hog_resize_h', 160, 64, 256, 8, 'HOG H')
_w_int('hog_orientations', 12, 4, 18, 1, 'orient')
_w_dropdown('hog_pixels_per_cell', [4, 8, 16], 4, 'cell')
_w_dropdown('hog_cells_per_block', [1, 2, 3], 2, 'block')
PARAM_WIDGETS['svm_C'] = widgets.FloatLogSlider(
    value=1.7783, base=10, min=-2, max=2, step=0.25,
    description='SVM C', continuous_update=False
)
_w_check('usar_selecao_salva', True, 'usar seleção')
_w_check('usar_test_real', True, 'avaliar test/')
_w_int('negativos_por_positivo', 2, 1, 8, 1, 'neg/pos')
_w_float('neg_iou_max', 0.05, 0.0, 0.40, 0.01, 'IoU neg')
_w_float('pos_margin_pct', 0.10, 0.0, 0.40, 0.02, 'margem pos')
_w_int('neg_attempts', 40, 5, 100, 5, 'tent neg')
_w_int('max_train_images', 0, 0, 3000, 50, 'max train')
_w_int('max_test_images', 0, 0, 3000, 50, 'max test')
_w_float('test_size', 0.40, 0.10, 0.50, 0.05, 'holdout')
_w_int('random_state', 42, 0, 999, 1, 'seed')

# ----------------------------
# Parâmetros internos fixados
# ----------------------------
_hidden_defaults = {
    'canny_aperture': 3,
    'canny_l2gradient': True,
    'hough_rho': 1.0,
    'hough_theta_deg': 1.0,
    'roi_aspect_min': 0.3,
    'roi_aspect_max': 15.0,
    'merge_gap_px': 18,
    'merge_angle_tol_deg': 5.0,
    'merge_perp_tol_px': 8,
    'line_support_source': 'canny_hough',
    'line_support_radius_px': 2,
    'line_support_min_grad_norm': 0.0,
}
for _nome, _valor in _hidden_defaults.items():
    if _nome not in PARAM_WIDGETS:
        _w_hidden(_nome, _valor)

DEFAULT_PARAM_VALUES = OrderedDict((nome, wid.value) for nome, wid in PARAM_WIDGETS.items())
DEFAULT_SETUP_ID = 'PADRAO'


# ----------------------------
# Funções para setups salvos V2
# ----------------------------
def valores_parametros_widgets():
    return OrderedDict((nome, wid.value) for nome, wid in PARAM_WIDGETS.items())


def _json_parametros(params):
    return json.dumps(dict(params), sort_keys=True, ensure_ascii=False, default=str)


def parametros_iguais(a, b):
    return _json_parametros(a) == _json_parametros(b)


def carregar_setups_parametricos():
    data = carregar_json(PARAM_SETUPS_PATH, default=None)
    if not isinstance(data, dict):
        data = {}
    if 'setups' not in data or not isinstance(data['setups'], dict):
        data['setups'] = {}
    return data


def salvar_setups_parametricos(data):
    data = dict(data)
    data['versao'] = 'setups_parametricos_hough_hog_svm_v2_pares_retas'
    data['atualizado_em'] = time.strftime('%Y-%m-%d %H:%M:%S')
    salvar_json(PARAM_SETUPS_PATH, data)
    try:
        registrar_pendencia_git(PARAM_SETUPS_PATH, 'setup paramétrico V2 salvo/alterado')
    except Exception:
        pass


def garantir_setup_v2_padrao():
    data = carregar_setups_parametricos()
    if 'S004_V2' not in data.get('setups', {}):
        data['setups']['S004_V2'] = {
            'nome': 'S004 adaptado V2 | somente pares de retas',
            'data_hora': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'origem': 'gerado a partir dos valores do S004, removendo evidências curvas',
            'params': dict(DEFAULT_PARAM_VALUES)
        }
        salvar_setups_parametricos(data)
    return data


def proximo_setup_id(data):
    existentes = set(data.get('setups', {}).keys())
    i = 1
    while f'SV2_{i:03d}' in existentes:
        i += 1
    return f'SV2_{i:03d}'


def obter_params_setup_base(setup_id=None):
    setup_id = setup_id or STATE.get('setup_id', DEFAULT_SETUP_ID)
    if setup_id == DEFAULT_SETUP_ID:
        return DEFAULT_PARAM_VALUES

    data = carregar_setups_parametricos()
    setup = data.get('setups', {}).get(setup_id, {})
    return OrderedDict(setup.get('params', {}))


def aplicar_parametros_no_widgets(params):
    """Aplica parâmetros aos widgets sem disparar várias renderizações intermediárias."""
    STATE['loading_setup'] = True
    try:
        for nome, valor in params.items():
            if nome not in PARAM_WIDGETS:
                continue
            wid = PARAM_WIDGETS[nome]
            try:
                wid.value = valor
            except Exception:
                pass
        definir_params_a_partir_widgets(PARAM_WIDGETS)
    finally:
        STATE['loading_setup'] = False


def atualizar_estado_modificacao_setup():
    base = obter_params_setup_base(STATE.get('setup_id', DEFAULT_SETUP_ID))
    atual = valores_parametros_widgets()
    STATE['setup_modificado'] = not parametros_iguais(atual, base)


def obter_setup_id_para_historico():
    if STATE.get('setup_modificado', False):
        return 'MANUAL_V2_NAO_SALVO'
    return STATE.get('setup_id', DEFAULT_SETUP_ID)


def obter_setup_nome_para_historico():
    if STATE.get('setup_modificado', False):
        base = STATE.get('setup_id', DEFAULT_SETUP_ID)
        return f'Manual V2 não salvo, derivado de {base}'
    return STATE.get('setup_label', 'Setup padrão V2')


w_setup_selector = widgets.Dropdown(
    options=[('PADRAO | Setup padrão V2', DEFAULT_SETUP_ID)],
    value=DEFAULT_SETUP_ID,
    description='setup:',
    layout=widgets.Layout(width='460px')
)

w_setup_nome = widgets.Text(
    value='',
    placeholder='Nome opcional para o novo setup V2',
    description='nome:',
    layout=widgets.Layout(width='520px')
)

btn_setup_padrao = widgets.Button(description='Setup padrão', icon='undo', layout=widgets.Layout(width='150px'))
btn_salvar_setup = widgets.Button(description='Salvar setup', button_style='success', icon='save', layout=widgets.Layout(width='145px'))
btn_excluir_setup = widgets.Button(description='Excluir setup', button_style='danger', icon='trash', layout=widgets.Layout(width='145px'))
btn_renomear_setup = widgets.Button(description='Renomear setup', button_style='info', icon='edit', layout=widgets.Layout(width='160px'))
btn_sobrescrever_setup = widgets.Button(description='Sobrescrever setup', button_style='warning', icon='refresh', layout=widgets.Layout(width='185px'))

w_confirmar_sobrescrita_setup = widgets.Checkbox(
    value=False,
    description='confirmar sobrescrita',
    indent=False,
    layout=widgets.Layout(width='240px')
)

out_setup_status = widgets.Output()
out_setup_resultados = widgets.Output()


def renderizar_resultados_setup_selecionado():
    with out_setup_resultados:
        clear_output(wait=True)
        setup_id = STATE.get('setup_id', DEFAULT_SETUP_ID)
        hist = carregar_historico()

        if hist is None or len(hist) == 0:
            display(HTML('<em>Nenhum treinamento V2 salvo ainda.</em>'))
            return

        if 'setup_id' not in hist.columns:
            display(HTML('<em>O histórico existente ainda não possui coluna setup_id.</em>'))
            return

        df_setup = hist[hist['setup_id'].astype(str) == str(setup_id)].copy()
        if len(df_setup) == 0:
            display(HTML(f'<em>Nenhum resultado salvo para o setup {setup_id}.</em>'))
            return

        colunas = [
            'data_hora', 'setup_id', 'setup_nome', 'zoom_out_pct', 'accuracy',
            'precision', 'recall', 'f1', 'tempo_s',
            'n_img_train', 'n_img_test', 'n_train', 'n_test', 'assinatura_curta'
        ]
        colunas = [c for c in colunas if c in df_setup.columns]

        display(HTML(f'<b>Resultados salvos para o setup {setup_id}</b>'))
        exibir_tabela_compacta(df_setup[colunas].tail(8), max_rows=8)


def atualizar_status_setup(mensagem=''):
    with out_setup_status:
        clear_output(wait=True)
        sid = obter_setup_id_para_historico()
        nome = obter_setup_nome_para_historico()
        marcador = ' | parâmetros alterados e ainda não salvos' if STATE.get('setup_modificado', False) else ''
        print(f'Setup atual para histórico: {sid} — {nome}{marcador}')
        if mensagem:
            print(mensagem)
        print(f'Arquivo de setups V2: {PARAM_SETUPS_PATH}')
        print('Modelo V2:', MODEL_PATH)


def atualizar_dropdown_setups(selecionar=None):
    data = carregar_setups_parametricos()
    setups = data.get('setups', {})

    opcoes = [('PADRAO | Setup padrão V2', DEFAULT_SETUP_ID)]
    for setup_id in sorted(setups.keys()):
        setup = setups[setup_id]
        nome = setup.get('nome', 'sem_nome')
        data_hora = setup.get('data_hora', '')
        opcoes.append((f'{setup_id} | {nome} | {data_hora}', setup_id))

    valores = [v for _, v in opcoes]
    selecionar = selecionar if selecionar in valores else DEFAULT_SETUP_ID

    STATE['setup_dropdown_updating'] = True
    try:
        w_setup_selector.options = opcoes
        w_setup_selector.value = selecionar
    finally:
        STATE['setup_dropdown_updating'] = False


def carregar_setup_por_id(setup_id, atualizar_visualizacao=True):
    if setup_id == DEFAULT_SETUP_ID:
        aplicar_parametros_no_widgets(DEFAULT_PARAM_VALUES)
        STATE['setup_id'] = DEFAULT_SETUP_ID
        STATE['setup_label'] = 'Setup padrão V2'
        STATE['setup_modificado'] = False
        if atualizar_visualizacao:
            atualizar_todas_visualizacoes()
        atualizar_status_setup('Setup padrão V2 carregado.')
        renderizar_resultados_setup_selecionado()
        return

    data = carregar_setups_parametricos()
    setup = data.get('setups', {}).get(setup_id)
    if setup is None:
        atualizar_status_setup(f'Setup {setup_id} não encontrado. O setup padrão foi mantido.')
        return

    params = OrderedDict(setup.get('params', {}))
    aplicar_parametros_no_widgets(params)
    STATE['setup_id'] = setup_id
    STATE['setup_label'] = setup.get('nome', setup_id)
    STATE['setup_modificado'] = False
    if atualizar_visualizacao:
        atualizar_todas_visualizacoes()
    atualizar_status_setup(f'Setup {setup_id} carregado.')
    renderizar_resultados_setup_selecionado()


def on_setup_selector_change(change):
    if STATE.get('setup_dropdown_updating', False):
        return
    if change.get('name') == 'value' and change.get('new') is not None:
        carregar_setup_por_id(change['new'])


def on_setup_padrao(_=None):
    atualizar_dropdown_setups(DEFAULT_SETUP_ID)
    carregar_setup_por_id(DEFAULT_SETUP_ID)


def widget_value(widget):
    if hasattr(widget, 'value'):
        return widget.value
    return widget


def _params_atuais_para_setup():
    return OrderedDict((k, widget_value(w)) for k, w in PARAM_WIDGETS.items())


def _setup_existente_por_nome(data, nome):
    nome_norm = str(nome).strip().lower()
    for sid, item in data.get('setups', {}).items():
        if str(item.get('nome', '')).strip().lower() == nome_norm:
            return sid, item
    return None, None


def _setup_params_diferentes(item, params):
    return not parametros_iguais(OrderedDict(item.get('params', {})), params)


def on_salvar_setup(_=None):
    data = carregar_setups_parametricos()
    params_atual = _params_atuais_para_setup()
    nome = w_setup_nome.value.strip() or f'Setup {proximo_setup_id(data)}'

    sid_existente, item_existente = _setup_existente_por_nome(data, nome)
    if sid_existente is not None:
        if _setup_params_diferentes(item_existente, params_atual):
            if not w_confirmar_sobrescrita_setup.value:
                atualizar_status_setup(
                    f"Já existe um setup chamado '{nome}' ({sid_existente}) com parâmetros diferentes. "
                    "Marque 'confirmar sobrescrita' e clique novamente em Salvar setup para substituir."
                )
                return
            data['setups'][sid_existente]['params'] = dict(params_atual)
            data['setups'][sid_existente]['data_hora'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            data['setups'][sid_existente]['sobrescrito_em'] = data['setups'][sid_existente]['data_hora']
            salvar_setups_parametricos(data)
            STATE['setup_id'] = sid_existente
            STATE['setup_label'] = nome
            STATE['setup_modificado'] = False
            w_confirmar_sobrescrita_setup.value = False
            atualizar_dropdown_setups(sid_existente)
            atualizar_status_setup(f"Setup existente {sid_existente} — '{nome}' sobrescrito com os parâmetros atuais.")
            renderizar_resultados_setup_selecionado()
            return
        else:
            STATE['setup_id'] = sid_existente
            STATE['setup_label'] = nome
            STATE['setup_modificado'] = False
            atualizar_dropdown_setups(sid_existente)
            atualizar_status_setup(f"Já existia um setup chamado '{nome}' com os mesmos parâmetros. Setup {sid_existente} carregado.")
            renderizar_resultados_setup_selecionado()
            return

    novo_id = proximo_setup_id(data)
    data['setups'][novo_id] = {
        'nome': nome,
        'data_hora': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'params': dict(params_atual)
    }
    salvar_setups_parametricos(data)

    STATE['setup_id'] = novo_id
    STATE['setup_label'] = nome
    STATE['setup_modificado'] = False
    w_confirmar_sobrescrita_setup.value = False
    atualizar_dropdown_setups(novo_id)
    atualizar_status_setup(f'Setup salvo como {novo_id}.')
    renderizar_resultados_setup_selecionado()


def on_excluir_setup(_=None):
    setup_id = w_setup_selector.value
    if setup_id == DEFAULT_SETUP_ID:
        atualizar_status_setup('O setup padrão não pode ser excluído.')
        return

    data = carregar_setups_parametricos()
    setups = data.get('setups', {})
    if setup_id in setups:
        nome = setups[setup_id].get('nome', setup_id)
        del setups[setup_id]
        data['setups'] = setups
        salvar_setups_parametricos(data)
        atualizar_dropdown_setups(DEFAULT_SETUP_ID)
        carregar_setup_por_id(DEFAULT_SETUP_ID)
        atualizar_status_setup(f'Setup {setup_id} — {nome} excluído. Setup padrão carregado.')
        renderizar_resultados_setup_selecionado()
    else:
        atualizar_status_setup(f'Setup {setup_id} não encontrado.')


def on_renomear_setup(_=None):
    setup_id = w_setup_selector.value
    novo_nome = w_setup_nome.value.strip()
    if setup_id == DEFAULT_SETUP_ID:
        atualizar_status_setup('O setup padrão não pode ser renomeado. Salve como um novo setup.')
        return
    if not novo_nome:
        atualizar_status_setup('Digite um novo nome no campo de nome antes de renomear.')
        return
    data = carregar_setups_parametricos()
    setups = data.get('setups', {})
    if setup_id not in setups:
        atualizar_status_setup(f'Setup {setup_id} não encontrado.')
        return
    sid_mesmo_nome, _ = _setup_existente_por_nome(data, novo_nome)
    if sid_mesmo_nome is not None and sid_mesmo_nome != setup_id:
        atualizar_status_setup(f"Já existe outro setup com o nome '{novo_nome}' ({sid_mesmo_nome}). Escolha outro nome.")
        return
    setups[setup_id]['nome'] = novo_nome
    setups[setup_id]['renomeado_em'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    salvar_setups_parametricos(data)
    STATE['setup_label'] = novo_nome
    atualizar_dropdown_setups(setup_id)
    atualizar_status_setup(f'Setup {setup_id} renomeado para {novo_nome}.')
    renderizar_resultados_setup_selecionado()


def on_sobrescrever_setup(_=None):
    setup_id = w_setup_selector.value
    if setup_id == DEFAULT_SETUP_ID:
        atualizar_status_setup('O setup padrão não pode ser sobrescrito. Salve como um novo setup.')
        return
    if not w_confirmar_sobrescrita_setup.value:
        atualizar_status_setup("Para sobrescrever o setup selecionado, marque 'confirmar sobrescrita' e clique novamente.")
        return
    data = carregar_setups_parametricos()
    setups = data.get('setups', {})
    if setup_id not in setups:
        atualizar_status_setup(f'Setup {setup_id} não encontrado.')
        return
    params_atual = _params_atuais_para_setup()
    nome = w_setup_nome.value.strip() or setups[setup_id].get('nome', setup_id)
    sid_mesmo_nome, _ = _setup_existente_por_nome(data, nome)
    if sid_mesmo_nome is not None and sid_mesmo_nome != setup_id:
        atualizar_status_setup(f"Já existe outro setup com o nome '{nome}' ({sid_mesmo_nome}). Escolha outro nome.")
        return
    setups[setup_id]['nome'] = nome
    setups[setup_id]['params'] = dict(params_atual)
    setups[setup_id]['data_hora'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    setups[setup_id]['sobrescrito_em'] = setups[setup_id]['data_hora']
    salvar_setups_parametricos(data)
    STATE['setup_id'] = setup_id
    STATE['setup_label'] = nome
    STATE['setup_modificado'] = False
    w_confirmar_sobrescrita_setup.value = False
    atualizar_dropdown_setups(setup_id)
    atualizar_status_setup(f'Setup {setup_id} sobrescrito com os parâmetros atuais.')
    renderizar_resultados_setup_selecionado()


def on_params_change(change=None):
    definir_params_a_partir_widgets(PARAM_WIDGETS)

    if STATE['params']['line_length_max'] < STATE['params']['line_length_min']:
        STATE['params']['line_length_max'] = STATE['params']['line_length_min']

    if STATE['params']['pair_dist_max'] < STATE['params']['pair_dist_min']:
        STATE['params']['pair_dist_max'] = STATE['params']['pair_dist_min']

    if not STATE.get('loading_setup', False):
        atualizar_estado_modificacao_setup()
        atualizar_status_setup()
        atualizar_todas_visualizacoes()


for wid in PARAM_WIDGETS.values():
    wid.observe(on_params_change, names='value')

w_setup_selector.observe(on_setup_selector_change, names='value')
btn_setup_padrao.on_click(on_setup_padrao)
btn_salvar_setup.on_click(on_salvar_setup)
btn_excluir_setup.on_click(on_excluir_setup)
btn_renomear_setup.on_click(on_renomear_setup)
btn_sobrescrever_setup.on_click(on_sobrescrever_setup)

# Inicializa os parâmetros e cria o setup S004_V2 se ainda não existir.
definir_params_a_partir_widgets(PARAM_WIDGETS)
garantir_setup_v2_padrao()
atualizar_dropdown_setups('S004_V2')
carregar_setup_por_id('S004_V2', atualizar_visualizacao=False)
atualizar_status_setup()
renderizar_resultados_setup_selecionado()


# ----------------------------
# Textos curtos e interface compacta por etapa
# ----------------------------
PARAM_HELP = {
    'clahe_ativo': 'Liga/desliga o realce local. Ajuda em baixo contraste.',
    'clahe_clip': 'Aumentar destaca bordas fracas; reduzir evita amplificar ruído.',
    'clahe_grid': 'Grade local do CLAHE. Maior = realce mais suave.',
    'bilateral_d': 'Tamanho da vizinhança. Maior = mais suavização.',
    'bilateral_sigma_color': 'Diferença de intensidade aceita na suavização.',
    'bilateral_sigma_space': 'Alcance espacial do filtro bilateral.',
    'canny_sigma': 'Sensibilidade do Canny adaptativo.',
    'edge_filter_ativo': 'Remove componentes pequenos do mapa de bordas.',
    'edge_min_area': 'Área mínima de componente de borda.',
    'edge_min_extent': 'Extensão mínima de traços no mapa de bordas.',
    'hough_source_any': 'Mapa usado para detectar retas.',
    'hough_threshold': 'Votos mínimos da Hough. Maior = retas mais fortes.',
    'hough_min_line_length': 'Comprimento mínimo detectado pela Hough.',
    'hough_max_line_gap': 'Falha máxima que a Hough conecta em uma reta.',
    'line_length_min': 'Remove retas curtas demais.',
    'line_length_max': 'Remove retas longas demais do fundo.',
    'merge_collinear_enabled': 'Une segmentos alinhados.',
    'line_support_enabled': 'Valida se há borda real acompanhando a reta.',
    'line_support_min_ratio': 'Fração mínima da reta com suporte em borda.',
    'line_support_max_gap_px': 'Maior falha contínua permitida ao longo da reta.',
    'pair_angle_tol_deg': 'Tolerância de paralelismo entre as duas retas.',
    'pair_dist_min': 'Menor largura aceitável entre laterais.',
    'pair_dist_max': 'Maior largura aceitável entre laterais.',
    'pair_overlap_min': 'Sobreposição longitudinal mínima entre as retas.',
    'pair_axis_gap_px': 'Desalinhamento longitudinal máximo permitido.',
    'pair_lw_ratio_target': 'Razão comprimento/largura ideal da ROI. Valor base sugerido: 5.',
    'pair_lw_ratio_tol_pct': 'Tolerância relativa antes de penalizar a razão L/W.',
    'pair_score_overlap_weight': 'Peso do overlap no score geométrico.',
    'pair_score_angle_weight': 'Peso da similaridade angular no score geométrico.',
    'pair_score_ratio_weight': 'Peso da proximidade com a razão L/W ideal.',
    'pair_score_min': 'Score geométrico mínimo do par para manter a ROI.',
    'roi_margin_px': 'Margem fixa adicionada à BBox.',
    'max_rois': 'Número máximo de ROIs avaliadas.',
    'roi_consolidation_enabled': 'Remove ou funde caixas muito parecidas.',
    'roi_consolidation_iou': 'IoU usado para consolidar caixas sobrepostas.',
    'roi_consolidation_overlap_small': 'Remove caixa pequena quase contida em outra.',
    'roi_consolidation_center_px': 'Distância máxima entre centros para consolidação.',
    'det_score_min': 'Score mínimo final para aceitar detecção.',
    'det_nms_iou': 'IoU do NMS final.',
    'det_iou_thr': 'IoU usado como acerto na avaliação.',
    'det_max_det': 'Máximo de detecções finais por imagem.',
    'det_eval_max_images': 'Limite de imagens avaliadas.',
    'hog_input': 'Entrada fixa do HOG: imagem bilateral.',
    'hog_resize_w': 'Largura padronizada do recorte HOG.',
    'hog_resize_h': 'Altura padronizada do recorte HOG.',
    'hog_orientations': 'Número de orientações do histograma de gradientes.',
    'hog_pixels_per_cell': 'Tamanho da célula HOG.',
    'hog_cells_per_block': 'Blocos de normalização do HOG.',
    'svm_C': 'Rigidez do SVM. Maior ajusta mais o treino; menor generaliza mais.',
    'usar_selecao_salva': 'Usa seleção manual persistente.',
    'usar_test_real': 'Usa pasta test/ para avaliação.',
    'negativos_por_positivo': 'Quantidade de negativos por positivo.',
    'neg_iou_max': 'IoU máximo permitido para um negativo.',
    'pos_margin_pct': 'Margem adicionada aos positivos.',
    'neg_attempts': 'Tentativas para encontrar negativos válidos.',
    'max_train_images': 'Limite de imagens de treino. Zero usa todas.',
    'max_test_images': 'Limite de imagens de teste. Zero usa todas.',
    'test_size': 'Fração de validação interna se não usar test/.',
    'random_state': 'Semente para reprodutibilidade.'
}
for _nome_param in PARAM_WIDGETS:
    PARAM_HELP.setdefault(_nome_param, 'Parâmetro interno mantido com valor padrão para simplificar a interface.')

for _wid in PARAM_WIDGETS.values():
    try:
        _wid.layout.width = '360px'
        _wid.layout.min_width = '320px'
    except Exception:
        pass
    try:
        _wid.style = {'description_width': '115px'}
    except Exception:
        pass


def intro_html(titulo, texto):
    return widgets.HTML(
        f"<div style='padding:6px 8px; border-left:3px solid #888; margin:4px 0 8px 0;'>"
        f"<b>{titulo}</b><br><small>{texto}</small></div>"
    )


def help_html(nome):
    return widgets.HTML(
        f"<div style='font-size:11px; color:#cfcfcf; line-height:1.25; white-space:normal;'>{PARAM_HELP.get(nome, '')}</div>",
        layout=widgets.Layout(width='330px', min_width='260px')
    )


def param_item(nome):
    return widgets.HBox(
        [PARAM_WIDGETS[nome], help_html(nome)],
        layout=widgets.Layout(width='720px', min_width='620px', align_items='center', margin='3px 0')
    )


def param_row(nomes):
    return widgets.HBox(
        [param_item(nome) for nome in nomes],
        layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start', gap='4px 14px', width='100%')
    )


box_setup = widgets.VBox([
    widgets.HTML('<b>Setup paramétrico V2</b>'),
    widgets.HTML('<small>V2 usa apenas pares de retas paralelas. O score da ROI combina overlap, paralelismo e razão comprimento/largura.</small>'),
    widgets.HBox([w_setup_selector, btn_setup_padrao, btn_salvar_setup, btn_renomear_setup, btn_sobrescrever_setup, btn_excluir_setup], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    out_setup_resultados,
    widgets.HBox([w_setup_nome, w_confirmar_sobrescrita_setup], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    out_setup_status
])

box_pre_canny = widgets.VBox([
    intro_html('1. Pré-processamento e Canny', 'Prepara o mapa de bordas para Hough, suporte de linha e formação de ROIs por pares de retas.'),
    widgets.HTML('<b>Realce e suavização</b>'),
    param_row(['clahe_ativo', 'clahe_clip']),
    param_row(['clahe_grid', 'bilateral_d']),
    param_row(['bilateral_sigma_color', 'bilateral_sigma_space']),
    widgets.HTML('<b>Canny e filtro de traços</b>'),
    param_row(['canny_sigma', 'edge_filter_ativo']),
    param_row(['edge_min_area', 'edge_min_extent']),
    widgets.HTML('<b>Mapa usado na Hough</b>'),
    param_row(['hough_source_any']),
])

box_linhas = widgets.VBox([
    intro_html('2. Hough, retas filtradas e suporte real', 'Detecta segmentos de reta, une trechos colineares e remove linhas sem borda real suficiente.'),
    param_row(['hough_threshold', 'hough_min_line_length']),
    param_row(['hough_max_line_gap', 'line_length_min']),
    param_row(['line_length_max', 'merge_collinear_enabled']),
    param_row(['line_support_enabled', 'line_support_min_ratio']),
    param_row(['line_support_max_gap_px']),
])

box_pares = widgets.VBox([
    intro_html('3. Pares paralelos e score geométrico', 'As ROIs são geradas somente por pares de retas paralelas. O score combina overlap, ângulo similar e razão L/W próxima ao alvo.'),
    param_row(['pair_angle_tol_deg', 'pair_dist_min']),
    param_row(['pair_dist_max', 'pair_overlap_min']),
    param_row(['pair_axis_gap_px', 'pair_lw_ratio_target']),
    param_row(['pair_lw_ratio_tol_pct', 'pair_score_min']),
    param_row(['pair_score_overlap_weight', 'pair_score_angle_weight']),
    param_row(['pair_score_ratio_weight']),
])

box_bbox_consolidacao = widgets.VBox([
    intro_html('4. BBox com margem fixa e consolidação', 'A BBox contém apenas o par de retas, acrescido de uma margem fixa em pixels. A razão L/W é usada somente no score geométrico, não para expandir a caixa.'),
    param_row(['roi_margin_px', 'max_rois']),
    param_row(['roi_consolidation_enabled', 'roi_consolidation_iou']),
    param_row(['roi_consolidation_overlap_small', 'roi_consolidation_center_px']),
    param_row(['det_score_min', 'det_nms_iou']),
    param_row(['det_iou_thr', 'det_max_det']),
    param_row(['det_eval_max_images']),
])

box_hog_svm = widgets.VBox([
    intro_html('5. HOG/SVM e amostragem', 'Define como os recortes das BBoxes são transformados em descritores HOG e como o SVM é treinado com positivos e negativos.'),
    widgets.HTML('<small><b>Entrada HOG fixa:</b> bilateral. O HOG/SVM usa a imagem suavizada com preservação de bordas.</small>'),
    param_row(['hog_resize_w', 'hog_resize_h']),
    param_row(['hog_orientations', 'hog_pixels_per_cell']),
    param_row(['hog_cells_per_block', 'svm_C']),
    param_row(['test_size', 'random_state']),
    widgets.HTML('<b>Amostras do dataset YOLO</b>'),
    param_row(['usar_selecao_salva', 'usar_test_real']),
    param_row(['negativos_por_positivo', 'neg_iou_max']),
    param_row(['pos_margin_pct', 'neg_attempts']),
    param_row(['max_train_images', 'max_test_images']),
])

accordion = widgets.Accordion(children=[box_pre_canny, box_linhas, box_pares, box_bbox_consolidacao, box_hog_svm])
for idx, title in enumerate(['Pré/Canny', 'Retas', 'Pares/score', 'BBox/Consolidação', 'HOG/SVM']):
    accordion.set_title(idx, title)

out_grade_diagnostico = widgets.Output(layout=widgets.Layout(width='100%', overflow='hidden'))

w_hog_roi = widgets.Dropdown(
    description='ROI HOG',
    options=[('Nenhuma ROI candidata', None)],
    value=None,
    layout=widgets.Layout(width='90%'),
    style={'description_width': '80px'}
)

out_hog_vis = widgets.Output()

box_ref = widgets.VBox([
    widgets.HTML('<b>Imagem de referência</b>'),
    widgets.HTML('<small>Somente imagens marcadas na seleção manual aparecem nesta lista. A imagem escolhida alimenta a grade diagnóstica em tempo real.</small>'),
    w_ref_image
])

box_visualizacoes = widgets.VBox([
    widgets.HTML('<b>Visualizações compactas da imagem de referência</b>'),
    widgets.HTML('<small>Esta grade mostra, em tempo real, como os parâmetros afetam o pré-processamento, as retas, as ROIs e as BBoxes finais.</small>'),
    out_grade_diagnostico,

    widgets.HTML('<hr><b>HOG aplicado à ROI candidata</b>'),
    widgets.HTML('<small>Escolha uma ROI candidata para inspecionar o recorte usado pelo HOG/SVM.</small>'),
    w_hog_roi,
    out_hog_vis
])

painel_parametros = widgets.VBox([
    widgets.HTML('<b>2) Ajuste interativo de parâmetros — V2 somente pares de retas</b>'),
    widgets.HTML('<small>Fluxo: bordas → retas → pares paralelos → BBox com margem fixa → HOG/SVM.</small>'),
    box_setup,
    widgets.HTML('<hr><b>Imagem usada no ajuste</b>'),
    box_ref,
    widgets.HTML('<hr><b>Listas de ajuste paramétrico</b>'),
    accordion,
    widgets.HTML('<hr><b>Resultado visual dos parâmetros atuais</b>'),
    box_visualizacoes
])

display(painel_parametros)


def render_filtros_referencia():
    pre = obter_preprocessamento_referencia()
    img_bgr_ref = obter_imagem_referencia()

    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada. Marque imagens na seleção manual e escolha uma referência.')
        return

    p = STATE['params']
    info = pre.get('edge_filter_info', {})

    if info.get('enabled', False):
        titulo_mapa_b = (
            f"Mapa B: Canny p/ Hough e ROIs\n"
            f"mantidos={info.get('n_kept', 0)} | removidos={info.get('n_removed', 0)}"
        )
    else:
        titulo_mapa_b = 'Mapa B: Canny p/ Hough e ROIs\nfiltro desativado'

    fonte = p.get('hough_source_any', 'canny_hough')
    edges = pre.get(fonte, pre.get('canny_hough', pre['canny_full']))

    lines_raw = detectar_linhas_hough(edges, p)
    lines_merged = unir_segmentos_colineares(lines_raw, p)
    lines_geom = filtrar_linhas_remanescentes(lines_merged, p)
    lines_ok, dbg = filtrar_linhas_por_suporte(lines_geom, pre, p)

    img_hough = pre['rgb'].copy()

    for line in lines_raw[:250]:
        x1, y1, x2, y2 = map(int, line)
        cv2.line(img_hough, (x1, y1), (x2, y2), (0, 220, 255), 1)

    for line in lines_ok[:180]:
        x1, y1, x2, y2 = map(int, line)
        cv2.line(img_hough, (x1, y1), (x2, y2), (0, 220, 255), 2)

    titulo_hough = (
        f"Hough + retas filtradas\n"
        f"fonte={fonte} | brutas={len(lines_raw)} | válidas={len(lines_ok)}"
    )

    try:
        det = detectar_candidatos_cilindro(img_bgr_ref, clf=None, p=p, aplicar_nms=False)
        info_det = det.get('info', {})

        candidatos_iniciais = list(
            info_det.get('candidatos_sem_consolidacao', det.get('candidatos', []))
        )[:int(_pget(p, 'max_rois', 50))]

        candidatos_finais = list(det.get('candidatos', []))[:int(_pget(p, 'max_rois', 50))]

        atualizar_opcoes_roi_hog(candidatos_finais)

        img_rois_iniciais = pre['rgb'].copy()
        for k, c in enumerate(candidatos_iniciais, start=1):
            _desenhar_roi_expandida_candidato(
                img_rois_iniciais, c, img_bgr_ref, pre, p,
                idx=k, line_thickness=2, bbox_thickness=1
            )

        img_bboxes_finais = pre['rgb'].copy()
        for k, c in enumerate(candidatos_finais, start=1):
            _desenhar_roi_expandida_candidato(
                img_bboxes_finais, c, img_bgr_ref, pre, p,
                idx=k, line_thickness=1, bbox_thickness=1
            )

        titulo_rois = (
            f"BBoxes antes do IoU\n"
            f"pares={info_det.get('n_candidatos_par', 0)} | caixas={len(candidatos_iniciais)} | alvo L/W={_pget(p, 'pair_lw_ratio_target', 5.0)}"
        )

        titulo_bboxes = (
            f"BBoxes finais + IoU\n"
            f"antes/depois={len(candidatos_iniciais)}/{len(candidatos_finais)}"
        )

    except Exception as e:
        candidatos_finais = []
        atualizar_opcoes_roi_hog(candidatos_finais)
        img_rois_iniciais = pre['rgb'].copy()
        img_bboxes_finais = pre['rgb'].copy()
        titulo_rois = f"ROIs iniciais\nerro: {type(e).__name__}"
        titulo_bboxes = f"BBoxes finais\nerro: {type(e).__name__}"
        print('Aviso: não foi possível gerar ROIs/BBoxes nesta atualização:', e)

    imagens = [
        (pre['rgb'], 'Original', None),
        (pre['gray_eq'], 'Cinza / CLAHE', 'gray'),
        (pre['bilateral'], 'Bilateral\nentrada fixa do HOG', 'gray'),
        (pre['canny_full'], f"Mapa A: Canny completo\nT1={pre['canny_lower']} | T2={pre['canny_upper']}", 'gray'),
        (pre['canny_hough'], titulo_mapa_b, 'gray'),
        (img_hough, titulo_hough, None),
        (img_rois_iniciais, titulo_rois, None),
        (img_bboxes_finais, titulo_bboxes, None),
    ]

    fig, axs = plt.subplots(2, 4, figsize=(32, 16), dpi=120, constrained_layout=False)
    axs = axs.ravel()

    for ax, (img, titulo, cmap) in zip(axs, imagens):
        ax.imshow(img, cmap=cmap, aspect="auto")
        ax.set_title(titulo, fontsize=18, pad=5)
        ax.axis("off")
        ax.margins(0)

    fig.subplots_adjust(left=0.001, right=0.999, top=0.965, bottom=0.001, wspace=0.003, hspace=0.12)
    plt.show()

    display(HTML(
        '<small><b>Legenda da grade:</b> '
        'ciano = retas usadas; laranja = BBox inicial; verde = BBox final.</small>'
    ))


def atualizar_opcoes_roi_hog(candidatos):
    """Atualiza o dropdown de ROIs incluindo o score geométrico."""
    antigo = w_hog_roi.value

    if not candidatos:
        novas_opcoes = [('Nenhuma ROI candidata', None)]
        novo_valor = None
    else:
        novas_opcoes = []
        for i, c in enumerate(candidatos):
            x1, y1, x2, y2 = map(int, c['bbox'])
            score = float(c.get('score_pair', c.get('score_roi', 0.0)))
            dist = c.get('distance', 0.0)
            ang = c.get('angle_diff', 0.0)
            ov = c.get('overlap', 0.0)
            lw = c.get('lw_ratio', 0.0)

            rotulo = (
                f'ROI {i+1} | score={score:.2f} | '
                f'bbox=({x1},{y1},{x2},{y2}) | '
                f'dist={float(dist):.1f}px | ang={float(ang):.1f}° | '
                f'ov={float(ov):.2f} | L/W={float(lw):.2f}'
            )
            novas_opcoes.append((rotulo, i))

        valores = [v for _, v in novas_opcoes]
        novo_valor = antigo if antigo in valores else valores[0]

    STATE['hog_roi_dropdown_updating'] = True
    try:
        w_hog_roi.options = novas_opcoes
        w_hog_roi.value = novo_valor
    finally:
        STATE['hog_roi_dropdown_updating'] = False

    return novo_valor


def render_hog_visualizacao():
    pre = obter_preprocessamento_referencia()
    img_bgr_ref = obter_imagem_referencia()

    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada.')
        return

    p = STATE['params']
    det = detectar_candidatos_cilindro(img_bgr_ref, clf=None, p=p, aplicar_nms=False)
    candidatos = det['candidatos']
    idx = atualizar_opcoes_roi_hog(candidatos)

    if idx is None or len(candidatos) == 0:
        print('Nenhuma ROI candidata foi gerada para esta imagem com os parâmetros atuais.')
        return

    idx = int(np.clip(int(idx), 0, len(candidatos) - 1))
    c = candidatos[idx]

    bbox_ref, cap, ref_info = expandir_candidato_orientado_por_score(img_bgr_ref, pre, c, None, p)
    if bbox_ref is None or not _bbox_valida(bbox_ref):
        print('A ROI selecionada gerou uma caixa inválida.')
        return

    x1, y1, x2, y2 = map(int, bbox_ref)
    crop_bgr = img_bgr_ref[y1:y2, x1:x2].copy()
    if crop_bgr.size == 0:
        print('A ROI selecionada gerou um recorte vazio.')
        return

    try:
        hog_input_resized, hog_image, features_hog = extrair_hog_visualizacao_de_imagem(crop_bgr, p)
        features_total = extrair_hog_de_imagem(crop_bgr, p)
    except Exception as e:
        print('Não foi possível gerar a visualização HOG:', e)
        traceback.print_exc(limit=1)
        return

    img_contexto = pre['rgb'].copy()
    _desenhar_roi_expandida_candidato(img_contexto, c, img_bgr_ref, pre, p, idx=idx + 1)

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    score = float(c.get('score_pair', c.get('score_roi', 0.0)))

    fig, axs = plt.subplots(1, 4, figsize=(16, 4.6))
    axs[0].imshow(img_contexto); axs[0].set_title(f'ROI/BBox selecionada\nROI {idx+1} | score={score:.2f}')
    axs[1].imshow(crop_rgb); axs[1].set_title('Recorte usado no SVM')
    axs[2].imshow(hog_input_resized, cmap='gray'); axs[2].set_title(f"Entrada HOG\n{p['hog_input']}")
    axs[3].imshow(hog_image, cmap='gray'); axs[3].set_title(f'HOG\nbase={len(features_hog)} | total={len(features_total)}')
    for ax in axs:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    df_info = pd.DataFrame([{
        'roi': idx + 1,
        'score_roi': round(score, 3),
        'overlap': round(float(c.get('overlap', 0.0)), 2),
        'ang_diff': round(float(c.get('angle_diff', 0.0)), 2),
        'L/W': round(float(c.get('lw_ratio', 0.0)), 2),
        'alvo_L/W': round(float(c.get('lw_target', _pget(p, 'pair_lw_ratio_target', 5.0))), 2),
        'bbox': c.get('bbox'),
        'hog_input': p['hog_input'],
        'hog_size': f"{p['hog_resize_w']}x{p['hog_resize_h']}",
        'orient': p['hog_orientations'],
        'px_cell': p['hog_pixels_per_cell'],
        'cells_block': p['hog_cells_per_block'],
        'n_atributos': len(features_total)
    }])
    exibir_tabela_compacta(df_info, max_rows=1)


def on_hog_roi_change(change):
    if STATE.get('hog_roi_dropdown_updating', False):
        return
    if change.get('name') == 'value':
        atualizar_renderer('hog_visualizacao')


w_hog_roi.observe(on_hog_roi_change, names='value')


def render_hough_orientado():
    pre = obter_preprocessamento_referencia()
    if pre is None:
        print('Nenhuma imagem de referência selecionada.')
        return
    p = STATE['params']
    fonte = p.get('hough_source_any', 'canny_hough')
    edges = pre.get(fonte, pre.get('canny_hough', pre['canny_full']))
    lines_raw = detectar_linhas_hough(edges, p)
    lines_merged = unir_segmentos_colineares(lines_raw, p)
    lines_geom = filtrar_linhas_remanescentes(lines_merged, p)
    lines_ok, dbg = filtrar_linhas_por_suporte(lines_geom, pre, p)
    img = pre['rgb'].copy()
    for line in lines_raw[:250]:
        x1, y1, x2, y2 = map(int, line)
        cv2.line(img, (x1, y1), (x2, y2), (0, 220, 255), 1)
    for line in lines_ok[:180]:
        x1, y1, x2, y2 = map(int, line)
        cv2.line(img, (x1, y1), (x2, y2), (0, 220, 255), 2)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(img)
    ax.set_title(f'Hough + suporte real | fonte={fonte} | brutas={len(lines_raw)} | unidas={len(lines_merged)} | geom={len(lines_geom)} | suporte={len(lines_ok)}')
    ax.axis('off')
    plt.tight_layout(); plt.show()


render_hough = render_hough_orientado
if 'render_hog_visualizacao_refinada' in globals():
    render_hog_visualizacao = render_hog_visualizacao_refinada

STATE['renderers'] = {}

registrar_renderer('grade_diagnostico', out_grade_diagnostico, render_filtros_referencia)
registrar_renderer('hog_visualizacao', out_hog_vis, render_hog_visualizacao)


## Treinamento HOG + SVM com bloqueio por botão

Esta célula **não treina automaticamente** quando você executa todas as células do notebook.

O treinamento só ocorre ao clicar no botão **Confirmar setup e treinar se necessário**.

Ao clicar, o notebook calcula uma assinatura considerando:

- imagens selecionadas nos JSONs de treino/teste;
- rótulos inferidos;
- tamanho e modificação dos arquivos;
- todos os parâmetros ajustados acima.

Se a assinatura for igual à do último modelo salvo, o treinamento é pulado e o histórico é apenas exibido.

In [ ]:
# ============================================================
# 12. Treinamento protegido por botão + histórico cumulativo
# ============================================================

btn_treinar = widgets.Button(
    description='Confirmar setup e treinar se necessário',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='330px')
)

btn_mostrar_historico = widgets.Button(
    description='Mostrar histórico',
    button_style='',
    icon='table',
    layout=widgets.Layout(width='180px')
)

out_train_status = widgets.Output()
out_history = widgets.Output()

display(widgets.HBox([btn_treinar, btn_mostrar_historico]))
display(out_train_status)
display(out_history)


def mostrar_historico():
    with out_history:
        clear_output(wait=True)

        hist = carregar_historico()
        if hist.empty:
            print('Ainda não há treinamentos registrados.')
            return

        cols_pref = [
            'data_hora', 'setup_id', 'setup_nome', 'refine_setup_id', 'refine_setup_nome', 'assinatura_curta', 'modo_eval',
            'n_img_train', 'n_img_test',
            'n_train', 'n_test', 'pos_train', 'neg_train', 'pos_test', 'neg_test',
            'accuracy', 'precision', 'recall', 'f1',
            'det_eval_imgs', 'n_gt_det', 'n_pred_det', 'det_iou_thr',
            'mean_iou_best', 'median_iou_best', 'mean_iou_tp',
            'precision_det', 'recall_det', 'f1_det', 'tp_det', 'fp_det', 'fn_det',
            'precision_det_30', 'recall_det_30', 'f1_det_30',
            'precision_det_50', 'recall_det_50', 'f1_det_50',
            'tempo_s',
            'hog_input', 'hog_size',
            'bilateral_d', 'sigma_color', 'sigma_space',
            'canny_sigma', 'edge_filter', 'edge_min_area', 'edge_min_extent',
            'hough_min_len', 'pair_dist', 'pair_angle_tol',
            'pair_lw_ratio_target', 'pair_lw_ratio_tol_pct', 'pair_score_overlap_weight', 'pair_score_angle_weight', 'pair_score_ratio_weight',
            'bbox_mode', 'roi_margin_px',
            'neg_por_pos', 'pos_margin', 'svm_C'
        ]

        cols = [c for c in cols_pref if c in hist.columns]
        exibir_tabela_compacta(hist[cols], max_rows=15)

        print(f'\nHistórico completo salvo em:\n{HISTORY_PATH}')


def treinar_se_necessario(_=None):
    with out_train_status:
        clear_output(wait=True)

        df = STATE.get('df_dataset', pd.DataFrame())
        p = dict(STATE['params'])

        if df is None or len(df) == 0:
            print('Dataset vazio. Recarregue a pasta de imagens antes de treinar.')
            return

        n_obj_train = int(df.loc[df['split'] == 'train', 'n_obj'].sum())
        n_obj_test = int(df.loc[df['split'] == 'test', 'n_obj'].sum())

        if n_obj_train < 2:
            print('Dataset de treino insuficiente: menos de 2 objetos anotados em train/labels.')
            print(resumo_dataset(df))
            return

        assinatura = assinatura_treinamento(df, p)
        assinatura_curta = assinatura[:12]

        meta = carregar_json(METADATA_PATH, default={})

        setup_id_hist = obter_setup_id_para_historico()
        setup_nome_hist = obter_setup_nome_para_historico()

        print('Setup atual:', setup_id_hist, '-', setup_nome_hist)
        print('Assinatura atual:', assinatura_curta)
        print('Modelo salvo:', MODEL_PATH.exists())
        print('Objetos anotados em train:', n_obj_train)
        print('Objetos anotados em test :', n_obj_test)

        if MODEL_PATH.exists() and meta.get('assinatura') == assinatura:
            print('\nNenhuma alteração detectada nos parâmetros/dataset.')
            print('Treinamento pulado. O modelo salvo continua válido para este setup.')
            mostrar_historico()
            if 'renderizar_resultados_setup_selecionado' in globals():
                renderizar_resultados_setup_selecionado()
            return

        print('\nAlteração detectada ou modelo inexistente.')
        print('Gerando amostras YOLO e treinando HOG + SVM...')

        t0 = time.time()

        def status_callback(txt):
            print(txt)

        try:
            X_train, X_test, y_train, y_test, stats, erros, info = preparar_dados_treino_teste(
                df, p, status_callback=status_callback
            )

            clf = make_pipeline(
                StandardScaler(),
                LinearSVC(
                    C=float(p['svm_C']),
                    class_weight='balanced',
                    max_iter=10000
                )
            )

            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, zero_division=0)
            rec = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

            tempo_s = time.time() - t0

            joblib.dump(clf, MODEL_PATH)

            # Avaliação do detector completo em imagem inteira.
            # Esta etapa mede localização de caixas via IoU, separada das métricas
            # de classificação HOG/SVM dos recortes.
            det_metrics = {}
            try:
                print('\nAvaliando detector completo por IoU...')
                det_metrics = avaliar_detector_completo(df, clf, p, status_callback=status_callback)
                if det_metrics:
                    print(
                        'Detector completo: '
                        f"imgs={det_metrics.get('det_eval_imgs', 0)} | "
                        f"GT={det_metrics.get('n_gt_det', 0)} | "
                        f"Pred={det_metrics.get('n_pred_det', 0)} | "
                        f"IoU médio(best)={det_metrics.get('mean_iou_best', 0):.3f} | "
                        f"F1@{det_metrics.get('det_iou_thr', 0.5):.2f}={det_metrics.get('f1_det', 0):.3f}"
                    )
            except Exception as e_det:
                print('Aviso: falha na avaliação do detector completo por IoU:', e_det)
                det_metrics = {}

            metadata = {
                'assinatura': assinatura,
                'assinatura_curta': assinatura_curta,
                'data_hora': time.strftime('%Y-%m-%d %H:%M:%S'),
                'model_path': str(MODEL_PATH),
                'history_path': str(HISTORY_PATH),
                'params': p,
                'setup_id': setup_id_hist,
                'setup_nome': setup_nome_hist,
                'setup_modificado': bool(STATE.get('setup_modificado', False)),
                'refine_setup_id': STATE.get('refine_setup_id', 'REFINO_ATUAL'),
                'refine_setup_nome': STATE.get('refine_setup_nome', 'parâmetros atuais/manuais'),
                'zoom_out_pct': int(obter_zoom_out_pct()),
                'zoomout_train_count': len(carregar_chaves_zoomout('train')),
                'zoomout_test_count': len(carregar_chaves_zoomout('test')),
                'dataset_path': str(STATE['data_path']),
                'stats': stats,
                'detector_eval': det_metrics
            }
            salvar_json(METADATA_PATH, metadata)

            row = {
                'data_hora': metadata['data_hora'],
                'setup_id': setup_id_hist,
                'setup_nome': setup_nome_hist,
                'refine_setup_id': STATE.get('refine_setup_id', 'REFINO_ATUAL'),
                'refine_setup_nome': STATE.get('refine_setup_nome', 'parâmetros atuais/manuais'),
                'zoom_out_pct': int(obter_zoom_out_pct()),
                'zoomout_train_count': len(carregar_chaves_zoomout('train')),
                'zoomout_test_count': len(carregar_chaves_zoomout('test')),
                'assinatura': assinatura,
                'assinatura_curta': assinatura_curta,
                'modo_eval': stats['modo_eval'],
                'n_img_train': stats['n_img_train'],
                'n_img_test': stats['n_img_test'],
                'n_total': stats['n_total'],
                'n_train': stats['n_train'],
                'n_test': stats['n_test'],
                'pos_train': stats['n_pos_train'],
                'neg_train': stats['n_neg_train'],
                'pos_test': stats['n_pos_test'],
                'neg_test': stats['n_neg_test'],
                'accuracy': abreviar_float(acc),
                'precision': abreviar_float(prec),
                'recall': abreviar_float(rec),
                'f1': abreviar_float(f1),
                'det_eval_imgs': int(det_metrics.get('det_eval_imgs', 0)),
                'n_gt_det': int(det_metrics.get('n_gt_det', 0)),
                'n_pred_det': int(det_metrics.get('n_pred_det', 0)),
                'det_iou_thr': abreviar_float(det_metrics.get('det_iou_thr', p.get('det_iou_thr', 0.5)), 2),
                'mean_iou_best': abreviar_float(det_metrics.get('mean_iou_best', 0.0), 4),
                'median_iou_best': abreviar_float(det_metrics.get('median_iou_best', 0.0), 4),
                'mean_iou_tp': abreviar_float(det_metrics.get('mean_iou_tp', 0.0), 4),
                'precision_det': abreviar_float(det_metrics.get('precision_det', 0.0), 4),
                'recall_det': abreviar_float(det_metrics.get('recall_det', 0.0), 4),
                'f1_det': abreviar_float(det_metrics.get('f1_det', 0.0), 4),
                'tp_det': int(det_metrics.get('tp_det', 0)),
                'fp_det': int(det_metrics.get('fp_det', 0)),
                'fn_det': int(det_metrics.get('fn_det', 0)),
                'precision_det_30': abreviar_float(det_metrics.get('precision_det_30', 0.0), 4),
                'recall_det_30': abreviar_float(det_metrics.get('recall_det_30', 0.0), 4),
                'f1_det_30': abreviar_float(det_metrics.get('f1_det_30', 0.0), 4),
                'precision_det_50': abreviar_float(det_metrics.get('precision_det_50', 0.0), 4),
                'recall_det_50': abreviar_float(det_metrics.get('recall_det_50', 0.0), 4),
                'f1_det_50': abreviar_float(det_metrics.get('f1_det_50', 0.0), 4),
                'tempo_s': abreviar_float(tempo_s, 3),
                'cm_tn': int(cm[0, 0]),
                'cm_fp': int(cm[0, 1]),
                'cm_fn': int(cm[1, 0]),
                'cm_tp': int(cm[1, 1]),
                'hog_input': p['hog_input'],
                'hog_size': f"{p['hog_resize_w']}x{p['hog_resize_h']}",
                'hog_orientations': int(p['hog_orientations']),
                'hog_cell': int(p['hog_pixels_per_cell']),
                'hog_block': int(p['hog_cells_per_block']),
                'bilateral_d': int(p['bilateral_d']),
                'sigma_color': int(p['bilateral_sigma_color']),
                'sigma_space': int(p['bilateral_sigma_space']),
                'canny_sigma': abreviar_float(p['canny_sigma'], 2),
                'edge_filter': bool(p.get('edge_filter_ativo', True)),
                'edge_min_area': int(p.get('edge_min_area', 0)),
                'edge_min_extent': int(p.get('edge_min_extent', 0)),
                'hough_min_len': int(p['hough_min_line_length']),
                'hough_gap': int(p['hough_max_line_gap']),
                'pair_dist': f"{p['pair_dist_min']}-{p['pair_dist_max']}",
                'pair_angle_tol': abreviar_float(p['pair_angle_tol_deg'], 1),
                'detector_strategy': 'pares_retas_v2',
                'hough_source_any': p.get('hough_source_any', 'canny_hough'),
                'merge_collinear': bool(p.get('merge_collinear_enabled', False)),
                'merge_gap': int(p.get('merge_gap_px', 0)),
                'merge_angle_tol': abreviar_float(p.get('merge_angle_tol_deg', 0), 1),
                'bbox_mode': 'margem_fixa_v2',
                'pair_lw_ratio_target': abreviar_float(p.get('pair_lw_ratio_target', 5.0), 2),
                'pair_lw_ratio_tol_pct': abreviar_float(p.get('pair_lw_ratio_tol_pct', 0.60), 2),
                'pair_score_overlap_weight': abreviar_float(p.get('pair_score_overlap_weight', 0.45), 2),
                'pair_score_angle_weight': abreviar_float(p.get('pair_score_angle_weight', 0.30), 2),
                'pair_score_ratio_weight': abreviar_float(p.get('pair_score_ratio_weight', 0.25), 2),
                'neg_por_pos': int(p['negativos_por_positivo']),
                'neg_iou_max': abreviar_float(p['neg_iou_max'], 2),
                'pos_margin': abreviar_float(p['pos_margin_pct'], 2),
                'svm_C': abreviar_float(p['svm_C'], 4),
                'dataset_path': str(STATE['data_path'])
            }

            salvar_linha_historico(row)

            print('\nTreinamento concluído.')
            print(f"Modo de avaliação: {stats['modo_eval']}")
            print(f'Tempo: {tempo_s:.3f} s')
            print(f'Acurácia: {acc:.4f} | Precisão: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}')
            if det_metrics:
                print(
                    f"Detector completo: IoU médio(best)={det_metrics.get('mean_iou_best', 0):.4f} | "
                    f"Precision_det={det_metrics.get('precision_det', 0):.4f} | "
                    f"Recall_det={det_metrics.get('recall_det', 0):.4f} | "
                    f"F1_det={det_metrics.get('f1_det', 0):.4f}"
                )
            print('\nAmostras:')
            print(f"Treino: {stats['n_train']}  | Pos: {stats['n_pos_train']}  | Neg: {stats['n_neg_train']}")
            print(f"Teste : {stats['n_test']}  | Pos: {stats['n_pos_test']}  | Neg: {stats['n_neg_test']}")
            print('\nMatriz de confusão [linhas: real 0/1; colunas: predito 0/1]:')
            print(cm)

            if erros:
                print(f'\nAtenção: {len(erros)} imagens falharam durante a extração de features.')
                for epath, emsg in erros[:5]:
                    print('-', Path(epath).name, '=>', emsg[:120])

            mostrar_historico()
            if 'renderizar_resultados_setup_selecionado' in globals():
                renderizar_resultados_setup_selecionado()

        except Exception as e:
            print('Erro durante o treinamento:')
            print(e)
            traceback.print_exc(limit=2)


btn_treinar.on_click(treinar_se_necessario)
btn_mostrar_historico.on_click(lambda _: mostrar_historico())

# Mostra histórico inicial, sem treinar.
mostrar_historico()


Output()

Output()

: 

## Validação visual por imagem selecionada

Esta seção mostra apenas **uma imagem por vez**, escolhida em uma lista. A comparação lado a lado evita a duplicação visual e deixa claro o efeito do treinamento/refino:

```text
esquerda  → antes do treinamento e antes da expansão/refino final da BBox
direita   → depois do treinamento HOG/SVM, expansão/refino da BBox e NMS
```

As duas imagens usam toda a largura disponível do notebook.


In [2]:
# ============================================================
# 13. Validação visual por imagem selecionada
# ============================================================

out_validacao_unica = widgets.Output()

# Cores de validação visual em RGB. Se a célula do detector orientado já foi executada,
# estas variáveis já existem; os fallbacks abaixo evitam erro ao reexecutar apenas esta célula.
COLOR_GT = globals().get('COLOR_GT', (255, 220, 0))
COLOR_ROI_INITIAL = globals().get('COLOR_ROI_INITIAL', (255, 140, 0))
COLOR_ROI_REFINED = globals().get('COLOR_ROI_REFINED', (0, 255, 80))
COLOR_NEG = globals().get('COLOR_NEG', (255, 120, 0))
COLOR_LINE_CANDIDATE = globals().get('COLOR_LINE_CANDIDATE', (0, 220, 255))

w_validacao_split = widgets.Dropdown(
    options=[
        ('Teste selecionado', 'test'),
        ('Treino selecionado', 'train'),
        ('Treino + teste selecionados', 'ambos')
    ],
    value='test',
    description='amostra:',
    layout=widgets.Layout(width='310px'),
    style={'description_width': '75px'}
)

w_validacao_imagem = widgets.Dropdown(
    options=[],
    description='imagem:',
    layout=widgets.Layout(width='680px'),
    style={'description_width': '65px'}
)

btn_atualizar_validacao = widgets.Button(
    description='Atualizar comparação',
    button_style='info',
    icon='refresh',
    layout=widgets.Layout(width='210px')
)

# Controles exclusivos da validação visual. Eles não alteram o treinamento;
# servem para inspecionar melhor as predições geradas pelas ROIs Hough+SVM.
w_vis_score_min = widgets.FloatSlider(
    value=0.0, min=-2.0, max=5.0, step=0.10,
    description='score mín.:',
    readout_format='.2f',
    layout=widgets.Layout(width='260px'),
    style={'description_width': '75px'}
)

w_vis_nms_iou = widgets.FloatSlider(
    value=0.30, min=0.05, max=0.90, step=0.05,
    description='NMS IoU:',
    readout_format='.2f',
    layout=widgets.Layout(width='250px'),
    style={'description_width': '70px'}
)

w_vis_max_det = widgets.IntSlider(
    value=8, min=1, max=30, step=1,
    description='máx. det.:',
    layout=widgets.Layout(width='230px'),
    style={'description_width': '70px'}
)

painel_validacao = widgets.VBox([
    widgets.HTML('<b>Validação visual por imagem</b>'),
    widgets.HTML(
        '<small>'
        'Esquerda: candidatos geométricos iniciais, antes do treinamento. '
        'Direita: resultado após treinamento HOG/SVM e NMS. '
        'Amarelo = YOLO/GT; ciano = retas usadas; laranja = ROI inicial; verde = predição final.'
        '</small>'
    ),
    widgets.HBox([w_validacao_split, btn_atualizar_validacao], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([w_validacao_imagem], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([w_vis_score_min, w_vis_nms_iou, w_vis_max_det], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    out_validacao_unica
])

display(painel_validacao)


# ------------------------------------------------------------
# Funções auxiliares de segurança
# ------------------------------------------------------------

def _valor_finito(v):
    """Retorna True apenas se o valor puder ser convertido para float finito."""
    try:
        return np.isfinite(float(v))
    except Exception:
        return False


def _bool_seguro(v, default=False):
    """
    Converte valores para booleano sem deixar NaN virar True.

    Problema corrigido:
    bool(np.nan) é True em Python. Então imagens sem zoom out, mas com campo
    zoom_out_aplicado = NaN, podiam ser tratadas como imagens com zoom out.
    """
    try:
        if v is None or pd.isna(v):
            return default
    except Exception:
        if v is None:
            return default

    if isinstance(v, str):
        txt = v.strip().lower()
        if txt in ['true', '1', 'sim', 'yes', 'y', 's']:
            return True
        if txt in ['false', '0', 'não', 'nao', 'no', 'n']:
            return False
        return default

    try:
        return bool(v)
    except Exception:
        return default


def _float_seguro(v, default=None):
    """Converte para float apenas se for finito."""
    try:
        f = float(v)
        return f if np.isfinite(f) else default
    except Exception:
        return default


def _int_seguro(v, default=None):
    """Converte para int apenas se for número finito."""
    try:
        f = float(v)
        if not np.isfinite(f):
            return default
        return int(round(f))
    except Exception:
        return default


def _normalizar_row_zoom_validacao(row):
    """
    Corrige campos problemáticos da linha do dataframe antes de carregar a imagem.

    Isso evita que imagens sem zoom out sejam interpretadas como zoom out aplicado
    quando os campos de zoom estão como NaN.
    """
    r = row.copy()

    if 'zoom_out_aplicado' in r.index:
        r['zoom_out_aplicado'] = _bool_seguro(r.get('zoom_out_aplicado'), default=False)

    # Se não há zoom out aplicado, neutraliza campos de recorte/escala que podem estar NaN.
    if not _bool_seguro(r.get('zoom_out_aplicado', False), default=False):
        campos_zoom = [
            'zoom_out_pct',
            'zoom_pct',
            'zoom_out_x1',
            'zoom_out_y1',
            'zoom_out_x2',
            'zoom_out_y2',
            'crop_x1',
            'crop_y1',
            'crop_x2',
            'crop_y2',
            'x_offset',
            'y_offset',
            'escala_zoom',
            'scale_zoom',
            'zoom_scale'
        ]

        for campo in campos_zoom:
            if campo in r.index:
                r[campo] = 0

    return r


def _normalizar_zoom_info(zoom_info):
    """
    Normaliza o dicionário zoom_info retornado por carregar_imagem_e_bboxes_row.
    Evita exibição incorreta e acesso a valores NaN.
    """
    if zoom_info is None or not isinstance(zoom_info, dict):
        return {'zoom_out_aplicado': False, 'zoom_out_pct': 0}

    z = dict(zoom_info)
    z['zoom_out_aplicado'] = _bool_seguro(z.get('zoom_out_aplicado'), default=False)

    pct = _float_seguro(z.get('zoom_out_pct'), default=0)
    z['zoom_out_pct'] = pct if pct is not None else 0

    return z


def _bbox_valida(bbox):
    """
    Verifica se bbox é válida antes de desenhar/converter para int.

    Esperado:
    bbox = [x1, y1, x2, y2]
    com todos os valores finitos e x2 > x1, y2 > y1.
    """
    try:
        if bbox is None:
            return False

        vals = [float(v) for v in bbox]

        if len(vals) != 4:
            return False

        if not all(np.isfinite(v) for v in vals):
            return False

        x1, y1, x2, y2 = vals
        return x2 > x1 and y2 > y1

    except Exception:
        return False


def _bbox_para_int(bbox):
    """
    Converte bbox válida para inteiros.
    Retorna None se houver NaN, inf ou formato inválido.
    """
    if not _bbox_valida(bbox):
        return None

    try:
        x1, y1, x2, y2 = [int(round(float(v))) for v in bbox]
        return x1, y1, x2, y2
    except Exception:
        return None


def _linha_valida(line):
    """
    Verifica se uma linha tem 4 coordenadas finitas.
    Esperado:
    line = [x1, y1, x2, y2]
    """
    try:
        if line is None:
            return False

        vals = [float(v) for v in line]

        if len(vals) != 4:
            return False

        return all(np.isfinite(v) for v in vals)

    except Exception:
        return False


def _linha_para_int(line):
    """Converte linha válida para inteiros."""
    if not _linha_valida(line):
        return None

    try:
        x1, y1, x2, y2 = [int(round(float(v))) for v in line]
        return x1, y1, x2, y2
    except Exception:
        return None


d
d
# ------------------------------------------------------------
# Modelo e desenho
# ------------------------------------------------------------

def _modelo_valido_para_assinatura_atual(verbose=True):
    if not MODEL_PATH.exists():
        if verbose:
            print('Ainda não há modelo treinado salvo.')
        return None, False

    meta = carregar_json(METADATA_PATH, default={})
    assinatura_atual = assinatura_treinamento(STATE['df_dataset'], dict(STATE['params']))
    modelo_desatualizado = meta.get('assinatura') != assinatura_atual

    if modelo_desatualizado:
        if verbose:
            print('Aviso: o modelo salvo foi treinado com outra assinatura de parâmetros/dataset/zoom out.')
            print("Clique em 'Confirmar setup e treinar se necessário' para atualizar o modelo antes da validação visual.")
        return None, False

    try:
        clf = joblib.load(MODEL_PATH)
        return clf, True
    except Exception as e:
        if verbose:
            print('Não foi possível carregar o modelo salvo:', e)
        return None, False


def _desenhar_gt_yolo(img_rgb, boxes, label='GT'):
    out = img_rgb.copy()

    if boxes is None:
        return out

    for idx, bbox in enumerate(boxes, start=1):
        bbox_int = _bbox_para_int(bbox)

        if bbox_int is None:
            continue

        x1, y1, x2, y2 = bbox_int
        cv2.rectangle(out, (x1, y1), (x2, y2), COLOR_GT, 1)
        cv2.putText(
            out,
            f'{label}{idx}',
            (x1, max(12, y1 - 4)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.35,
            COLOR_GT,
            1,
            cv2.LINE_AA
        )

    return out


def _melhor_iou_com_gt(bbox, boxes_gt):
    if not boxes_gt or not _bbox_valida(bbox):
        return 0.0

    vals = [
        iou_bbox(bbox, gt)
        for gt in boxes_gt
        if _bbox_valida(gt)
    ]

    return float(max(vals)) if vals else 0.0


def _desenhar_linhas_candidato(img_rgb, c, color=COLOR_LINE_CANDIDATE, thickness=2):
    for key in ['line_base', 'line_oposta', 'line1', 'line2']:
        line = c.get(key)

        line_int = _linha_para_int(line)
        if line_int is None:
            continue

        x1, y1, x2, y2 = line_int
        cv2.line(img_rgb, (x1, y1), (x2, y2), color, thickness)

    return img_rgb


def _desenhar_poligono_seguro(img_rgb, corners, color=COLOR_ROI_INITIAL, thickness=1):
    """
    Desenha polígono orientado se existir função global e se os pontos forem válidos.
    Evita erro com NaN em oriented_corners.
    """
    try:
        if corners is None:
            return img_rgb

        arr = np.asarray(corners, dtype=float)

        if arr.size == 0:
            return img_rgb

        arr = arr.reshape(-1, 2)

        if len(arr) < 3:
            return img_rgb

        if not np.all(np.isfinite(arr)):
            return img_rgb

        if '_desenhar_poligono_orientado' in globals():
            _desenhar_poligono_orientado(img_rgb, corners, color=color, thickness=thickness)
        else:
            pts = np.round(arr).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(img_rgb, [pts], True, color, thickness)

    except Exception:
        pass

    return img_rgb


d
# ------------------------------------------------------------
# Visualização antes do treinamento/refino
# ------------------------------------------------------------

def desenhar_estado_inicial_sem_treinamento(row):
    """
    Mostra os candidatos geométricos antes do SVM e antes da consolidação final da BBox.
    """
    row = _normalizar_row_zoom_validacao(row)

    img_bgr, boxes_gt, zoom_info = carregar_imagem_e_bboxes_row(row)
    zoom_info = _normalizar_zoom_info(zoom_info)

    p = STATE['params']
    pre = aplicar_preprocessamento(img_bgr, p)

    img_out = pre['rgb'].copy()
    img_out = _desenhar_gt_yolo(img_out, boxes_gt, label='GT')

    info = {'n_candidatos_total': 0, 'erro_rois': 0}
    candidatos = []

    try:
        if 'candidatos_linhas_pares_v2' in globals():
            candidatos, info = candidatos_linhas_pares_v2(img_bgr, pre, p)
        else:
            edges = pre.get('canny_hough', pre.get('canny'))
            lines = detectar_linhas_hough(edges, p)
            candidatos = encontrar_rois_por_pares_de_linhas(lines, pre['gray'].shape, p)
            info = {'n_candidatos_total': len(candidatos), 'n_linhas': len(lines)}

    except Exception as e:
        info['erro_rois'] = 1
        info['erro_msg'] = str(e)[:120]
        candidatos = []

    limite = min(len(candidatos), int(_pget(p, 'max_rois', 80)))

    for idx, c in enumerate(candidatos[:limite], start=1):
        _desenhar_linhas_candidato(img_out, c, color=COLOR_LINE_CANDIDATE, thickness=2)

        if c.get('oriented_corners') is not None:
            _desenhar_poligono_seguro(
                img_out,
                c.get('oriented_corners'),
                color=COLOR_ROI_INITIAL,
                thickness=1
            )

        bbox_int = _bbox_para_int(c.get('bbox'))
        if bbox_int is not None:
            x1, y1, x2, y2 = bbox_int
            cv2.rectangle(img_out, (x1, y1), (x2, y2), COLOR_ROI_INITIAL, 1)
            cv2.putText(
                img_out,
                f'ROI {idx}',
                (x1, max(12, y1 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.35,
                COLOR_ROI_INITIAL,
                1,
                cv2.LINE_AA
            )

    info['n_candidatos_total'] = len(candidatos)

    return img_out, candidatos, boxes_gt, zoom_info, info


# ------------------------------------------------------------
# Visualização depois do modelo/refino/NMS
# ------------------------------------------------------------

def aplicar_modelo_em_row(
    row,
    clf=None,
    desenhar_gt=True,
    aplicar_nms=True,
    score_min=None,
    nms_iou=None,
    max_det=None
):
    """
    Aplica a pipeline completa com refino da ROI, NMS.
    """
    row = _normalizar_row_zoom_validacao(row)

    img_bgr, boxes_gt, zoom_info = carregar_imagem_e_bboxes_row(row)
    zoom_info = _normalizar_zoom_info(zoom_info)

    p = STATE['params']

    score_min = float(w_vis_score_min.value if score_min is None else score_min)
    nms_iou = float(w_vis_nms_iou.value if nms_iou is None else nms_iou)
    max_det = int(w_vis_max_det.value if max_det is None else max_det)

    det = detectar_candidatos_cilindro(
        img_bgr,
        clf=clf,
        p=p,
        score_min=score_min,
        nms_iou=nms_iou,
        max_det=max_det,
        aplicar_nms=aplicar_nms
    )

    pre = det['pre']
    candidatos = det['candidatos']
    resultados = det['resultados']
    positivos_finais = det['positivos_finais']
    info = det['info']

    img_out = pre['rgb'].copy()

    if desenhar_gt and boxes_gt:
        img_out = _desenhar_gt_yolo(img_out, boxes_gt, label='GT')

    if clf is None:
        # Modelo ausente: mostra somente o estado geométrico/refinado disponível.
        for idx, c in enumerate(candidatos, start=1):
            bbox_int = _bbox_para_int(c.get('bbox'))

            if bbox_int is None:
                continue

            x1, y1, x2, y2 = bbox_int

            _desenhar_linhas_candidato(img_out, c, color=COLOR_LINE_CANDIDATE, thickness=2)

            cv2.rectangle(img_out, (x1, y1), (x2, y2), COLOR_ROI_INITIAL, 1)
            cv2.putText(
                img_out,
                f'ROI {idx}',
                (x1, max(12, y1 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.35,
                COLOR_ROI_INITIAL,
                1,
                cv2.LINE_AA
            )

        info.update({
            'pred_raw': 0,
            'pred_final': 0,
            'erro_rois': det['info'].get('erro_rois', 0)
        })

        return img_out, resultados, candidatos, boxes_gt, zoom_info, info

    for r in positivos_finais:
        bbox_int = _bbox_para_int(r.get('bbox'))

        # Correção principal:
        # Antes o código fazia direto: x1, y1, x2, y2 = map(int, r['bbox'])
        # Se a bbox refinada viesse com NaN, a célula quebrava.
        if bbox_int is None:
            continue

        x1, y1, x2, y2 = bbox_int

        # Caixa original da Hough/ROI em laranja, refinada em verde.
        bbox_original_int = _bbox_para_int(r.get('bbox_original'))

        if bbox_original_int is not None:
            ox1, oy1, ox2, oy2 = bbox_original_int
            cv2.rectangle(img_out, (ox1, oy1), (ox2, oy2), COLOR_ROI_INITIAL, 1)

        cv2.rectangle(img_out, (x1, y1), (x2, y2), COLOR_ROI_REFINED, 2)


        texto = f'CIL {r.get("roi", "?")}'

        score_final = _float_seguro(r.get('score_final'), default=None)
        if score_final is not None:
            texto += f' sf={score_final:.2f}'

        cv2.putText(
            img_out,
            texto,
            (x1, max(14, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.38,
            COLOR_ROI_REFINED,
            1,
            cv2.LINE_AA
        )

    pred_boxes = [
        r['bbox']
        for r in positivos_finais
        if _bbox_valida(r.get('bbox'))
    ]

    info['mean_best_iou_pred'] = round(
        float(np.mean([_melhor_iou_com_gt(b, boxes_gt) for b in pred_boxes]))
        if pred_boxes else 0.0,
        3
    )

    info['det_match_iou50'] = associar_predicoes_gt(
        pred_boxes,
        boxes_gt,
        iou_thr=0.50
    )

    return img_out, resultados, candidatos, boxes_gt, zoom_info, info


# ------------------------------------------------------------
# Dataframe da validação visual
# ------------------------------------------------------------

def _df_validacao_visual():
    df = STATE.get('df_dataset', pd.DataFrame())

    if df is None or len(df) == 0:
        return pd.DataFrame()

    modo = w_validacao_split.value
    partes = []

    if modo in ['train', 'ambos']:
        partes.append(df_por_selecao_persistente(df, 'train'))

    if modo in ['test', 'ambos']:
        partes.append(df_por_selecao_persistente(df, 'test'))

    partes = [
        p
        for p in partes
        if p is not None and len(p) > 0
    ]

    if not partes:
        return pd.DataFrame(columns=df.columns)

    return pd.concat(partes, ignore_index=True).reset_index(drop=True)


def atualizar_lista_imagens_validacao(selecionar=0):
    df_val = _df_validacao_visual()
    STATE['df_validacao_visual_atual'] = df_val

    if df_val is None or len(df_val) == 0:
        w_validacao_imagem.options = [('Nenhuma imagem selecionada disponível', -1)]
        w_validacao_imagem.value = -1
        return

    opcoes = []

    for i, row in df_val.iterrows():
        nome = str(row.get('arquivo', Path(row.get('path', '')).name))
        split = str(row.get('split', ''))

        label = f"{i + 1:03d} | {split} | {nome}"
        opcoes.append((label, int(i)))

    w_validacao_imagem.options = opcoes

    valores = [v for _, v in opcoes]
    w_validacao_imagem.value = int(selecionar) if int(selecionar) in valores else valores[0]


# ------------------------------------------------------------
# Renderização da imagem selecionada
# ------------------------------------------------------------

def render_validacao_imagem_selecionada(_=None):
    with out_validacao_unica:
        clear_output(wait=True)

        df_val = STATE.get('df_validacao_visual_atual')

        if df_val is None or len(df_val) == 0:
            atualizar_lista_imagens_validacao()
            df_val = STATE.get('df_validacao_visual_atual')

        idx = int(w_validacao_imagem.value) if w_validacao_imagem.value is not None else -1

        if df_val is None or len(df_val) == 0 or idx < 0 or idx >= len(df_val):
            print('Nenhuma imagem selecionada para validação visual.')
            return

        row = df_val.iloc[idx]
        row = _normalizar_row_zoom_validacao(row)

        clf, modelo_ok = _modelo_valido_para_assinatura_atual(verbose=True)

        if not modelo_ok:
            clf = None

        try:
            img_antes, candidatos_ini, boxes_gt_ini, zoom_info_ini, info_ini = desenhar_estado_inicial_sem_treinamento(row)

            img_depois, resultados, candidatos, boxes_gt, zoom_info, info = aplicar_modelo_em_row(
                row,
                clf,
                desenhar_gt=True,
                score_min=w_vis_score_min.value,
                nms_iou=w_vis_nms_iou.value,
                max_det=w_vis_max_det.value
            )

        except Exception as e:
            import traceback

            print('Erro ao validar imagem selecionada:')
            print(str(e))

            print('\nTraceback completo:')
            print(traceback.format_exc())

            print('\nLinha selecionada no dataframe:')
            try:
                display(row.to_frame('valor'))
            except Exception:
                print(row)

            return

        nome = str(row.get('arquivo', Path(row.get('path', '')).name))
        split = str(row.get('split', ''))

        zoom_info = _normalizar_zoom_info(zoom_info)

        if _bool_seguro(zoom_info.get('zoom_out_aplicado'), default=False):
            pct = zoom_info.get('zoom_out_pct', 0)
            ztxt = f" | zoom out {pct}%"
        else:
            ztxt = ''

        fig, axs = plt.subplots(
            1,
            2,
            figsize=(18, 7),
            constrained_layout=False
        )

        axs[0].imshow(img_antes, interpolation='nearest')
        axs[0].set_title(
            f"Antes — candidatos geométricos iniciais\n"
            f"{split} | {nome[:42]} | GT={len(boxes_gt_ini)} | ROIs={len(candidatos_ini)}{ztxt}",
            fontsize=10,
            pad=6
        )
        axs[0].axis('off')

        axs[1].imshow(img_depois, interpolation='nearest')

        if modelo_ok:
            axs[1].set_title(
                f"Depois — HOG/SVM + classificação + NMS\n"
                f"pred={info.get('pred_final', 0)}/{info.get('pred_raw', 0)} | "
                f"ROIs={len(candidatos)} | "
                f"IoU médio={info.get('mean_best_iou_pred', 0):.2f}",
                fontsize=10,
                pad=6
            )
        else:
            axs[1].set_title(
                f"Depois — modelo ausente/desatualizado\n"
                f"ROIs geométricas={len(candidatos)}",
                fontsize=10,
                pad=6
            )

        axs[1].axis('off')

        fig.subplots_adjust(
            left=0.01,
            right=0.99,
            top=0.90,
            bottom=0.02,
            wspace=0.03
        )

        plt.show()

        msg = (
            f"<small>"
            f"<b>Resumo:</b> "
            f"arquivo={nome} | "
            f"split={split} | "
            f"GT={len(boxes_gt)} | "
            f"ROIs iniciais={len(candidatos_ini)} | "
            f"ROIs após pipeline={len(candidatos)} | "
            f"predições finais={info.get('pred_final', 0)} | "
            f"erros de ROI={info.get('erro_rois', 0)}."
            f"</small>"
        )

        display(HTML(msg))


# ------------------------------------------------------------
# Eventos dos widgets
# ------------------------------------------------------------

def on_validacao_split_change(change):
    if change.get('name') == 'value':
        atualizar_lista_imagens_validacao()
        render_validacao_imagem_selecionada()


def on_validacao_param_change(change):
    if change.get('name') == 'value':
        render_validacao_imagem_selecionada()


w_validacao_split.observe(on_validacao_split_change, names='value')
w_vis_score_min.observe(on_validacao_param_change, names='value')
w_vis_nms_iou.observe(on_validacao_param_change, names='value')
w_vis_max_det.observe(on_validacao_param_change, names='value')
btn_atualizar_validacao.on_click(render_validacao_imagem_selecionada)

atualizar_lista_imagens_validacao()
render_validacao_imagem_selecionada()


NameError: name 'widgets' is not defined

## Painel final de Git seguro

Este painel ajuda a revisar o que foi alterado pelo notebook antes de preparar commit (registro local de alterações).

Ele foi projetado para aceitar somente arquivos do notebook clássico e setups de configuração. Datasets, resultados, pesos, vídeos, modelos `.joblib`, CSVs de histórico e arquivos temporários são bloqueados.


In [ ]:
# ============================================================
# 14. Painel final de Git seguro
# ============================================================
# Este painel executa apenas comandos locais de Git.
# O botão de push NÃO envia nada automaticamente; ele apenas mostra o comando sugerido.

import subprocess
import shlex

out_git_seguro = widgets.Output()

ALLOWED_GIT_PATHS = {
    'vision/cylinders_detect/classic_vision/detector_cilindros_hough_hog_svm_interativo.ipynb',
    'vision/config/user/cylinders_detect/setups_parametricos_user.json',
    'vision/config/user/cylinders_detect/setups_refinamento_expansao_user.json',
}

PROHIBITED_PREFIXES = (
    'vision/datasets/',
    'vision/results/',
    'results/',
    'datasets/',
    '.ipynb_checkpoints/',
)

PROHIBITED_SUFFIXES = (
    '.pt', '.onnx', '.mp4', '.webm', '.avi', '.mov',
    '.joblib', '.pkl', '.pickle',
    '.csv', '.zip', '.tar', '.gz',
    '.tmp', '.temp', '.bak',
)

btn_git_status = widgets.Button(
    description='Verificar status',
    button_style='info',
    icon='search',
    layout=widgets.Layout(width='180px')
)

btn_git_add = widgets.Button(
    description='Preparar permitidos',
    button_style='warning',
    icon='plus',
    layout=widgets.Layout(width='210px')
)

btn_git_commit = widgets.Button(
    description='Criar commit local',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='190px')
)

btn_git_push_cmd = widgets.Button(
    description='Mostrar comando push',
    button_style='',
    icon='upload',
    layout=widgets.Layout(width='210px')
)

w_git_commit_msg = widgets.Text(
    value='Revisa detector clássico de cilindros',
    description='mensagem:',
    layout=widgets.Layout(width='560px'),
    style={'description_width': '80px'}
)

w_git_confirm_commit = widgets.Checkbox(
    value=False,
    description='confirmo criar commit apenas com arquivos permitidos',
    indent=False,
    layout=widgets.Layout(width='430px')
)


def _run_git(args):
    proc = subprocess.run(
        ['git'] + list(args),
        cwd=str(PROJECT_ROOT),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    return proc.returncode, proc.stdout.strip(), proc.stderr.strip()


def _parse_git_status_short(text):
    itens = []
    for line in str(text).splitlines():
        if not line.strip():
            continue
        status = line[:2]
        path = line[3:].strip() if len(line) > 3 else ''
        # Em renames, usa o destino depois de " -> ".
        if ' -> ' in path:
            path = path.split(' -> ', 1)[1].strip()
        itens.append((status, path))
    return itens


def _is_prohibited_path(path):
    p = str(path).replace('\\', '/')
    if any(p.startswith(prefix) for prefix in PROHIBITED_PREFIXES):
        return True
    return p.lower().endswith(PROHIBITED_SUFFIXES)


def _status_atual():
    rc_branch, branch, err_branch = _run_git(['branch', '--show-current'])
    rc_status, status, err_status = _run_git(['status', '--short'])
    return branch, status, err_status or err_branch


def _paths_modificados_permitidos(status_text):
    itens = _parse_git_status_short(status_text)
    paths = [p for _, p in itens]
    permitidos_modificados = [p for p in paths if p in ALLOWED_GIT_PATHS]
    proibidos = [p for p in paths if _is_prohibited_path(p)]
    outros = [p for p in paths if p not in ALLOWED_GIT_PATHS and not _is_prohibited_path(p)]
    return permitidos_modificados, proibidos, outros, itens


def render_git_status(extra_msg=''):
    with out_git_seguro:
        clear_output(wait=True)
        branch, status, erro = _status_atual()

        print('Branch (linha de trabalho separada) atual:', branch or '(não identificada)')
        print('')
        print('Saída de git status --short:')
        print(status if status else '(sem alterações)')
        if erro:
            print('')
            print('Aviso/erro:', erro)

        pend = listar_pendencias_git() if 'listar_pendencias_git' in globals() else {}
        if pend:
            print('')
            print('Pendências registradas por botões do notebook:')
            for p, meta in pend.items():
                print(f"- {p} | {meta.get('motivo','')} | {meta.get('registrado_em','')}")
        else:
            print('')
            print('Nenhuma pendência registrada pelos botões de setup nesta sessão.')

        permitidos, proibidos, outros, _ = _paths_modificados_permitidos(status)
        print('')
        print('Arquivos permitidos modificados:', permitidos if permitidos else '(nenhum)')
        if proibidos:
            print('BLOQUEIO: há arquivos proibidos modificados/não rastreados:', proibidos)
        if outros:
            print('Atenção: há outros arquivos fora da lista automática:', outros)

        if extra_msg:
            print('')
            print(extra_msg)


def preparar_arquivos_permitidos(_=None):
    with out_git_seguro:
        clear_output(wait=True)

        branch, status, erro = _status_atual()
        permitidos, proibidos, outros, _ = _paths_modificados_permitidos(status)

        # Também inclui pendências registradas pelos botões do notebook, desde que sejam permitidas.
        pend = listar_pendencias_git() if 'listar_pendencias_git' in globals() else {}
        for p in pend.keys():
            if p in ALLOWED_GIT_PATHS and p not in permitidos:
                permitidos.append(p)

        print('Primeiro, conferência obrigatória:')
        print('git status --short')
        print(status if status else '(sem alterações)')
        print('')

        if proibidos:
            print('Preparação bloqueada: existem arquivos proibidos modificados/não rastreados.')
            for p in proibidos:
                print('-', p)
            print('')
            print('Remova, ignore ou descarte esses arquivos antes de preparar commit.')
            return

        if not permitidos:
            print('Nenhum arquivo permitido modificado foi encontrado para preparar.')
            if outros:
                print('Outros arquivos apareceram no status, mas não serão preparados automaticamente:')
                for p in outros:
                    print('-', p)
            return

        rc, out, err = _run_git(['add', '--'] + permitidos)
        print('Arquivos enviados para a área de preparação do commit:')
        for p in permitidos:
            print('-', p)

        if rc != 0:
            print('')
            print('Erro ao executar git add:')
            print(err or out)
            return

        print('')
        print('Status após git add:')
        rc2, out2, err2 = _run_git(['status', '--short'])
        print(out2 if out2 else '(sem alterações)')


def criar_commit_local(_=None):
    with out_git_seguro:
        clear_output(wait=True)

        branch, status, erro = _status_atual()
        print('Conferência obrigatória antes do commit:')
        print('git status --short')
        print(status if status else '(sem alterações)')
        print('')

        if not w_git_confirm_commit.value:
            print("Commit bloqueado: marque a caixa de confirmação antes de criar commit.")
            return

        msg = w_git_commit_msg.value.strip()
        if not msg:
            print('Commit bloqueado: escreva uma mensagem de commit.')
            return

        itens = _parse_git_status_short(status)
        staged = [(st, p) for st, p in itens if len(st) >= 1 and st[0] != ' ']
        if not staged:
            print('Commit bloqueado: nenhum arquivo está preparado. Clique primeiro em "Preparar permitidos".')
            return

        staged_paths = [p for _, p in staged]
        proibidos_staged = [p for p in staged_paths if _is_prohibited_path(p)]
        fora_lista = [p for p in staged_paths if p not in ALLOWED_GIT_PATHS]

        if proibidos_staged:
            print('Commit bloqueado: há arquivos proibidos preparados:')
            for p in proibidos_staged:
                print('-', p)
            return

        if fora_lista:
            print('Commit bloqueado: há arquivos preparados fora da lista automática permitida:')
            for p in fora_lista:
                print('-', p)
            print('')
            print('Remova esses arquivos da área de preparação antes de tentar novamente.')
            return

        rc, out, err = _run_git(['commit', '-m', msg])
        print(out or err)
        if rc == 0:
            w_git_confirm_commit.value = False
            try:
                GIT_PENDING_PATHS.clear()
            except Exception:
                pass
            print('')
            print('Commit local criado com sucesso.')
            print('Próximo passo sugerido: clique em "Mostrar comando push".')


def mostrar_comando_push(_=None):
    with out_git_seguro:
        clear_output(wait=True)
        branch, status, erro = _status_atual()
        print('Branch (linha de trabalho separada) atual:', branch or '(não identificada)')
        print('')
        if not branch:
            print('Não foi possível identificar a branch atual.')
            return
        if not branch.startswith('feature/'):
            print('Atenção: a branch atual não começa com feature/. Confira antes de enviar ao GitHub.')
            print('')
        print('Este botão NÃO executa push.')
        print('Para enviar ao GitHub depois de revisar o commit, rode no terminal:')
        print('')
        print(f'git push -u origin {branch}')


btn_git_status.on_click(lambda _: render_git_status())
btn_git_add.on_click(preparar_arquivos_permitidos)
btn_git_commit.on_click(criar_commit_local)
btn_git_push_cmd.on_click(mostrar_comando_push)

painel_git_seguro = widgets.VBox([
    widgets.HTML('<b>Painel Git seguro</b>'),
    widgets.HTML('<small>Use primeiro "Verificar status". O notebook só prepara arquivos permitidos. O commit local exige confirmação explícita. O push é apenas mostrado como comando para executar no terminal.</small>'),
    widgets.HBox([btn_git_status, btn_git_add, btn_git_commit, btn_git_push_cmd], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([w_git_commit_msg], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    w_git_confirm_commit,
    out_git_seguro
], layout=widgets.Layout(border='1px solid #444', padding='8px', margin='10px 0'))

display(painel_git_seguro)
render_git_status()


## Observações desta versão V2

- Somente pares de retas paralelas válidas geram ROIs.
- A estratégia por reta única foi removida.
- A BBox não é mais expandida até evidências curvas; ela é calculada por proporção comprimento/largura.
- O score da ROI combina overlap entre as retas, similaridade angular e proximidade da razão L/W com o valor alvo ajustável.
- Como a formulação das ROIs mudou, o modelo HOG/SVM deve ser retreinado e salvo como versão V2.
